# Line-by-line commented version of `Job_Delta_Reference_WF_BUDGET_D_Only_Platform Tool Updated Job and Table.py.python`

This notebook was generated from the uploaded Databricks `.dbc` file.

What this version does:
- preserves the original notebook structure by command cell
- adds a comment **before each non-empty line**
- explains what each line is doing in simple terms
- keeps the original code directly under each explanation

Notes:
- this is primarily an **explanation notebook**
- some lines depend on Databricks features such as `dbutils`, Spark, mounts, and `%run`
- the uploaded `__Utilities__.py.ipynb` is referenced by the original notebook and is likely required for full execution


## Dependency note

The original notebook contains a `%run ./__Utilities__.py` command.

That means this notebook relies on helper code from the external utilities notebook.  
I kept that reference and explained it inline, but full execution still depends on the helper notebook existing in the expected location.


## Command cell 1

This section corresponds to command 1 from the original Databricks notebook.


In [ ]:
# Import specific objects from a module so they can be used directly in later code: from datetime import datetime.
from datetime import datetime
# Capture the current timestamp so the notebook can stamp logs, paths, or outputs.
now = datetime.now()
# Format a date or time value into a string representation.
now.strftime("%H:%M:%S")

## Command cell 2

This section corresponds to command 2 from the original Databricks notebook.


In [ ]:
# Import the module(s) required for the next part of the workflow: os.
import os
# Import the module(s) required for the next part of the workflow: docx.
import docx
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.types import *.
from pyspark.sql.types import *

# Assign the result on the right-hand side to `max_excel_row` so it can be reused later.
max_excel_row = 1000
# Assign the result on the right-hand side to `max_rows_permitted` so it can be reused later.
max_rows_permitted = 5000000

## Command cell 3

This section corresponds to command 3 from the original Databricks notebook.


In [ ]:
# Run the external notebook or helper script ./__Utilities__.py so its functions, variables, or setup become available in this notebook.
%run ./__Utilities__.py 

## Command cell 4

This section corresponds to command 4 from the original Databricks notebook.


In [ ]:

# Create, remove, or manage Databricks widgets so the notebook can accept runtime parameters from the user.
dbutils.widgets.removeAll()
# Create, remove, or manage Databricks widgets so the notebook can accept runtime parameters from the user.
dbutils.widgets.dropdown("Env","DEV",["DEV","QA","PROD","PROD_DR"])
# Create, remove, or manage Databricks widgets so the notebook can accept runtime parameters from the user.
dbutils.widgets.dropdown("Comparison Options","Netezza Vs Synapse-(7 Years Transactional),Netizza Vs Gold-(Full)",["Netezza Vs Synapse-(7 Years Transactional),Netizza Vs Gold-(Full)","Netezza Vs Synapse-Non Transactional Full,Netizza to Gold-(Full)"])
# Create, remove, or manage Databricks widgets so the notebook can accept runtime parameters from the user.
dbutils.widgets.text("Log_Id","1")
# Create, remove, or manage Databricks widgets so the notebook can accept runtime parameters from the user.
dbutils.widgets.dropdown("Load Type","Historical",["Historical","Non-Historical"])
# Create, remove, or manage Databricks widgets so the notebook can accept runtime parameters from the user.
dbutils.widgets.dropdown("Report Type","Summary Report",["Summary Report","Detailed Report"])
# Create, remove, or manage Databricks widgets so the notebook can accept runtime parameters from the user.
dbutils.widgets.dropdown("Level", "Job",["Job","Table"])
# Create, remove, or manage Databricks widgets so the notebook can accept runtime parameters from the user.
dbutils.widgets.text("Job", "JB_HRM_BUILD_SG")
# Create, remove, or manage Databricks widgets so the notebook can accept runtime parameters from the user.
dbutils.widgets.text("Tables", "WAREHOUSE.CONTACTS_HIST,WAREHOUSE.ADDRESS_HIST,WAREHOUSE.SKU_PRC_HIST,STAGING.TICKET_ITEM_LINK,STAGING.INVENTORY_LINK")

## Command cell 5

This section corresponds to command 5 from the original Databricks notebook.


In [ ]:
# Read the selected widget value and store it in a Python variable for later branching logic.
Env = dbutils.widgets.get("Env")
# Read the selected widget value and store it in a Python variable for later branching logic.
TR_NonTR= dbutils.widgets.get("Comparison Options")
# Read the selected widget value and store it in a Python variable for later branching logic.
Log_Id = dbutils.widgets.get("Log_Id")
# Read the selected widget value and store it in a Python variable for later branching logic.
Load_Type = dbutils.widgets.get("Load Type")
# Read the selected widget value and store it in a Python variable for later branching logic.
RptType=dbutils.widgets.get("Report Type")
# Read the selected widget value and store it in a Python variable for later branching logic.
Level = dbutils.widgets.get("Level")

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if Level == 'Job':
# Read the selected widget value and store it in a Python variable for later branching logic.
    Job = str(dbutils.widgets.get("Job")).upper()
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if Job == '':
# Print a message or value to the notebook output for validation or debugging.
        print("No job name entered for validation. Enter a job name")
# Execute this line as part of the notebook's workflow logic.
        dbutils.notebook.exit('No job name entered for validation')
# Run the fallback branch when the earlier conditions do not match.
    else:
# Original notebook comment retained.
        ## Check if job name is valid
# Assign the result on the right-hand side to `job_list_query` so it can be reused later.
        job_list_query = '''(select distinct UPPER(Job) as job from dbo.DataflowSourceTargets)Config'''
# Assign the result on the right-hand side to `job_list_df` so it can be reused later.
        job_list_df = readfromdbSQL(Env, job_list_query)
# Assign the result on the right-hand side to `job_list` so it can be reused later.
        job_list = [row[0] for row in job_list_df.collect()]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
        if Job not in job_list:
# Print a message or value to the notebook output for validation or debugging.
            print('{Job} is not a valid job entry. Please enter a valid job name')
# Execute this line as part of the notebook's workflow logic.
            dbutils.notebook.exit('Invalid job entry')
# Run the fallback branch when the earlier conditions do not match.
        else:
# Print a message or value to the notebook output for validation or debugging.
            print(f"Performing validation for job: {Job}")

# Check the next condition if the previous condition was not met.
elif Level == 'Table':
# Read the selected widget value and store it in a Python variable for later branching logic.
    Tables = str(dbutils.widgets.get("Tables"))
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if Tables == '':
# Print a message or value to the notebook output for validation or debugging.
        print("No tables listed for validation. Enter a list of tables to be configured")
# Execute this line as part of the notebook's workflow logic.
        dbutils.notebook.exit('No tables listed for validation')
# Run the fallback branch when the earlier conditions do not match.
    else:
# Print a message or value to the notebook output for validation or debugging.
        print(f"Performing validation on the specified tables")

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if TR_NonTR == 'Netezza Vs Synapse-Non Transactional Full,Netizza to Gold-(Full)':
# Assign the result on the right-hand side to `PL_TL` so it can be reused later.
    PL_TL='Full_GS'
# Check the next condition if the previous condition was not met.
elif TR_NonTR == 'Netezza Vs Synapse-(7 Years Transactional),Netizza Vs Gold-(Full)':
# Assign the result on the right-hand side to `PL_TL` so it can be reused later.
    PL_TL='7GS'

# Print a message or value to the notebook output for validation or debugging.
print(PL_TL,RptType)
# Print a message or value to the notebook output for validation or debugging.
print(Load_Type)

## Command cell 6

This section corresponds to command 6 from the original Databricks notebook.


In [ ]:
# Import specific objects from a module so they can be used directly in later code: from datetime import datetime.
from datetime import datetime
# Assign the result on the right-hand side to `mountSilver` so it can be reused later.
mountSilver = "/mnt/silver/"
# Capture the current date for use in file paths, filtering, or logging.
today = date.today()
# Original notebook comment retained.
#print(today)
# Capture the current date for use in file paths, filtering, or logging.
dt = date.today()
# Capture the current timestamp so the notebook can stamp logs, paths, or outputs.
ct=datetime.now()
# Assign the result on the right-hand side to `t` so it can be reused later.
t=str(ct)[11:19]
# Assign the result on the right-hand side to `times` so it can be reused later.
times = t.replace(":", "-")

## Command cell 7

This section corresponds to command 7 from the original Databricks notebook.


In [ ]:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if Level == 'Job':
# Assign the result on the right-hand side to `path` so it can be reused later.
    path = f"/dbfs/mnt/silver/DPT_TEMP/{dt}/{times}->{Job}/"
# Check the next condition if the previous condition was not met.
elif Level == 'Table':
# Assign the result on the right-hand side to `path` so it can be reused later.
    path = f"/dbfs/mnt/silver/DPT_TEMP/{dt}/{times}->Tables/"
# Assign the result on the right-hand side to `os.makedirs(path, exist_ok` so it can be reused later.
os.makedirs(path, exist_ok=True)
# Print a message or value to the notebook output for validation or debugging.
print(path)

## Command cell 8

This section corresponds to command 8 from the original Databricks notebook.


In [ ]:
# Define the function `save_log` so this block can be reused later in the notebook.
def save_log(doc):
# Save the current object or output to the specified destination.
    doc.save('log.docx')
# Execute this line as part of the notebook's workflow logic.
    shutil.copy2('log.docx',path+'log.docx')
# Execute this line as part of the notebook's workflow logic.
    os.remove('log.docx')
# Execute this line as part of the notebook's workflow logic.
    return

## Command cell 9

This section corresponds to command 9 from the original Databricks notebook.


In [ ]:
# Original notebook comment retained.
## Create log header
# Assign the result on the right-hand side to `doc` so it can be reused later.
doc = docx.Document()
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if Level == 'Job':
# Add a heading to the Word document being used as a log or report.
    doc.add_heading(f'Logs for {RptType} for {Load_Type} load of {Job} run in {Env} environment on {dt}', 0)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if Level == 'Table':
# Add a heading to the Word document being used as a log or report.
    doc.add_heading(f'Logs for {RptType} for {Load_Type} load of specified tables run in {Env} environment on {dt}', 0)
# Add a heading to the Word document being used as a log or report.
doc.add_heading(f'Comparison Option: {TR_NonTR}', 1)
# Add a paragraph to the Word document so the log captures another message or detail.
doc.add_paragraph('')

## Command cell 10

This section corresponds to command 10 from the original Databricks notebook.


In [ ]:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if Level == 'Job':
# Create a SQL query string that will be executed later against Spark or another data source.
    query = """
# This line is part of the SQL statement being built for the data comparison or extraction step.
    (select UPPER(Job) as Job, DataSource as SchemaName,EntityName as TableName,
# Execute this line as part of the notebook's workflow logic.
    concat(TRIM(DataSource),'.',TRIM(EntityName)) as FullTableName
# Execute this line as part of the notebook's workflow logic.
    from dbo.DataflowSourceTargets
# Assign the result on the right-hand side to `where UPPER(Job)` so it can be reused later.
    where UPPER(Job) = '{}' and 
# Assign the result on the right-hand side to `SourceType` so it can be reused later.
        SourceType = 'Netezza'
# Assign the result on the right-hand side to `and EntityUsage` so it can be reused later.
        and EntityUsage = 'Target'
# This line is part of the SQL statement being built for the data comparison or extraction step.
        group by Job,EntityName,DataSource
# Execute this line as part of the notebook's workflow logic.
        )Config""".format(Job)
# Print a message or value to the notebook output for validation or debugging.
    print(query)

## Command cell 11

This section corresponds to command 11 from the original Databricks notebook.


In [ ]:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if Level == 'Job':
# Assign the result on the right-hand side to `job_tables` so it can be reused later.
    job_tables = readfromdbSQL(Env, query)
# Render the object in the notebook output so the user can inspect it visually.
    display(job_tables)

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if job_tables.count() == 0:
# Print a message or value to the notebook output for validation or debugging.
        print(f"No table configured for job {Job}")
# Execute this line as part of the notebook's workflow logic.
        dbutils.notebook.exit(f"No table configured for job {Job}")

## Command cell 12

This section corresponds to command 12 from the original Databricks notebook.


In [ ]:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if Level == 'Job':
# Assign the result on the right-hand side to `tables_list` so it can be reused later.
    tables_list = job_tables.select("FullTableName").rdd.flatMap(lambda x: x).collect()
# Assign the result on the right-hand side to `tables_list` so it can be reused later.
    tables_list = "("+",".join(["'"+str(x)+"'" for x in tables_list])+")"
# Print a message or value to the notebook output for validation or debugging.
    print(tables_list)

## Command cell 13

This section corresponds to command 13 from the original Databricks notebook.


In [ ]:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if Level == 'Table':
# Assign the result on the right-hand side to `tables_list` so it can be reused later.
    tables_list = "("+",".join(["'"+str(x)+"'" for x in Tables.split(",")])+")"
# Print a message or value to the notebook output for validation or debugging.
    print(tables_list)

## Command cell 14

This section corresponds to command 14 from the original Databricks notebook.


In [ ]:
# Assign the result on the right-hand side to `configquery` so it can be reused later.
configquery = """(SELECT * FROM DBO.FWK_GOLD_TABLES 
# This line is part of the SQL statement being built for the data comparison or extraction step.
WHERE [DATABASE]+'.'+TABLE_NAME IN {} )Config""".format(tables_list)
# Assign the result on the right-hand side to `gold_df1` so it can be reused later.
gold_df1 = readfromdbSQL(Env,configquery)
# Render the object in the notebook output so the user can inspect it visually.
display(gold_df1)

## Command cell 15

This section corresponds to command 15 from the original Databricks notebook.


In [ ]:
# Assign the result on the right-hand side to `silver_list` so it can be reused later.
silver_list=['STAGING','WAREHOUSE','HRSTAGING','SDS_STAGING','SDS_CU']
# Assign the result on the right-hand side to `gold_list` so it can be reused later.
gold_list = ['ACCOUNTING','FINANCE','HH_ALLINK','HRMART','HYBRIS','MARKETING','MART','OPERATIONS','REALESTATE']

## Command cell 16

This section corresponds to command 16 from the original Databricks notebook.


In [ ]:
# Original notebook comment retained.
## The table details required for filtering the transactional table for the 7 years filter
# Assign the result on the right-hand side to `trQuery` so it can be reused later.
trQuery = "(select distinct SUBSTRING(TableName, PATINDEX('%[.]%', TableName)+1,LEN(TableName)) TableName ,nullif(RetentionColumn,'UNKNOWN') Filter7 from  [dbo].[DtIngestionMetadata] where RetentionColumn<>'UNKNOWN')trQuery"
# Assign the result on the right-hand side to `TR_df` so it can be reused later.
TR_df = readfromdbSQL(Env,trQuery)
# Render the object in the notebook output so the user can inspect it visually.
display(TR_df)

## Command cell 17

This section corresponds to command 17 from the original Databricks notebook.


In [ ]:
# Original notebook comment retained.
## Join tables based on the comparison option choosen
# Original notebook comment retained.
## This dataframe determines which tables will be included in the validation
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if (PL_TL == '7GS'):
# Original notebook comment retained.
    ## Narrows the table select to only transactional tables involved in the job
# Execute this line as part of the notebook's workflow logic.
    gold_df = gold_df1.join(TR_df,[TR_df.TableName == gold_df1.TABLE_NAME]
# Execute this line as part of the notebook's workflow logic.
    ,'inner'
# Execute this line as part of the notebook's workflow logic.
    ).select(gold_df1['*'],TR_df.Filter7).distinct() 
# Run the fallback branch when the earlier conditions do not match.
else:
# Original notebook comment retained.
    ## Includes all tables involved in the job
# Execute this line as part of the notebook's workflow logic.
    gold_df = gold_df1.join(TR_df,[TR_df.TableName == gold_df1.TABLE_NAME]
# Execute this line as part of the notebook's workflow logic.
    ,'left'
# Execute this line as part of the notebook's workflow logic.
    ).select(gold_df1['*'],TR_df.Filter7)#.where(TR_df.TableName.isNull())
# Original notebook comment retained.
#display(gold_df.select(gold_df.TABLE_NAME,TR_df.Filter7))
# Execute this line as part of the notebook's workflow logic.
gold_df.cache().count()
# Execute this line as part of the notebook's workflow logic.
gold_df.createOrReplaceTempView("gold_df")

## Command cell 18

This section corresponds to command 18 from the original Databricks notebook.


In [ ]:
# Render the object in the notebook output so the user can inspect it visually.
display(gold_df)

## Command cell 19

This section corresponds to command 19 from the original Databricks notebook.


In [ ]:
# Assign the result on the right-hand side to `cnttable` so it can be reused later.
cnttable=gold_df.count()
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if cnttable==0:
# Execute this line as part of the notebook's workflow logic.
    dbutils.notebook.exit(cnttable) 

## Command cell 20

This section corresponds to command 20 from the original Databricks notebook.


In [ ]:
# Original notebook comment retained.
## List of unique databases with tables to be validated
# Assign the result on the right-hand side to `validation_databases` so it can be reused later.
validation_databases = [row.DATABASE for row in gold_df.select('DATABASE').distinct().collect()]

# Original notebook comment retained.
## Instantiate list of dictionaries
# Assign the result on the right-hand side to `validation_tables_info` so it can be reused later.
validation_tables_info = []

# Original notebook comment retained.
## Append dictionary of database with list of distinct tables
# Start a loop so the same logic is applied repeatedly across multiple items.
for db in validation_databases:
# Original notebook comment retained.
    #print(db)
# Assign the result on the right-hand side to `dict_schema` so it can be reused later.
    dict_schema={}
# Execute this line as part of the notebook's workflow logic.
    list_tables = [row.TABLE_NAME for row in gold_df.select(['TABLE_NAME']).where(gold_df.DATABASE == db).distinct().collect()]
# Assign the result on the right-hand side to `dict_schema[db]` so it can be reused later.
    dict_schema[db]=[*set(list_tables)]
# Assign the result on the right-hand side to `cp_dict_schema` so it can be reused later.
    cp_dict_schema=dict_schema.copy()
# Original notebook comment retained.
    #print(dict_schema)
# Execute this line as part of the notebook's workflow logic.
    validation_tables_info.append(cp_dict_schema) 
# Print a message or value to the notebook output for validation or debugging.
print(validation_tables_info)

## Command cell 21

This section corresponds to command 21 from the original Databricks notebook.


In [ ]:
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.types import *.
from pyspark.sql.types import *
# Import the module(s) required for the next part of the workflow: pyspark.sql.functions as sf.
import pyspark.sql.functions as sf
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.functions import col,when.
from pyspark.sql.functions import col,when
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql import SQLContext.
from pyspark.sql import SQLContext
# Import specific objects from a module so they can be used directly in later code: from datetime import datetime.
from datetime import datetime
# Import specific objects from a module so they can be used directly in later code: from datetime import date.
from datetime import date
# Import the module(s) required for the next part of the workflow: datetime.
import datetime
# Import the module(s) required for the next part of the workflow: pandas as pd.
import pandas as pd
# Import the module(s) required for the next part of the workflow: shutil.
import shutil

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if PL_TL == '7GS':
# Assign the result on the right-hand side to `netezza_sub` so it can be reused later.
    netezza_sub = '7 yrs'
# Run the fallback branch when the earlier conditions do not match.
else:
# Assign the result on the right-hand side to `netezza_sub` so it can be reused later.
    netezza_sub = 'Full'

# Original notebook comment retained.
## Header for log document
# Add a heading to the Word document being used as a log or report.
doc.add_heading('Log From Metadata Info Generation and Summary Report', 2)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
try:
# Assign the result on the right-hand side to `tab` so it can be reused later.
    tab = 0

# Original notebook comment retained.
    ## Define the schema for the netezza metadata report
# Assign the result on the right-hand side to `netezza_metadata_schema` so it can be reused later.
    netezza_metadata_schema = StructType([StructField('NETEZZA_DATABASE', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('NETEZZA_TABLE_NAME', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('NETEZZA_COLUMN_NAME', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('NETEZZA_DATATYPE', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('NETEZZA_ORDINAL_POSITION', StringType(), True)])

# Original notebook comment retained.
    ## Define the schema for the synapse metadata report
# Assign the result on the right-hand side to `synapse_metadata_schema` so it can be reused later.
    synapse_metadata_schema = StructType([StructField('SYNAPSE_DATABASE', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('SYNAPSE_TABLE_NAME', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('SYNAPSE_COLUMN_NAME', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('SYNAPSE_DATATYPE', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('SYNAPSE_ORDINAL_POSITION', StringType(), True)])

# Original notebook comment retained.
    ## Define the schema for the gold metadata report
# Assign the result on the right-hand side to `gold_metadata_schema` so it can be reused later.
    gold_metadata_schema = StructType([StructField('GOLD_DATABASE', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('GOLD_TABLE_NAME', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('GOLD_COLUMN_NAME', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('GOLD_DATATYPE', StringType(), True)
# Execute this line as part of the notebook's workflow logic.
                                ])

# Original notebook comment retained.
    ## Define the schema for the silver metadata report
# Assign the result on the right-hand side to `silver_metadata_schema` so it can be reused later.
    silver_metadata_schema = StructType([StructField('SILVER_DATABASE', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('SILVER_TABLE_NAME', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('SILVER_COLUMN_NAME', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                StructField('SILVER_DATATYPE', StringType(), True)
# Execute this line as part of the notebook's workflow logic.
                                ])

# Original notebook comment retained.
    ## Define the schema for the summary report


# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if (PL_TL == '7GS') | (Load_Type != 'Historical'):
# Assign the result on the right-hand side to `report_dataframe_schema_1` so it can be reused later.
        report_dataframe_schema_1= StructType([StructField('NETEZZA_SCHEMA', StringType(), True), #SAP HANA SCHEMA NAME
# Execute this line as part of the notebook's workflow logic.
                                    StructField('NETEZZA_TABLE', StringType(), True), #SAP HANA TABLE NAME
# Execute this line as part of the notebook's workflow logic.
                                    StructField('SYNAPSE_SCHEMA', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                    StructField('SYNAPSE_TABLE', StringType(), True), #ADLS DELTA TABLE NAME
# Execute this line as part of the notebook's workflow logic.
                                    StructField('MISMATCHED_COLUMNS (NETEZZA - SYNAPSE)', ArrayType(StringType()), True), #list of columns in Netezza which is not present in Synapse                                                                                               
# Execute this line as part of the notebook's workflow logic.
                                    StructField('MISMATCHED_COLUMNS (SYNAPSE - NETEZZA)', ArrayType(StringType()), True),#list of columns in Synapse which is not present in Netezza 
# Execute this line as part of the notebook's workflow logic.
                                    StructField(f'NETEZZA_TABLE_ROW_COUNT ({netezza_sub})', StringType(), True), # row count in Netezza TABLE
# Execute this line as part of the notebook's workflow logic.
                                    StructField('SYNAPSE_TABLE_ROW_COUNT', StringType(), True), # row count in Netezza TABLE
# Original notebook comment retained.
                                    #StructField('GOLD_TABLE_ROW_COUNT', StringType(), True), # row count in Synapse TABLE
# Execute this line as part of the notebook's workflow logic.
                                    StructField('ROW_COUNT_DIFFERENCE(NETEZZA - SYNAPSE)', StringType(), True)]) # difference in row counts
# Original notebook comment retained.
                                    #StructField('ROW_COUNT_DIFFERENCE(NETEZZA - GOLD)', StringType(), True)]) # difference in row counts

# Run the fallback branch when the earlier conditions do not match.
    else:
# Assign the result on the right-hand side to `report_dataframe_schema_1` so it can be reused later.
        report_dataframe_schema_1= StructType([StructField('NETEZZA_SCHEMA', StringType(), True), #SAP HANA SCHEMA NAME
# Execute this line as part of the notebook's workflow logic.
                                    StructField('NETEZZA_TABLE', StringType(), True), #SAP HANA TABLE NAME
# Execute this line as part of the notebook's workflow logic.
                                    StructField('SYNAPSE_SCHEMA', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                    StructField('SYNAPSE_TABLE', StringType(), True), #ADLS DELTA TABLE NAME
# Execute this line as part of the notebook's workflow logic.
                                    StructField('MISMATCHED_COLUMNS (NETEZZA - SYNAPSE)', ArrayType(StringType()), True), #list of columns in Netezza which is not present in Synapse                                                                                               
# Execute this line as part of the notebook's workflow logic.
                                    StructField('MISMATCHED_COLUMNS (SYNAPSE - NETEZZA)', ArrayType(StringType()), True),#list of columns in Synapse which is not present in Netezza 
# Execute this line as part of the notebook's workflow logic.
                                    StructField(f'NETEZZA_TABLE_ROW_COUNT ({netezza_sub})', StringType(), True), # row count in Netezza TABLE
# Execute this line as part of the notebook's workflow logic.
                                    StructField('SYNAPSE_TABLE_ROW_COUNT', StringType(), True), # row count in Netezza TABLE
# Execute this line as part of the notebook's workflow logic.
                                    StructField('GOLD_TABLE_ROW_COUNT', StringType(), True), # row count in Synapse TABLE
# Execute this line as part of the notebook's workflow logic.
                                    StructField('ROW_COUNT_DIFFERENCE(NETEZZA - SYNAPSE)', StringType(), True), # difference in row counts
# Execute this line as part of the notebook's workflow logic.
                                    StructField('ROW_COUNT_DIFFERENCE(NETEZZA - GOLD)', StringType(), True)]) # difference in row counts



# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if Load_Type == 'Historical':
# Assign the result on the right-hand side to `report_dataframe_schema_2` so it can be reused later.
        report_dataframe_schema_2= StructType([StructField('NETEZZA_SCHEMA', StringType(), True), #SAP HANA SCHEMA NAME
# Execute this line as part of the notebook's workflow logic.
                                    StructField('NETEZZA_TABLE', StringType(), True), #SAP HANA TABLE NAME
# Execute this line as part of the notebook's workflow logic.
                                    StructField('SILVER_SCHEMA', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                    StructField('SILVER_TABLE', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                    StructField('MISMATCHED_COLUMNS (NETEZZA - SILVER)', ArrayType(StringType()), True),
# Execute this line as part of the notebook's workflow logic.
                                    StructField('MISMATCHED_COLUMNS (SILVER - NETEZZA)', ArrayType(StringType()), True),
# Execute this line as part of the notebook's workflow logic.
                                    StructField(f'NETEZZA_TABLE_ROW_COUNT ({netezza_sub})', StringType(), True), # row count in Netezza TABLE
# Execute this line as part of the notebook's workflow logic.
                                    StructField('SILVER_TABLE_ROW_COUNT', StringType(), True),
# Execute this line as part of the notebook's workflow logic.
                                    StructField('ROW_COUNT_DIFFERENCE(NETEZZA - SILVER)', StringType(), True)])


# Assign the result on the right-hand side to `report_dataframe_info` so it can be reused later.
    report_dataframe_info=[]

# Original notebook comment retained.
    ## Create list of gold database with tables to be validated
# Assign the result on the right-hand side to `relevant_gold_db` so it can be reused later.
    relevant_gold_db = set(validation_databases).intersection(set(gold_list))
# Original notebook comment retained.
    ## Create list of silver database with tables to be validated
# Assign the result on the right-hand side to `relevant_silver_db` so it can be reused later.
    relevant_silver_db = set(validation_databases).intersection(set(silver_list))




# Original notebook comment retained.
    ## Summary report for netezza v. synapse v. gold tables
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if len(relevant_gold_db) > 0:

# Assign the result on the right-hand side to `netezza_metadata_1` so it can be reused later.
        netezza_metadata_1 = []
# Assign the result on the right-hand side to `synapse_metadata` so it can be reused later.
        synapse_metadata = []
# Assign the result on the right-hand side to `gold_metadata` so it can be reused later.
        gold_metadata = []
# Assign the result on the right-hand side to `report_dataframe_info_1` so it can be reused later.
        report_dataframe_info_1=[]


# Start a loop so the same logic is applied repeatedly across multiple items.
        for database in relevant_gold_db:

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
            if Env == 'QA':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = f"GDC_{database}"
# Check the next condition if the previous condition was not met.
            elif Env == 'PROD_DR':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = f"{database}_DR"
# Run the fallback branch when the earlier conditions do not match.
            else:
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = database

# Original notebook comment retained.
            #print(database)
# Execute this line as part of the notebook's workflow logic.
            tables=[v for d in validation_tables_info for k,v in d.items() if k == database][0]
# Start a loop so the same logic is applied repeatedly across multiple items.
            for table in tables:
# Print a message or value to the notebook output for validation or debugging.
                print(f'{database}.{table}')

# Original notebook comment retained.
                ## Header for log document
# Add a heading to the Word document being used as a log or report.
                doc.add_heading(f'{database}.{table}', 3)

# Start a new indented code block for the statement above.
                try : 
# Original notebook comment retained.
                    ## NETEZZA METADATA
# Assign the result on the right-hand side to `query_for_columns_info` so it can be reused later.
                    query_for_columns_info = f"""select attname,format_type,attnum from  {db_tmp}._V_RELATION_COLUMN WHERE NAME = '{table}'""" 
# Assign the result on the right-hand side to `netezza_df` so it can be reused later.
                    netezza_df=readfromNetizza(Env,query_for_columns_info,database)
# Original notebook comment retained.
                    ## List of columns in the table
# Assign the result on the right-hand side to `netezza_side_columns` so it can be reused later.
                    netezza_side_columns = [row[0] for row in netezza_df.select('attname').collect()]  #netezza_df.rdd.map(lambda x: x[0]).collect()
# Original notebook comment retained.
                    ## List of datatype for each column in table
# Assign the result on the right-hand side to `netezza_side_datatype` so it can be reused later.
                    netezza_side_datatype= [row[0] for row in netezza_df.select('format_type').collect()]
# Original notebook comment retained.
                    ## List of ordinal position
# Assign the result on the right-hand side to `netezza_ordinal_position` so it can be reused later.
                    netezza_ordinal_position = [row[0] for row in netezza_df.select('attnum').collect()]

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                    if (PL_TL == '7GS'):
# Original notebook comment retained.
                        ## Retrieve the filter column for the table. This is a date column used to identify the minimum date in which data from in table was loaded from Netizza
# Execute this line as part of the notebook's workflow logic.
                        filterColumn=gold_df.select('Filter7').where((gold_df.TABLE_NAME==table)&(gold_df.DATABASE==database)).distinct()  #.rdd.map(lambda x: x.Filter7).collect()
# Assign the result on the right-hand side to `filterColumn` so it can be reused later.
                        filterColumn = [row[0] for row in filterColumn.collect()]
# Original notebook comment retained.
                        #print(filterColumn)
# Assign the result on the right-hand side to `filterColumn` so it can be reused later.
                        filterColumn=str(filterColumn)
# Assign the result on the right-hand side to `filterColumn` so it can be reused later.
                        filterColumn=filterColumn.replace("[","").replace("]","").replace("'","")
# Original notebook comment retained.
                        #print('filterCol', filterColumn)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if filterColumn!='None':
# Original notebook comment retained.
                            #print('filterCol2:', filterColumn) 
# Original notebook comment retained.
                            ## Fetch the minimum date from synapse to avoid mismatch on the day where data is not loaded in netizza
# Assign the result on the right-hand side to `query_forValuefilter` so it can be reused later.
                            query_forValuefilter= f"""select CAST(min({filterColumn}) as varchar(10)) {filterColumn} from {database}.{table}""" 
# Original notebook comment retained.
                            #print(query_forValuefilter)
# Assign the result on the right-hand side to `filterValue1` so it can be reused later.
                            filterValue1=readfromSynapse(Env,query_forValuefilter)
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                            filterValue=[row[0] for row in filterValue1.collect()]  #filterValue1.rdd.map(lambda x: x[0]).collect()
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                            filterValue=str(filterValue)
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                            filterValue=filterValue.replace("[","").replace("]","")
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if filterValue!='None':
# Assign the result on the right-hand side to `netezza_count_info_query` so it can be reused later.
                                netezza_count_info_query = f"""select count(*) from {db_tmp}.{table} where {filterColumn} >= {filterValue}""" 
# Print a message or value to the notebook output for validation or debugging.
                                print('filtering Netezza for 7 years only')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('filtering Netezza for 7 years only')
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `netezza_count_info_query` so it can be reused later.
                                netezza_count_info_query = f"""select count(*) from {db_tmp}.{table}"""
# Print a message or value to the notebook output for validation or debugging.
                                print('No 7 year filter value configured. Using full Netezza table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('No 7 year filter value configured. Using full Netezza table', style='List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                        else:     
# Assign the result on the right-hand side to `netezza_count_info_query` so it can be reused later.
                            netezza_count_info_query = f"""select count(*) from {db_tmp}.{table}"""
# Print a message or value to the notebook output for validation or debugging.
                            print('No 7 year filter column configured. Using full Netezza table')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph('No 7 year filter column configured. Using full Netezza table', style='List Bullet')  
# Run the fallback branch when the earlier conditions do not match.
                    else:
# Assign the result on the right-hand side to `netezza_count_info_query` so it can be reused later.
                        netezza_count_info_query = f"""select count(*) from {db_tmp}.{table}""" 
# Print a message or value to the notebook output for validation or debugging.
                        print('Using full Netezza table')
# Add a paragraph to the Word document so the log captures another message or detail.
                        doc.add_paragraph('Using full Netezza table', style='List Bullet')

# Original notebook comment retained.
                    ## Get number of rows in the table   
# Assign the result on the right-hand side to `netezza_df_count` so it can be reused later.
                    netezza_df_count=readfromNetizza(Env,netezza_count_info_query,database)
# Assign the result on the right-hand side to `netezza_side_no_of_rows` so it can be reused later.
                    netezza_side_no_of_rows = netezza_df_count.collect()[0][0]

# Start a loop so the same logic is applied repeatedly across multiple items.
                    for i in range(0, len(netezza_side_columns)):
# Assign the result on the right-hand side to `netezza_metadata_row` so it can be reused later.
                        netezza_metadata_row = [database, table, netezza_side_columns[i], netezza_side_datatype[i], netezza_ordinal_position[i]]
# Execute this line as part of the notebook's workflow logic.
                        netezza_metadata_1.append(netezza_metadata_row)

# Print a message or value to the notebook output for validation or debugging.
                    print(f"Successfully generate Netezza metadata info for {db_tmp}.{table}")
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f"Successfully generate Netezza metadata info for {db_tmp}.{table}", style='List Bullet')

# Handle an error raised in the preceding try block.
                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                    print(f"Failed to generate Netezza metadata info for {db_tmp}.{table}")
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f"Failed to generate Netezza metadata info for {db_tmp}.{table}", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'{e}', style='Intense Quote')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                try:
# Original notebook comment retained.
                    ## SYNAPSE METADATA 
# Assign the result on the right-hand side to `synapse_query_for_columns_info` so it can be reused later.
                    synapse_query_for_columns_info = f"""select column_name, data_type, ordinal_position from INFORMATION_SCHEMA.COLUMNS where table_schema= '{database}' and table_name='{table}'""" 
# Original notebook comment retained.
                                                    # query can be customized
# Assign the result on the right-hand side to `synapse_columns_info` so it can be reused later.
                    synapse_columns_info=readfromSynapse(Env,synapse_query_for_columns_info)

# Original notebook comment retained.
                    ## List of columns in the table
# Assign the result on the right-hand side to `synapse_side_columns` so it can be reused later.
                    synapse_side_columns = [result['column_name'] for result in synapse_columns_info.select('*').collect()]
# Original notebook comment retained.
                    ## List of datatype for each column in table
# Assign the result on the right-hand side to `synapse_side_datatype` so it can be reused later.
                    synapse_side_datatype = [result['data_type'] for result in synapse_columns_info.select('*').collect()]
# Original notebook comment retained.
                    ## Create list of ordinal position
# Assign the result on the right-hand side to `synapse_ordinal_position` so it can be reused later.
                    synapse_ordinal_position = [result['ordinal_position'] for result in synapse_columns_info.select('*').collect()]

# Assign the result on the right-hand side to `synapse_count_info_query` so it can be reused later.
                    synapse_count_info_query = f"""select count(*) as record_count from {database}.{table}"""
# Original notebook comment retained.
                    #print(synapse_count_info_query)
# Assign the result on the right-hand side to `synapse_df_count` so it can be reused later.
                    synapse_df_count =readfromSynapse(Env,synapse_count_info_query)
# Assign the result on the right-hand side to `synapse_side_no_of_rows` so it can be reused later.
                    synapse_side_no_of_rows = synapse_df_count.select(['record_count']).collect()[0][0]

# Start a loop so the same logic is applied repeatedly across multiple items.
                    for i in range(0, len(synapse_side_columns)):
# Assign the result on the right-hand side to `synapse_metadata_row` so it can be reused later.
                        synapse_metadata_row = [database, table, synapse_side_columns[i], synapse_side_datatype[i], synapse_ordinal_position[i]]
# Execute this line as part of the notebook's workflow logic.
                        synapse_metadata.append(synapse_metadata_row)

# Print a message or value to the notebook output for validation or debugging.
                    print(f"Successfully generate Synapse metadata info for {database}.{table}")
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f"Successfully generate Synapse metadata info for {database}.{table}", style='List Bullet')

# Handle an error raised in the preceding try block.
                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                    print(f"Failed to generate Synapse metadata info for {database}.{table}")
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f"Failed to generate Synapse metadata info for {database}.{table}", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'{e}', style='Intense Quote')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                try:
# Original notebook comment retained.
                    ## GOLD METADATA
# Read data from storage into a DataFrame or Python object.
                    gold_columns_info=spark.read.format("delta").option("header","true").load("/mnt/gold/"+database+"/ADMIN/"+table) 
# Execute this line as part of the notebook's workflow logic.
                    gold_columns_info.createOrReplaceTempView("gold_columns_info")
# Original notebook comment retained.
                    ## List of columns in the table
# Assign the result on the right-hand side to `gold_side_columns` so it can be reused later.
                    gold_side_columns = [result[0] for result in gold_columns_info.dtypes]
# Original notebook comment retained.
                    #print(col)
# Original notebook comment retained.
                    ## List of datatype for each column in table
# Assign the result on the right-hand side to `gold_side_datatype` so it can be reused later.
                    gold_side_datatype = [result[1] for result in gold_columns_info.dtypes]
# Original notebook comment retained.
                    #print(datatype)

# Assign the result on the right-hand side to `gold_df_count` so it can be reused later.
                    gold_df_count =spark.sql(f"select count(*) as record_count from gold_columns_info")
# Assign the result on the right-hand side to `gold_side_no_of_rows` so it can be reused later.
                    gold_side_no_of_rows = gold_df_count.select(['record_count']).collect()[0][0]

# Start a loop so the same logic is applied repeatedly across multiple items.
                    for i in range(0, len(gold_side_columns)):
# Assign the result on the right-hand side to `gold_metadata_row` so it can be reused later.
                        gold_metadata_row = [database, table, gold_side_columns[i], gold_side_datatype[i]]
# Execute this line as part of the notebook's workflow logic.
                        gold_metadata.append(gold_metadata_row)

# Print a message or value to the notebook output for validation or debugging.
                    print(f"Successfully generate Gold metadata info for {database}.{table}")
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f"Successfully generate Gold metadata info for {database}.{table}", style='List Bullet')

# Handle an error raised in the preceding try block.
                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                    print(f"Failed to generate Gold metadata info for {database}.{table}")
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f"Failed to generate Gold metadata info for {database}.{table}", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'{e}', style='Intense Quote')


# Original notebook comment retained.
                ## SUMMARY REPORT
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                try:
# Assign the result on the right-hand side to `table_report` so it can be reused later.
                    table_report=[]                
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(database); table_report.append(table); 
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(database);table_report.append(table);
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(list(set(netezza_side_columns)-set(synapse_side_columns))); # comparing columns column in netezza not in synapse
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(list(set(synapse_side_columns)-set(netezza_side_columns))); # column in synapse but not in netezza
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(str(netezza_side_no_of_rows)); 
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(str(synapse_side_no_of_rows));

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                    if (PL_TL == '7GS') | (Load_Type != 'Historical'):
# Execute this line as part of the notebook's workflow logic.
                        table_report.append(str(netezza_side_no_of_rows - synapse_side_no_of_rows));
# Run the fallback branch when the earlier conditions do not match.
                    else:
# Execute this line as part of the notebook's workflow logic.
                        table_report.append(str(gold_side_no_of_rows));
# Execute this line as part of the notebook's workflow logic.
                        table_report.append(str(netezza_side_no_of_rows - synapse_side_no_of_rows))
# Execute this line as part of the notebook's workflow logic.
                        table_report.append(str(netezza_side_no_of_rows - gold_side_no_of_rows));


# Execute this line as part of the notebook's workflow logic.
                    report_dataframe_info_1.append(table_report)
# Print a message or value to the notebook output for validation or debugging.
                    print(f'Successfully generated summary report for {database}.{table}')
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'Successfully generated summary report for {database}.{table}', style='List Bullet')        

# Handle an error raised in the preceding try block.
                except KeyError:
# Assign the result on the right-hand side to `table_report` so it can be reused later.
                    table_report=[]
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(database); table_report.append(table);
# Execute this line as part of the notebook's workflow logic.
                    table_report.append('Missing');table_report.append('Missing in SYNAPSE');
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(None); #difference in row count of SAP and ADLS-DELTA Table
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(None);
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(str(netezza_side_dictionary[database][table]['no_of_rows'])); 
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(''); 

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                    if (PL_TL == '7GS') | (Load_Type != 'Historical'):
# Execute this line as part of the notebook's workflow logic.
                        table_report.append('FAILED');
# Run the fallback branch when the earlier conditions do not match.
                    else:
# Execute this line as part of the notebook's workflow logic.
                        table_report.append(''); 
# Execute this line as part of the notebook's workflow logic.
                        table_report.append('FAILED');
# Execute this line as part of the notebook's workflow logic.
                        table_report.append('FAILED');


# Execute this line as part of the notebook's workflow logic.
                    report_dataframe_info_1.append(table_report)
# Print a message or value to the notebook output for validation or debugging.
                    print(f'Failed to generate summary report for {database}.{table}')
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'Failed to generate summary report for {database}.{table}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                    print(KeyError)
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'{KeyError}')

# Assign the result on the right-hand side to `tab +` so it can be reused later.
                tab += 1
# Print a message or value to the notebook output for validation or debugging.
                print(tab)




# Assign the result on the right-hand side to `netezza_metadata_dataframe_1` so it can be reused later.
        netezza_metadata_dataframe_1 = spark.createDataFrame(netezza_metadata_1,netezza_metadata_schema)
# Assign the result on the right-hand side to `netezza_metadata_dataframe_pd_1` so it can be reused later.
        netezza_metadata_dataframe_pd_1 = netezza_metadata_dataframe_1.toPandas()

# Assign the result on the right-hand side to `synapse_metadata_dataframe` so it can be reused later.
        synapse_metadata_dataframe = spark.createDataFrame(synapse_metadata,synapse_metadata_schema)
# Assign the result on the right-hand side to `synapse_metadata_dataframe_pd` so it can be reused later.
        synapse_metadata_dataframe_pd = synapse_metadata_dataframe.toPandas()

# Assign the result on the right-hand side to `gold_metadata_dataframe` so it can be reused later.
        gold_metadata_dataframe = spark.createDataFrame(gold_metadata,gold_metadata_schema)
# Assign the result on the right-hand side to `gold_metadata_dataframe_pd` so it can be reused later.
        gold_metadata_dataframe_pd = gold_metadata_dataframe.toPandas()

# Assign the result on the right-hand side to `report_dataframe_1` so it can be reused later.
        report_dataframe_1=spark.createDataFrame(report_dataframe_info_1,report_dataframe_schema_1).withColumn("Rpt_Exec_Ts",sf.current_timestamp())
# Assign the result on the right-hand side to `report_dataframe_pd_1` so it can be reused later.
        report_dataframe_pd_1 = report_dataframe_1.toPandas()


# Original notebook comment retained.
    ## Summary report for netezza v. silver tables
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if len(relevant_silver_db) > 0:

# Assign the result on the right-hand side to `netezza_metadata_2` so it can be reused later.
        netezza_metadata_2 = []
# Assign the result on the right-hand side to `silver_metadata` so it can be reused later.
        silver_metadata = []
# Assign the result on the right-hand side to `report_dataframe_info_2` so it can be reused later.
        report_dataframe_info_2=[]


# Start a loop so the same logic is applied repeatedly across multiple items.
        for database in list(relevant_silver_db):

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
            if Env == 'QA':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = f"GDC_{database}"
# Check the next condition if the previous condition was not met.
            elif Env == 'PROD_DR':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = f"{database}_DR"
# Run the fallback branch when the earlier conditions do not match.
            else:
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = database

# Original notebook comment retained.
            #print(database)
# Execute this line as part of the notebook's workflow logic.
            tables=[v for d in validation_tables_info for k,v in d.items() if k == database][0]
# Start a loop so the same logic is applied repeatedly across multiple items.
            for table in tables:
# Print a message or value to the notebook output for validation or debugging.
                print(f'{database}.{table}')

# Original notebook comment retained.
                ## Header for log document
# Add a heading to the Word document being used as a log or report.
                doc.add_heading(f'{database}.{table}', 3)

# Start a new indented code block for the statement above.
                try : 
# Original notebook comment retained.
                    ## NETEZZA METADATA
# Assign the result on the right-hand side to `query_for_columns_info` so it can be reused later.
                    query_for_columns_info = f"""select attname,format_type,attnum from  {db_tmp}._V_RELATION_COLUMN WHERE NAME = '{table}'""" 
# Assign the result on the right-hand side to `netezza_df` so it can be reused later.
                    netezza_df=readfromNetizza(Env,query_for_columns_info,database)
# Original notebook comment retained.
                    ## List of columns in the table
# Assign the result on the right-hand side to `netezza_side_columns` so it can be reused later.
                    netezza_side_columns = [row[0] for row in netezza_df.select('attname').collect()]  #netezza_df.rdd.map(lambda x: x[0]).collect()
# Original notebook comment retained.
                    ## List of datatype for each column in table
# Assign the result on the right-hand side to `netezza_side_datatype` so it can be reused later.
                    netezza_side_datatype= [row[0] for row in netezza_df.select('format_type').collect()]
# Original notebook comment retained.
                    ## List of ordinal position
# Assign the result on the right-hand side to `netezza_ordinal_position` so it can be reused later.
                    netezza_ordinal_position = [row[0] for row in netezza_df.select('attnum').collect()]

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                    if (PL_TL == '7GS'):
# Original notebook comment retained.
                        ## Retrieve the filter column for the table. This is a date column used to identify the minimum date in which data from in table was loaded from Netizza
# Execute this line as part of the notebook's workflow logic.
                        filterColumn=gold_df.select('Filter7').where((gold_df.TABLE_NAME==table)&(gold_df.DATABASE==database)).distinct()  #.rdd.map(lambda x: x.Filter7).collect()
# Assign the result on the right-hand side to `filterColumn` so it can be reused later.
                        filterColumn = [row[0] for row in filterColumn.collect()]
# Original notebook comment retained.
                        #print(filterColumn)
# Assign the result on the right-hand side to `filterColumn` so it can be reused later.
                        filterColumn=str(filterColumn)
# Assign the result on the right-hand side to `filterColumn` so it can be reused later.
                        filterColumn=filterColumn.replace("[","").replace("]","").replace("'","")
# Original notebook comment retained.
                        #print('filterCol', filterColumn)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if filterColumn!='None':
# Original notebook comment retained.
                            #print('filterCol2:', filterColumn) 
# Original notebook comment retained.
                            ## Fetch the minimum date from synapse to avoid mismatch on the day where data is not loaded in netizza
# Assign the result on the right-hand side to `query_forValuefilter` so it can be reused later.
                            query_forValuefilter= f"""select CAST(min({filterColumn}) as varchar(10)) {filterColumn} from {database}.{table}""" 
# Original notebook comment retained.
                            #print(query_forValuefilter)
# Assign the result on the right-hand side to `filterValue1` so it can be reused later.
                            filterValue1=readfromSynapse(Env,query_forValuefilter)
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                            filterValue=[row[0] for row in filterValue1.collect()]  #filterValue1.rdd.map(lambda x: x[0]).collect()
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                            filterValue=str(filterValue)
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                            filterValue=filterValue.replace("[","").replace("]","")
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if filterValue!='None':
# Assign the result on the right-hand side to `netezza_count_info_query` so it can be reused later.
                                netezza_count_info_query = f"""select count(*) from {db_tmp}.{table} where {filterColumn} >= {filterValue}"""
# Print a message or value to the notebook output for validation or debugging.
                                print('filtering Netezza for 7 years only')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('filtering Netezza for 7 years only')
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `netezza_count_info_query` so it can be reused later.
                                netezza_count_info_query = f"""select count(*) from {db_tmp}.{table}"""
# Print a message or value to the notebook output for validation or debugging.
                                print('No 7 year filter value configured. Using full Netezza table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('No 7 year filter value configured. Using full Netezza table', style='List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                        else:     
# Assign the result on the right-hand side to `netezza_count_info_query` so it can be reused later.
                            netezza_count_info_query = f"""select count(*) from {db_tmp}.{table}"""  
# Print a message or value to the notebook output for validation or debugging.
                            print('No 7 year filter column configured. Using full Netezza table')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph('No 7 year filter column configured. Using full Netezza table', style='List Bullet')  
# Run the fallback branch when the earlier conditions do not match.
                    else:
# Assign the result on the right-hand side to `netezza_count_info_query` so it can be reused later.
                        netezza_count_info_query = f"""select count(*) from {db_tmp}.{table}""" 
# Print a message or value to the notebook output for validation or debugging.
                        print('Using full Netezza table')
# Add a paragraph to the Word document so the log captures another message or detail.
                        doc.add_paragraph('Using full Netezza table', style='List Bullet')

# Original notebook comment retained.
                    ## Get number of rows in the table   
# Assign the result on the right-hand side to `netezza_df_count` so it can be reused later.
                    netezza_df_count=readfromNetizza(Env,netezza_count_info_query,database)
# Assign the result on the right-hand side to `netezza_side_no_of_rows` so it can be reused later.
                    netezza_side_no_of_rows = netezza_df_count.collect()[0][0]

# Start a loop so the same logic is applied repeatedly across multiple items.
                    for i in range(0, len(netezza_side_columns)):
# Assign the result on the right-hand side to `netezza_metadata_row` so it can be reused later.
                        netezza_metadata_row = [database, table, netezza_side_columns[i], netezza_side_datatype[i], netezza_ordinal_position[i]]
# Execute this line as part of the notebook's workflow logic.
                        netezza_metadata_2.append(netezza_metadata_row)

# Print a message or value to the notebook output for validation or debugging.
                    print(f"Successfully generate Netezza metadata info for {db_tmp}.{table}")
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f"Successfully generate Netezza metadata info for {db_tmp}.{table}", style='List Bullet')
# Handle an error raised in the preceding try block.
                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                    print(f"Failed to generate Netezza metadata info for {db_tmp}.{table}")
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f"Failed to generate Netezza metadata info for {db_tmp}.{table}", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'{e}', style='Intense Quote')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                try:
# Original notebook comment retained.
                    ## SILVER METADATA
# Read data from storage into a DataFrame or Python object.
                    silver_columns_info=spark.read.format("delta").option("header","true").load("/mnt/silver/"+database+"/ADMIN/"+table) 
# Execute this line as part of the notebook's workflow logic.
                    silver_columns_info.createOrReplaceTempView("silver_columns_info")
# Original notebook comment retained.
                    ## List of columns in the table
# Assign the result on the right-hand side to `silver_side_columns` so it can be reused later.
                    silver_side_columns = [result[0] for result in silver_columns_info.dtypes]
# Original notebook comment retained.
                    #print(col)
# Original notebook comment retained.
                    ## List of datatype for each column in table
# Assign the result on the right-hand side to `silver_side_datatype` so it can be reused later.
                    silver_side_datatype = [result[1] for result in silver_columns_info.dtypes]
# Original notebook comment retained.
                    #print(datatype)

# Assign the result on the right-hand side to `silver_df_count` so it can be reused later.
                    silver_df_count =spark.sql(f"select count(*) as record_count from silver_columns_info")
# Assign the result on the right-hand side to `silver_side_no_of_rows` so it can be reused later.
                    silver_side_no_of_rows = silver_df_count.select(['record_count']).collect()[0][0]

# Start a loop so the same logic is applied repeatedly across multiple items.
                    for i in range(0, len(silver_side_columns)):
# Assign the result on the right-hand side to `silver_metadata_row` so it can be reused later.
                        silver_metadata_row = [database, table, silver_side_columns[i], silver_side_datatype[i]]
# Execute this line as part of the notebook's workflow logic.
                        silver_metadata.append(silver_metadata_row)

# Print a message or value to the notebook output for validation or debugging.
                    print(f"Successfully generate Silver metadata info for {database}.{table}")
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f"Successfully generate Silver metadata info for {database}.{table}", style='List Bullet')
# Handle an error raised in the preceding try block.
                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                    print(f"Failed to generate Silver metadata info for {database}.{table}")
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f"Failed to generate Silver metadata info for {database}.{table}", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'{e}', style='Intense Quote')


# Begin a protected block so the notebook can handle runtime errors more gracefully.
                try:
# Assign the result on the right-hand side to `table_report` so it can be reused later.
                    table_report=[]
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(database); table_report.append(table); 
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(database);table_report.append(table);
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(list(set(netezza_side_columns)-set(silver_side_columns)));
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(list(set(silver_side_columns)-set(netezza_side_columns)));
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(str(netezza_side_no_of_rows)); 
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(str(silver_side_no_of_rows));
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(str(netezza_side_no_of_rows - silver_side_no_of_rows))

# Execute this line as part of the notebook's workflow logic.
                    report_dataframe_info_2.append(table_report)
# Print a message or value to the notebook output for validation or debugging.
                    print(f'Successfully generated summary report for {database}.{table}')
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'Successfully generated summary report for {database}.{table}', style='List Bullet') 

# Handle an error raised in the preceding try block.
                except KeyError:
# Assign the result on the right-hand side to `table_report` so it can be reused later.
                    table_report=[]
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(database); table_report.append(table);
# Execute this line as part of the notebook's workflow logic.
                    table_report.append('Missing');table_report.append('Missing in SILVER');
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(None); #difference in row count of SAP and ADLS-DELTA Table
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(None);
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(str(netezza_side_dictionary[database][table]['no_of_rows'])); 
# Execute this line as part of the notebook's workflow logic.
                    table_report.append(''); 
# Execute this line as part of the notebook's workflow logic.
                    table_report.append('FAILED')

# Execute this line as part of the notebook's workflow logic.
                    report_dataframe_info_2.append(table_report)
# Print a message or value to the notebook output for validation or debugging.
                    print(f'Failed to generate summary report for {database}.{table}')
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'Failed to generate summary report for {database}.{table}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                    print(KeyError)
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'{KeyError}') 

# Assign the result on the right-hand side to `tab +` so it can be reused later.
                tab += 1
# Print a message or value to the notebook output for validation or debugging.
                print(tab)

# Assign the result on the right-hand side to `netezza_metadata_dataframe_2` so it can be reused later.
        netezza_metadata_dataframe_2 = spark.createDataFrame(netezza_metadata_2,netezza_metadata_schema)
# Assign the result on the right-hand side to `netezza_metadata_dataframe_pd_2` so it can be reused later.
        netezza_metadata_dataframe_pd_2 = netezza_metadata_dataframe_2.toPandas()

# Assign the result on the right-hand side to `silver_metadata_dataframe` so it can be reused later.
        silver_metadata_dataframe = spark.createDataFrame(silver_metadata,silver_metadata_schema)
# Assign the result on the right-hand side to `silver_metadata_dataframe_pd` so it can be reused later.
        silver_metadata_dataframe_pd = silver_metadata_dataframe.toPandas()

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
        if Load_Type == 'Historical':
# Assign the result on the right-hand side to `report_dataframe_2` so it can be reused later.
            report_dataframe_2=spark.createDataFrame(report_dataframe_info_2,report_dataframe_schema_2).withColumn("Rpt_Exec_Ts",sf.current_timestamp())
# Assign the result on the right-hand side to `report_dataframe_pd_2` so it can be reused later.
            report_dataframe_pd_2 = report_dataframe_2.toPandas()







# Original notebook comment retained.
    ## Define excel writer
# Begin a protected block so the notebook can handle runtime errors more gracefully.
    try:
# Open a context-managed resource so it is handled safely and closed automatically after use.
        with pd.ExcelWriter("METADATA_INFO.xlsx", engine="openpyxl", mode = 'w') as writer: 
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
            if len(relevant_gold_db) > 0:
# Original notebook comment retained.
                ## Write metadata info for Netezza, Synapse and Gold to sheet in excel workbook
# Assign the result on the right-hand side to `sheet_name` so it can be reused later.
                sheet_name =  "NETEZZA v. SYNAPSE v. GOLD"
# Assign the result on the right-hand side to `netezza_metadata_dataframe_pd_1.to_excel(writer, sheet_name` so it can be reused later.
                netezza_metadata_dataframe_pd_1.to_excel(writer, sheet_name = sheet_name, startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `synapse_metadata_dataframe_pd.to_excel(writer, sheet_name` so it can be reused later.
                synapse_metadata_dataframe_pd.to_excel(writer, sheet_name = sheet_name, startrow = 0, startcol=6, index = False)
# Assign the result on the right-hand side to `gold_metadata_dataframe_pd.to_excel(writer, sheet_name` so it can be reused later.
                gold_metadata_dataframe_pd.to_excel(writer, sheet_name = sheet_name, startrow = 0, startcol=12, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
            if len(relevant_silver_db) > 0:
# Original notebook comment retained.
                ## Write metadata info for Netezza and Silver to sheet in excel workbook
# Assign the result on the right-hand side to `sheet_name` so it can be reused later.
                sheet_name =  "NETEZZA v. SILVER"
# Assign the result on the right-hand side to `netezza_metadata_dataframe_pd_2.to_excel(writer, sheet_name` so it can be reused later.
                netezza_metadata_dataframe_pd_2.to_excel(writer, sheet_name = sheet_name, startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `silver_metadata_dataframe_pd.to_excel(writer, sheet_name` so it can be reused later.
                silver_metadata_dataframe_pd.to_excel(writer, sheet_name = sheet_name, startrow = 0, startcol=6, index = False)

# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
        workbook_path = path + 'METADATA_INFO.xlsx'            
# Execute this line as part of the notebook's workflow logic.
        shutil.copy2('METADATA_INFO.xlsx',workbook_path)
# Print a message or value to the notebook output for validation or debugging.
        print(f"Successfully uploaded metadata info to path {workbook_path.replace('/dbfs/mnt', '')}")
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph(f"Successfully uploaded metadata info to path {workbook_path.replace('/dbfs/mnt', '')}", style='List Bullet')
# Execute this line as part of the notebook's workflow logic.
        os.remove('METADATA_INFO.xlsx')

# Handle an error raised in the preceding try block.
    except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
        print(f"Failed to upload metadata info")
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph(f"Failed to upload metadata info", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph(f'{e}', style='Intense Quote')


# Begin a protected block so the notebook can handle runtime errors more gracefully.
    try:
# Open a context-managed resource so it is handled safely and closed automatically after use.
        with pd.ExcelWriter("SUMMARY_REPORT.xlsx", engine="openpyxl", mode = 'w') as writer1:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
            if len(relevant_gold_db) > 0:
# Original notebook comment retained.
                ## Write summary report for Netezza, Synapse and Gold to sheet in excel workbook
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                if (PL_TL == '7GS') | (Load_Type != 'Historical'):
# Assign the result on the right-hand side to `sheet_name` so it can be reused later.
                    sheet_name =  "NETEZZA v. SYNAPSE"
# Run the fallback branch when the earlier conditions do not match.
                else:
# Assign the result on the right-hand side to `sheet_name` so it can be reused later.
                    sheet_name =  "NETEZZA v. SYNAPSE v. GOLD"
# Assign the result on the right-hand side to `report_dataframe_pd_1.to_excel(writer1, sheet_name` so it can be reused later.
                report_dataframe_pd_1.to_excel(writer1, sheet_name = sheet_name, startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
            if (Load_Type == 'Historical') & (len(relevant_silver_db) > 0):
# Original notebook comment retained.
                ## Write summary report for Netezza and Silver to sheet in excel workbook
# Assign the result on the right-hand side to `sheet_name` so it can be reused later.
                sheet_name =  "NETEZZA v. SILVER"
# Assign the result on the right-hand side to `report_dataframe_pd_2.to_excel(writer1, sheet_name` so it can be reused later.
                report_dataframe_pd_2.to_excel(writer1, sheet_name = sheet_name, startrow = 0, startcol=0, index = False)

# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
        workbook_path = path + 'SUMMARY_REPORT.xlsx'            
# Execute this line as part of the notebook's workflow logic.
        shutil.copy2('SUMMARY_REPORT.xlsx',workbook_path)
# Print a message or value to the notebook output for validation or debugging.
        print(f"Successfully uploaded summary report to path {workbook_path.replace('/dbfs/mnt', '')}")
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph(f"Successfully uploaded summary report to path {workbook_path.replace('/dbfs/mnt', '')}", style='List Bullet')
# Execute this line as part of the notebook's workflow logic.
        os.remove('SUMMARY_REPORT.xlsx')

# Handle an error raised in the preceding try block.
    except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
        print(f"Failed to upload summary report")
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph(f"Failed to upload summary report", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph(f'{e}', style='Intense Quote')




# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph('')

# Original notebook comment retained.
    ## Upload log file
# Execute this line as part of the notebook's workflow logic.
    save_log(doc)

# Handle an error raised in the preceding try block.
except Exception as e:
# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph("Data Validation Run Failed", style="List Bullet")
# Print a message or value to the notebook output for validation or debugging.
    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph(f'{e}', style="Intense Quote")
# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph('')

# Original notebook comment retained.
    ## Upload log file
# Execute this line as part of the notebook's workflow logic.
    save_log(doc)

# Original notebook comment retained.
    ## Exit the notebook run
# Execute this line as part of the notebook's workflow logic.
    dbutils.notebook.exit("Data Validation Run Failed") 




## Command cell 22

This section corresponds to command 22 from the original Databricks notebook.


In [ ]:
# Import specific objects from a module so they can be used directly in later code: from datetime import datetime.
from datetime import datetime
# Capture the current timestamp so the notebook can stamp logs, paths, or outputs.
now = datetime.now()
# Format a date or time value into a string representation.
now.strftime("%H:%M:%S")

## Command cell 23

This section corresponds to command 23 from the original Databricks notebook.


In [ ]:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if RptType=='Summary Report':

# Original notebook comment retained.
    ## Upload the log file
# Save the current object or output to the specified destination.
    doc.save('log.docx')
# Execute this line as part of the notebook's workflow logic.
    shutil.copy2('log.docx',path+'log.docx')
# Execute this line as part of the notebook's workflow logic.
    os.remove('log.docx') 

# Original notebook comment retained.
    ## Exit the notebook run
# Execute this line as part of the notebook's workflow logic.
    dbutils.notebook.exit(RptType) 

## Command cell 24

This section corresponds to command 24 from the original Databricks notebook.


In [ ]:
# Original notebook comment retained.
## Instantiate list of dictionaries
# Assign the result on the right-hand side to `validation_tables_info_2` so it can be reused later.
validation_tables_info_2 = []

# Start a loop so the same logic is applied repeatedly across multiple items.
for db in validation_databases:  
# Original notebook comment retained.
    #print(db)
# Assign the result on the right-hand side to `dict_schema` so it can be reused later.
    dict_schema={}
# Execute this line as part of the notebook's workflow logic.
    list_tables = [row.TABLE_NAME for row in gold_df.select(['TABLE_NAME']).where(gold_df.DATABASE == db).distinct().collect()]
# Assign the result on the right-hand side to `indi_schema_list` so it can be reused later.
    indi_schema_list=[]
# Start a loop so the same logic is applied repeatedly across multiple items.
    for table in list_tables:
# Assign the result on the right-hand side to `dict_table` so it can be reused later.
        dict_table={}
# Assign the result on the right-hand side to `list_primary_key` so it can be reused later.
        list_primary_key=[row.KEY_COLUMN for row in gold_df.select(['KEY_COLUMN'])\
# Execute this line as part of the notebook's workflow logic.
                                .where((gold_df.TABLE_NAME==table) & (gold_df.DATABASE == db)).distinct().collect()]
# Assign the result on the right-hand side to `dict_table[table]` so it can be reused later.
        dict_table[table]=list_primary_key
# Execute this line as part of the notebook's workflow logic.
        indi_schema_list.append(dict_table)
# Assign the result on the right-hand side to `dict_schema[db]` so it can be reused later.
    dict_schema[db]=indi_schema_list
# Assign the result on the right-hand side to `cp_dict_schema` so it can be reused later.
    cp_dict_schema=dict_schema.copy()

# Execute this line as part of the notebook's workflow logic.
    validation_tables_info_2.append(cp_dict_schema)
# Print a message or value to the notebook output for validation or debugging.
print(validation_tables_info_2)

## Command cell 25

This section corresponds to command 25 from the original Databricks notebook.


In [ ]:
# Execute this line as part of the notebook's workflow logic.
gold_df.createOrReplaceTempView("gold_df")
# Render the object in the notebook output so the user can inspect it visually.
display(gold_df)

## Command cell 26

This section corresponds to command 26 from the original Databricks notebook.


In [ ]:
# Import the module(s) required for the next part of the workflow: pandas as pd.
import pandas as pd 
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.functions import col.
from pyspark.sql.functions import col
# Import the module(s) required for the next part of the workflow: pyspark.pandas as ps.
import pyspark.pandas as ps

# Define the function `get_pandas` so this block can be reused later in the notebook.
def get_pandas(spark_df):
# Original notebook comment retained.
    #find all decimal columns in your SparkDF
# Assign the result on the right-hand side to `decimals_cols` so it can be reused later.
    decimals_cols = [c for c in spark_df.columns if 'Decimal' in str(spark_df.schema[c].dataType)]
# Assign the result on the right-hand side to `timestamp_cols` so it can be reused later.
    timestamp_cols = [c for c in spark_df.columns if 'Timestamp' in str(spark_df.schema[c].dataType)]

# Original notebook comment retained.
    #convert all decimals columns to floats
# Start a loop so the same logic is applied repeatedly across multiple items.
    for col in decimals_cols:
# Assign the result on the right-hand side to `spark_df` so it can be reused later.
        spark_df = spark_df.withColumn(col, spark_df[col].cast(FloatType()))

# Original notebook comment retained.
    #Convert all timestamp columns to string
# Start a loop so the same logic is applied repeatedly across multiple items.
    for col in timestamp_cols:
# Assign the result on the right-hand side to `spark_df` so it can be reused later.
        spark_df = spark_df.withColumn(col, spark_df[col].cast(StringType()))

# Original notebook comment retained.
    #pd_df = spark_df.toPandas()
# Assign the result on the right-hand side to `pd_df` so it can be reused later.
    pd_df = ps.DataFrame(spark_df)
# Return a value from the current function back to the caller.
    return pd_df

## Command cell 27

This section corresponds to command 27 from the original Databricks notebook.


In [ ]:
# Define the function `trim_space` so this block can be reused later in the notebook.
def trim_space(spark_df):
# Assign the result on the right-hand side to `string_cols` so it can be reused later.
    string_cols = [c for c in spark_df.columns if 'String' in str(spark_df.schema[c].dataType)]

# Start a loop so the same logic is applied repeatedly across multiple items.
    for col in string_cols:
# Assign the result on the right-hand side to `spark_df` so it can be reused later.
        spark_df = spark_df.withColumn(col, trim(col))

# Return a value from the current function back to the caller.
    return spark_df 

## Command cell 28

This section corresponds to command 28 from the original Databricks notebook.


In [ ]:
# Import the module(s) required for the next part of the workflow: pyspark.sql.functions as F.
import pyspark.sql.functions as F
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql import Row.
from pyspark.sql import Row
# Import the module(s) required for the next part of the workflow: datetime.
import datetime
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.types import StructType, StructField, StringType.
from pyspark.sql.types import StructType, StructField, StringType
# Import the module(s) required for the next part of the workflow: pyspark.pandas as ps.
import pyspark.pandas as ps 


# Define the function `find_Missing` so this block can be reused later in the notebook.
def find_Missing(data1, data2, id_columns): 
# Assign the result on the right-hand side to `data1` so it can be reused later.
    data1 = data1.select( *[ F.when(F.col(column).isNull(),'').otherwise(F.col(column)).alias(column) for column in data1.columns])
# Assign the result on the right-hand side to `data2` so it can be reused later.
    data2 = data2.select( *[ F.when(F.col(column).isNull(),'').otherwise(F.col(column)).alias(column) for column in data2.columns])

# Assign the result on the right-hand side to `cols` so it can be reused later.
    cols = data1.columns

# Print a message or value to the notebook output for validation or debugging.
    print('WORKING ON MISSING RECORDS')
# Assign the result on the right-hand side to `append_str` so it can be reused later.
    append_str='tgt_'
# Assign the result on the right-hand side to `pre_res` so it can be reused later.
    pre_res = [append_str + sub for sub in id_columns]
# Assign the result on the right-hand side to `pre_res1` so it can be reused later.
    pre_res1 = [append_str + sub for sub in cols]
# Assign the result on the right-hand side to `data_difference_1` so it can be reused later.
    data_difference_1 = data1.subtract(data2)
# Execute this line as part of the notebook's workflow logic.
    data_difference_1.cache()
# Original notebook comment retained.
    #join_on_col = list(set(id_columns).intersection(set(data_difference_1.columns)))
# Assign the result on the right-hand side to `data_difference_1` so it can be reused later.
    data_difference_1 = data_difference_1.join(data2.select(*[F.col(i).alias(f'tgt_{i}') if i not in id_columns else F.col(i).alias(i) for i in cols]), on = id_columns, how = 'left')
# Execute this line as part of the notebook's workflow logic.
    data_difference_1.cache()

# Assign the result on the right-hand side to `data_difference_1` so it can be reused later.
    data_difference_1 = Missing_condition(pre_res1,pre_res,data_difference_1)
# Execute this line as part of the notebook's workflow logic.
    data_difference_1.cache()
# Original notebook comment retained.
    #counts = data_difference_1.select([F.count(i).alias(i) for i in data_difference_1.columns]).toPandas()
# Assign the result on the right-hand side to `counts` so it can be reused later.
    counts = ps.DataFrame(data_difference_1.select([F.count(i).alias(i) for i in data_difference_1.columns]))


# Assign the result on the right-hand side to `output` so it can be reused later.
    output = data_difference_1.select(*counts.columns[counts.ne(0).iloc[0]]).orderBy(id_columns)
# Execute this line as part of the notebook's workflow logic.
    output.cache()
# Assign the result on the right-hand side to `cnt` so it can be reused later.
    cnt=output.count() 
# Assign the result on the right-hand side to `output_pd` so it can be reused later.
    output_pd = get_pandas(output)   #.sort_values(by=id_columns)
# Execute this line as part of the notebook's workflow logic.
    output_pd.spark.cache()
# Execute this line as part of the notebook's workflow logic.
    '''
# Original notebook comment retained.
    ## TEMP - ONLY for TESTING LOGIC
# Assign the result on the right-hand side to `cnt` so it can be reused later.
    cnt = 0
# Original notebook comment retained.
    ## END
# Execute this line as part of the notebook's workflow logic.
    '''
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if cnt == 0:
# Assign the result on the right-hand side to `columns` so it can be reused later.
        columns = StructType([StructField('Missing Records',
# Execute this line as part of the notebook's workflow logic.
                                  StringType(), True)])
# Assign the result on the right-hand side to `output` so it can be reused later.
        output = spark.createDataFrame(data = [],
# Assign the result on the right-hand side to `schema` so it can be reused later.
                           schema = columns)
# Assign the result on the right-hand side to `output_pd` so it can be reused later.
        output_pd = get_pandas(output)
# Execute this line as part of the notebook's workflow logic.
        output_pd.spark.cache()
# Print a message or value to the notebook output for validation or debugging.
        print("No Missing Records Found")
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph("No Missing Records Found", style='List Bullet')
# Check the next condition if the previous condition was not met.
    elif cnt > max_excel_row:
# Write the current DataFrame or object to storage.
        output.write.format('delta').mode('overwrite').save(path.replace("/dbfs", '')+f"{container}_{db}_{table_name}_missing_records")


# Execute this line as part of the notebook's workflow logic.
    '''
# Original notebook comment retained.
    ## Convert to pandas dataframe
# Original notebook comment retained.
    #output = convertDecimalToFloat(output).toPandas()
# Assign the result on the right-hand side to `output` so it can be reused later.
    output = get_pandas(output.limit(max_excel_row))
# Execute this line as part of the notebook's workflow logic.
    output.spark.cache()
# Execute this line as part of the notebook's workflow logic.
    '''



# Return a value from the current function back to the caller.
    return cnt, output_pd



# Define the function `find_Mismatched` so this block can be reused later in the notebook.
def find_Mismatched(data1, data2, id_columns):
# Assign the result on the right-hand side to `data1` so it can be reused later.
    data1 = data1.select( *[ F.when(F.col(column).isNull(),'').otherwise(F.col(column)).alias(column) for column in data1.columns])
# Assign the result on the right-hand side to `data2` so it can be reused later.
    data2 = data2.select( *[ F.when(F.col(column).isNull(),'').otherwise(F.col(column)).alias(column) for column in data2.columns])

# Print a message or value to the notebook output for validation or debugging.
    print('WORKING ON MISMATCHED RECORDS')
# Assign the result on the right-hand side to `cols` so it can be reused later.
    cols = data1.columns
# Assign the result on the right-hand side to `data_difference` so it can be reused later.
    data_difference = data2.subtract(data1) # First you find the difference in 
# Execute this line as part of the notebook's workflow logic.
    data_difference.cache()
# Print a message or value to the notebook output for validation or debugging.
    print(data_difference.columns)
# Original notebook comment retained.
    #join_on_col = list(set(id_columns).intersection(set(data_difference.columns)))
# Print a message or value to the notebook output for validation or debugging.
    print('id_columns', id_columns)
# Assign the result on the right-hand side to `data_difference` so it can be reused later.
    data_difference = data_difference.join(data1.select(*[F.col(i).alias(f'SRC_{i}') if i not in id_columns else F.col(i).alias(i) for i in cols]), on = id_columns, how = 'left')
# Execute this line as part of the notebook's workflow logic.
    data_difference.cache()
# Assign the result on the right-hand side to `mismatched_record_df` so it can be reused later.
    mismatched_record_df = data_difference.alias("mismatched_record_df").orderBy(id_columns)
# Original notebook comment retained.
    #display(mismatched_record_df.limit(5))
# Execute this line as part of the notebook's workflow logic.
    mismatched_record_df.cache()
# Assign the result on the right-hand side to `cnt_diff` so it can be reused later.
    cnt_diff = mismatched_record_df.count()
# Assign the result on the right-hand side to `mismatched_record_df_pd` so it can be reused later.
    mismatched_record_df_pd = get_pandas(mismatched_record_df)   #.sort_values(by=id_columns)
# Execute this line as part of the notebook's workflow logic.
    mismatched_record_df_pd.spark.cache()
# Execute this line as part of the notebook's workflow logic.
    '''
# Original notebook comment retained.
    ## TEMP - ONLY for TESTING LOGIC
# Assign the result on the right-hand side to `cnt_diff` so it can be reused later.
    cnt_diff = 0
# Original notebook comment retained.
    ## END
# Execute this line as part of the notebook's workflow logic.
    '''
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if cnt_diff == 0:
# Assign the result on the right-hand side to `columns` so it can be reused later.
        columns = StructType([StructField('Mismatched Records', StringType(), True)])
# Assign the result on the right-hand side to `mismatched_record_df` so it can be reused later.
        mismatched_record_df = spark.createDataFrame(data = [], schema = columns)
# Assign the result on the right-hand side to `mismatched_record_df_pd` so it can be reused later.
        mismatched_record_df_pd = get_pandas(mismatched_record_df)
# Execute this line as part of the notebook's workflow logic.
        mismatched_record_df_pd.spark.cache()
# Print a message or value to the notebook output for validation or debugging.
        print("No Mismatched Records Found")
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph("No Mismatched Records Found", style='List Bullet')
# Check the next condition if the previous condition was not met.
    elif cnt_diff > max_excel_row:
# Write the current DataFrame or object to storage.
        mismatched_record_df.write.format('delta').mode("overwrite").save(path.replace("/dbfs", '')+f"{container}_{db}_{table_name}_mismatched_records")


# Print a message or value to the notebook output for validation or debugging.
    print('WORKING ON MISMATCHED RECORDS DETAILS')
# Original notebook comment retained.
    # you join the differences with the second dataframe
# Original notebook comment retained.
    #final_df=data_difference.select( *[ F.when(F.col(column).isNull(),'').otherwise(F.col(column)).alias(column) for column in data_difference.columns]).orderBy(id_columns)
# Original notebook comment retained.
    #display(final_df.limit(5))
# Original notebook comment retained.
    #final_df.cache()
# Assign the result on the right-hand side to `final_df` so it can be reused later.
    final_df=data_difference.select(*[is_different(x, f"SRC_{x}",data_difference.columns, data_difference) for x in cols]).orderBy(id_columns)
# Original notebook comment retained.
    #display(final_df.limit(5))
# Execute this line as part of the notebook's workflow logic.
    final_df.cache()
# Assign the result on the right-hand side to `cnt_final` so it can be reused later.
    cnt_final=final_df.count()
# Assign the result on the right-hand side to `final_df_pd` so it can be reused later.
    final_df_pd = get_pandas(final_df)   #.sort_values(by=id_columns)
# Execute this line as part of the notebook's workflow logic.
    final_df_pd.spark.cache()
# Execute this line as part of the notebook's workflow logic.
    '''
# Original notebook comment retained.
    ## TEMP - ONLY for TESTING LOGIC
# Assign the result on the right-hand side to `cnt_final` so it can be reused later.
    cnt_final = 0
# Original notebook comment retained.
    ## END
# Execute this line as part of the notebook's workflow logic.
    '''
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if cnt_final == 0:
# Assign the result on the right-hand side to `columns` so it can be reused later.
        columns = StructType([StructField('Mismatched Records Details', StringType(), True)])
# Assign the result on the right-hand side to `final_df` so it can be reused later.
        final_df = spark.createDataFrame(data = [], schema = columns)
# Assign the result on the right-hand side to `final_df_pd` so it can be reused later.
        final_df_pd = get_pandas(final_df)
# Execute this line as part of the notebook's workflow logic.
        final_df_pd.spark.cache()
# Print a message or value to the notebook output for validation or debugging.
        print("No Mismatched Records Found")
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph("No Mismatched Records Found", style='List Bullet')
# Check the next condition if the previous condition was not met.
    elif cnt_final > max_excel_row:
# Write the current DataFrame or object to storage.
        final_df.write.format('delta').mode('overwrite').save(path.replace("/dbfs", '')+f"{container}_{db}_{table_name}_mismatched_details")

# Execute this line as part of the notebook's workflow logic.
    '''
# Original notebook comment retained.
    ## Convert to pandas dataframe
# Original notebook comment retained.
    #mismatched_record_df = convertDecimalToFloat(mismatched_record_df).toPandas()
# Assign the result on the right-hand side to `mismatched_record_df` so it can be reused later.
    mismatched_record_df = get_pandas(mismatched_record_df.limit(max_excel_row))
# Execute this line as part of the notebook's workflow logic.
    mismatched_record_df.spark.cache()
# Original notebook comment retained.
    #final_df = convertDecimalToFloat(final_df).toPandas()
# Assign the result on the right-hand side to `final_df` so it can be reused later.
    final_df = get_pandas(final_df.limit(max_excel_row))
# Execute this line as part of the notebook's workflow logic.
    final_df.spark.cache
# Execute this line as part of the notebook's workflow logic.
    '''

# Return a value from the current function back to the caller.
    return cnt_diff, mismatched_record_df_pd, cnt_final, final_df_pd




## Command cell 29

This section corresponds to command 29 from the original Databricks notebook.


In [ ]:
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.types import FloatType.
from pyspark.sql.types import FloatType
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.functions import col.
from pyspark.sql.functions import col



# Define the function `find_difference` so this block can be reused later in the notebook.
def find_difference(data1, data2, id_columns):
# Assign the result on the right-hand side to `data1` so it can be reused later.
    data1 = data1.select( *[ F.when(F.col(column).isNull(),'').otherwise(F.col(column)).alias(column) for column in data1.columns])
# Assign the result on the right-hand side to `data2` so it can be reused later.
    data2 = data2.select( *[ F.when(F.col(column).isNull(),'').otherwise(F.col(column)).alias(column) for column in data2.columns])

# Print a message or value to the notebook output for validation or debugging.
    print('WORKING ON MISMATCHED RECORDS')
# Assign the result on the right-hand side to `cols` so it can be reused later.
    cols = data1.columns
# Assign the result on the right-hand side to `data_difference` so it can be reused later.
    data_difference = data2.subtract(data1)
# Execute this line as part of the notebook's workflow logic.
    data_difference.cache() # First you find the difference in rows
# Original notebook comment retained.
    #join_on_col = list(set(id_columns).intersection(set(data_difference.columns)))
# Assign the result on the right-hand side to `data_difference` so it can be reused later.
    data_difference = data_difference.join(data1.select(*[F.col(i).alias(f'SRC_{i}') if i not in id_columns else F.col(i).alias(i) for i in cols]), on = id_columns, how = 'left')
# Execute this line as part of the notebook's workflow logic.
    data_difference.cache()
# Assign the result on the right-hand side to `mismatched_record_df` so it can be reused later.
    mismatched_record_df = data_difference.alias("mismatched_record_df").orderBy(id_columns)
# Original notebook comment retained.
    #display(mismatched_record_df.limit(5))
# Execute this line as part of the notebook's workflow logic.
    mismatched_record_df.cache()
# Assign the result on the right-hand side to `cnt_diff` so it can be reused later.
    cnt_diff = mismatched_record_df.count()
# Original notebook comment retained.
    #print(cnt_diff)
# Original notebook comment retained.
    #mismatched_record_df = get_pandas(mismatched_record_df.limit(max_excel_row))
# Assign the result on the right-hand side to `mismatched_record_df_pd` so it can be reused later.
    mismatched_record_df_pd = get_pandas(mismatched_record_df)
# Execute this line as part of the notebook's workflow logic.
    mismatched_record_df_pd.spark.cache()
# Execute this line as part of the notebook's workflow logic.
    '''
# Original notebook comment retained.
    ## TEMP - ONLY for TESTING LOGIC
# Assign the result on the right-hand side to `cnt_diff` so it can be reused later.
    cnt_diff = 0
# Original notebook comment retained.
    ## END
# Execute this line as part of the notebook's workflow logic.
    '''
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if cnt_diff == 0:
# Assign the result on the right-hand side to `columns` so it can be reused later.
        columns = StructType([StructField('Mismatched Records', StringType(), True)])
# Assign the result on the right-hand side to `mismatched_record_df` so it can be reused later.
        mismatched_record_df = spark.createDataFrame(data = [], schema = columns)
# Assign the result on the right-hand side to `mismatched_record_df_pd` so it can be reused later.
        mismatched_record_df_pd = get_pandas(mismatched_record_df)
# Execute this line as part of the notebook's workflow logic.
        mismatched_record_df_pd.spark.cache()
# Print a message or value to the notebook output for validation or debugging.
        print("No Mismatched Records Found")
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph("No Mismatched Records Found", style='List Bullet')
# Check the next condition if the previous condition was not met.
    elif cnt_diff > max_excel_row:
# Write the current DataFrame or object to storage.
        mismatched_record_df.write.format('delta').mode("overwrite").save(path.replace("/dbfs", '')+f"{container}_{db}_{table_name}_mismatched_records")



# Print a message or value to the notebook output for validation or debugging.
    print('WORKING ON MISMATCHED RECORDS DETAILS')
# Original notebook comment retained.
    # you join the differences with the second dataframe
# Original notebook comment retained.
    #final_df=data_difference.select( *[ F.when(F.col(column).isNull(),'').otherwise(F.col(column)).alias(column) for column in data_difference.columns]).orderBy(id_columns)
# Original notebook comment retained.
    #display(final_df.limit(5))
# Original notebook comment retained.
    #final_df.cache()
# Assign the result on the right-hand side to `final_df` so it can be reused later.
    final_df=data_difference.select(*[is_different(x, f"SRC_{x}",data_difference.columns, data_difference) for x in cols]).orderBy(id_columns)
# Original notebook comment retained.
    #display(final_df.limit(5))
# Execute this line as part of the notebook's workflow logic.
    final_df.cache()
# Assign the result on the right-hand side to `cnt_final` so it can be reused later.
    cnt_final=final_df.count()
# Original notebook comment retained.
    #final_df = get_pandas(final_df.limit(max_excel_row))  #
# Assign the result on the right-hand side to `final_df_pd` so it can be reused later.
    final_df_pd = get_pandas(final_df)
# Execute this line as part of the notebook's workflow logic.
    final_df_pd.spark.cache()
# Execute this line as part of the notebook's workflow logic.
    '''
# Original notebook comment retained.
    ## TEMP - ONLY for TESTING LOGIC
# Assign the result on the right-hand side to `cnt_final` so it can be reused later.
    cnt_final = 0
# Original notebook comment retained.
    ## END
# Execute this line as part of the notebook's workflow logic.
    '''
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if cnt_final == 0:
# Assign the result on the right-hand side to `columns` so it can be reused later.
        columns = StructType([StructField('Mismatched Records Details', StringType(), True)])
# Assign the result on the right-hand side to `final_df` so it can be reused later.
        final_df = spark.createDataFrame(data = [], schema = columns)
# Assign the result on the right-hand side to `final_df_pd` so it can be reused later.
        final_df_pd = get_pandas(final_df)
# Execute this line as part of the notebook's workflow logic.
        final_df_pd.spark.cache()
# Print a message or value to the notebook output for validation or debugging.
        print("No Mismatched Records Found")
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph("No Mismatched Records Found", style='List Bullet')
# Check the next condition if the previous condition was not met.
    elif cnt_final > max_excel_row:
# Write the current DataFrame or object to storage.
        final_df.write.format('delta').mode('overwrite').save(path.replace("/dbfs", '')+f"{container}_{db}_{table_name}_mismatched_details")




# Original notebook comment retained.
    ## Find missing records
# Print a message or value to the notebook output for validation or debugging.
    print('WORKING ON MISSING RECORDS')
# Assign the result on the right-hand side to `append_str` so it can be reused later.
    append_str='tgt_'
# Assign the result on the right-hand side to `pre_res` so it can be reused later.
    pre_res = [append_str + sub for sub in id_columns]

# Assign the result on the right-hand side to `pre_res1` so it can be reused later.
    pre_res1 = [append_str + sub for sub in cols]
# Original notebook comment retained.
    #print(type(pre_res1))
# Assign the result on the right-hand side to `data_difference_1` so it can be reused later.
    data_difference_1 = data1.subtract(data2)
# Execute this line as part of the notebook's workflow logic.
    data_difference_1.cache()
# Original notebook comment retained.
    #join_on_col = list(set(id_columns).intersection(set(data_difference_1.columns)))
# Assign the result on the right-hand side to `data_difference_1` so it can be reused later.
    data_difference_1 = data_difference_1.join(data2.select(*[F.col(i).alias(f'tgt_{i}') if i not in id_columns else F.col(i).alias(i) for i in cols]), on = id_columns, how = 'left')
# Execute this line as part of the notebook's workflow logic.
    data_difference_1.cache()


# Assign the result on the right-hand side to `data_difference_1` so it can be reused later.
    data_difference_1 = Missing_condition(pre_res1,pre_res,data_difference_1)
# Execute this line as part of the notebook's workflow logic.
    data_difference_1.cache()

# Original notebook comment retained.
    #counts = data_difference_1.select([F.count(i).alias(i) for i in data_difference_1.columns]).toPandas()
# Assign the result on the right-hand side to `counts` so it can be reused later.
    counts = ps.DataFrame(data_difference_1.select([F.count(i).alias(i) for i in data_difference_1.columns]))


# Assign the result on the right-hand side to `output` so it can be reused later.
    output = data_difference_1.select(*counts.columns[counts.ne(0).iloc[0]]).orderBy(id_columns)
# Execute this line as part of the notebook's workflow logic.
    output.cache()
# Assign the result on the right-hand side to `cnt` so it can be reused later.
    cnt=output.count()
# Original notebook comment retained.
    #output = get_pandas(output.limit(max_excel_row))  #
# Assign the result on the right-hand side to `output_pd` so it can be reused later.
    output_pd = get_pandas(output)
# Execute this line as part of the notebook's workflow logic.
    output_pd.spark.cache()

# Execute this line as part of the notebook's workflow logic.
    '''
# Original notebook comment retained.
    ## TEMP - ONLY for TESTING LOGIC
# Assign the result on the right-hand side to `cnt` so it can be reused later.
    cnt = 0
# Original notebook comment retained.
    ## END
# Execute this line as part of the notebook's workflow logic.
    '''
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if cnt == 0:
# Assign the result on the right-hand side to `columns` so it can be reused later.
        columns = StructType([StructField('Missing Records',
# Execute this line as part of the notebook's workflow logic.
                                  StringType(), True)])
# Assign the result on the right-hand side to `output` so it can be reused later.
        output = spark.createDataFrame(data = [],
# Assign the result on the right-hand side to `schema` so it can be reused later.
                           schema = columns)
# Assign the result on the right-hand side to `output_pd` so it can be reused later.
        output_pd = get_pandas(output)
# Execute this line as part of the notebook's workflow logic.
        output_pd.spark.cache()
# Print a message or value to the notebook output for validation or debugging.
        print("No Missing Records Found")
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph("No Missing Records Found", style='List Bullet')
# Check the next condition if the previous condition was not met.
    elif cnt > max_excel_row:
# Write the current DataFrame or object to storage.
        output.write.format('delta').mode('overwrite').save(path.replace("/dbfs", '')+f"{container}_{db}_{table_name}_missing_records")


# Execute this line as part of the notebook's workflow logic.
    '''
# Original notebook comment retained.
    ## Convert to pandas dataframe
# Original notebook comment retained.
    #mismatched_record_df = convertDecimalToFloat(mismatched_record_df).toPandas()
# Assign the result on the right-hand side to `mismatched_record_df` so it can be reused later.
    mismatched_record_df = get_pandas(mismatched_record_df.limit(max_excel_row))
# Execute this line as part of the notebook's workflow logic.
    mismatched_record_df.spark.cache()
# Original notebook comment retained.
    #final_df = convertDecimalToFloat(final_df).toPandas()
# Assign the result on the right-hand side to `final_df` so it can be reused later.
    final_df = get_pandas(final_df.limit(max_excel_row))
# Execute this line as part of the notebook's workflow logic.
    final_df.spark.cache()
# Original notebook comment retained.
    #output = convertDecimalToFloat(output).toPandas()
# Assign the result on the right-hand side to `output` so it can be reused later.
    output = get_pandas(output.limit(max_excel_row))
# Execute this line as part of the notebook's workflow logic.
    output.spark.cache()
# Execute this line as part of the notebook's workflow logic.
    '''

# Return a value from the current function back to the caller.
    return cnt_diff, mismatched_record_df_pd, cnt_final, final_df_pd, cnt, output_pd

## Command cell 30

This section corresponds to command 30 from the original Databricks notebook.


In [ ]:
# Assign the result on the right-hand side to `netezza_synapse_database_name` so it can be reused later.
netezza_synapse_database_name = []
# Assign the result on the right-hand side to `netezza_synapse_table_name` so it can be reused later.
netezza_synapse_table_name = []
# Assign the result on the right-hand side to `netezza_synapse_partition_count_mismatched_cnt` so it can be reused later.
netezza_synapse_partition_count_mismatched_cnt = []
# Assign the result on the right-hand side to `netezza_synapse_measure_level_mismatched_cnt_diff` so it can be reused later.
netezza_synapse_measure_level_mismatched_cnt_diff = []
# Assign the result on the right-hand side to `netezza_synapse_partition_level_mismatched_cnt_diff` so it can be reused later.
netezza_synapse_partition_level_mismatched_cnt_diff = []
# Assign the result on the right-hand side to `netezza_synapse_partition_level_missing_cnt` so it can be reused later.
netezza_synapse_partition_level_missing_cnt = []
# Assign the result on the right-hand side to `netezza_synapse_table_level_mismatched_cnt_diff` so it can be reused later.
netezza_synapse_table_level_mismatched_cnt_diff = []
# Assign the result on the right-hand side to `netezza_synapse_table_level_missing_cnt` so it can be reused later.
netezza_synapse_table_level_missing_cnt = []
# Assign the result on the right-hand side to `netezza_synapse_measure_level_diff_cnt` so it can be reused later.
netezza_synapse_measure_level_diff_cnt = []
# Assign the result on the right-hand side to `netezza_synapse_partition_level_diff_cnt` so it can be reused later.
netezza_synapse_partition_level_diff_cnt = []
# Assign the result on the right-hand side to `netezza_synapse_table_level_diff_cnt` so it can be reused later.
netezza_synapse_table_level_diff_cnt = []
# Assign the result on the right-hand side to `netezza_synapse_detailed_report_path` so it can be reused later.
netezza_synapse_detailed_report_path = []

## Command cell 31

This section corresponds to command 31 from the original Databricks notebook.


In [ ]:
# Assign the result on the right-hand side to `tab_sy_cnt` so it can be reused later.
tab_sy_cnt = 0

## Command cell 32

This section corresponds to command 32 from the original Databricks notebook.


In [ ]:
# Original notebook comment retained.
#Netezza Tables with  Synapse Comparsion.
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.functions import *.
from pyspark.sql.functions import *
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.types import DecimalType, IntegerType, DateType, StructType, TimestampType.
from pyspark.sql.types import DecimalType, IntegerType, DateType, StructType, TimestampType
# Import specific objects from a module so they can be used directly in later code: from datetime import date, timedelta, datetime.
from datetime import date, timedelta, datetime
# Import specific objects from a module so they can be used directly in later code: from delta.tables import *.
from delta.tables import *

# Original notebook comment retained.
# Header for log document
# Add a heading to the Word document being used as a log or report.
doc.add_heading('Log From Detailed Report Generation Comparing Netezza v. Synapse', 2)
# Add a heading to the Word document being used as a log or report.
doc.add_heading('KEY TYPE UNIQUE = Y', 3)
# Assign the result on the right-hand side to `container` so it can be reused later.
container = 'synapse'

# Begin a protected block so the notebook can handle runtime errors more gracefully.
try:

# Start a loop so the same logic is applied repeatedly across multiple items.
    for db in relevant_gold_db:

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
        if Env == 'QA':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
            db_tmp = f"GDC_{db}"
# Check the next condition if the previous condition was not met.
        elif Env == 'PROD_DR':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
            db_tmp = f"{db}_DR"
# Run the fallback branch when the earlier conditions do not match.
        else:
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
            db_tmp = db
# Print a message or value to the notebook output for validation or debugging.
        print('DATABASE: ', db)


# Start a loop so the same logic is applied repeatedly across multiple items.
        for table_dict in [v for d in validation_tables_info_2 for k,v in d.items() if k == db][0]:
# Original notebook comment retained.
            #print(table_dict)
# Assign the result on the right-hand side to `table_name` so it can be reused later.
            table_name=list(table_dict.keys())[0]
# Print a message or value to the notebook output for validation or debugging.
            print('TABLE:',table_name)      


# Begin a protected block so the notebook can handle runtime errors more gracefully.
            try:
# Assign the result on the right-hand side to `partition_key` so it can be reused later.
                partition_key=list(table_dict.values())[0][0]
# Original notebook comment retained.
                #print(type(partition_key))
# Assign the result on the right-hand side to `primary_key` so it can be reused later.
                primary_key=list(table_dict.values())[0]
# Print a message or value to the notebook output for validation or debugging.
                print('PRIMARY KEY:', primary_key)
# Assign the result on the right-hand side to `Partition_column` so it can be reused later.
                Partition_column=spark.sql(f"select distinct PARTITION_COLUMN from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_COLUMN is not null")
# Assign the result on the right-hand side to `KEY_TYPE` so it can be reused later.
                KEY_TYPE=spark.sql(f"select distinct KEY_TYPE_UNIQUE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and KEY_TYPE_UNIQUE is not null")
# Assign the result on the right-hand side to `KEY_TYPES` so it can be reused later.
                KEY_TYPES=KEY_TYPE.collect()[0][0]



# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                if KEY_TYPES == "Y" :   
# Assign the result on the right-hand side to `tab_sy_cnt +` so it can be reused later.
                    tab_sy_cnt += 1
# Print a message or value to the notebook output for validation or debugging.
                    print(tab_sy_cnt)

# Add a heading to the Word document being used as a log or report.
                    doc.add_heading(f"{db}.{table_name}", 4)             

# Original notebook comment retained.
                    ## Check if a partition column is configured for this table
# Print a message or value to the notebook output for validation or debugging.
                    print(f'Checking if a partition column is configured for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'Checking if a partition column is configured for {db}.{table_name}', style='List Bullet')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                    if Partition_column.count() == 0:
# Assign the result on the right-hand side to `Partition_col` so it can be reused later.
                        Partition_col = None
# Print a message or value to the notebook output for validation or debugging.
                        print(f'No partition column configured for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                        doc.add_paragraph(f'No partition column configured for {db}.{table_name}', style='List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                    else:
# Assign the result on the right-hand side to `Partition_col` so it can be reused later.
                        Partition_col =Partition_column.collect()[0][0]
# Print a message or value to the notebook output for validation or debugging.
                        print(f'Partition column configured for {db}.{table_name} is {Partition_col}')
# Add a paragraph to the Word document so the log captures another message or detail.
                        doc.add_paragraph(f'Partition column configured for {db}.{table_name} is {Partition_col}', style='List Bullet')


# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                    if ((Partition_col != None) and (Partition_col != 'NA') ):
# Original notebook comment retained.
                        ## Retrieve partition, measure and field validation queries from metadata table
# Print a message or value to the notebook output for validation or debugging.
                        print(f'Retrieving Partition, Measure and Field Validation Queries from Fwk_Gold_Tables for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                        doc.add_paragraph(f'Retrieving Partition, Measure and Field Validation Queries from Fwk_Gold_Tables for {db}.{table_name}', style='List Bullet')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (PL_TL == '7GS'):  
# Assign the result on the right-hand side to `Query_Agg_Netezza` so it can be reused later.
                            Query_Agg_Netezza=spark.sql(f"select distinct PARTITON_QUERY_NETEZZA_SYNAPSE_SEVEN_YEARS from gold_df \
# Assign the result on the right-hand side to `where TABLE_NAME` so it can be reused later.
                                                    where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITON_QUERY_NETEZZA_SYNAPSE_SEVEN_YEARS is not null") 
# Assign the result on the right-hand side to `Query_Agg_Synapse` so it can be reused later.
                            Query_Agg_Synapse=spark.sql(f"select distinct PARTITION_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}'\
# This line is part of the SQL statement being built for the data comparison or extraction step.
                                                    and PARTITION_QUERY_SYNAPSE is not null")    
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f"select distinct MEASURE_QUERY_NETEZZA_SEVEN_YEARS from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}'\
# This line is part of the SQL statement being built for the data comparison or extraction step.
                                                            and MEASURE_QUERY_NETEZZA_SEVEN_YEARS is not null")
# Assign the result on the right-hand side to `Query_Measure_Synapse` so it can be reused later.
                            Query_Measure_Synapse=spark.sql(f"select distinct MEASURE_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}'\
# This line is part of the SQL statement being built for the data comparison or extraction step.
                                                    and MEASURE_QUERY_SYNAPSE is not null")                      
# Start a new indented code block for the statement above.
                        else :
# Original notebook comment retained.
                            #below is for NON transactional and transactional where we have full data in synapse as well
# Assign the result on the right-hand side to `Query_Agg_Netezza` so it can be reused later.
                            Query_Agg_Netezza=spark.sql(f"select distinct PARTITION_QUERY_NETEZZA from gold_df where TABLE_NAME = '{table_name}' and DATABASE ='{db}' and PARTITION_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Agg_Synapse` so it can be reused later.
                            Query_Agg_Synapse=spark.sql(f"select distinct PARTITION_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_QUERY_SYNAPSE is not null")  
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f" select distinct MEASURE_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Synapse` so it can be reused later.
                            Query_Measure_Synapse=spark.sql(f" select distinct MEASURE_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_SYNAPSE is not null")

# Assign the result on the right-hand side to `Query_Field_Level_Netezza` so it can be reused later.
                        Query_Field_Level_Netezza=spark.sql(f"select distinct FIELD_VALIDATION_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and FIELD_VALIDATION_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Field_Level_Synapse` so it can be reused later.
                        Query_Field_Level_Synapse=spark.sql(f"select distinct FIELD_VALIDATION_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and FIELD_VALIDATION_QUERY_SYNAPSE is not null")   

# Original notebook comment retained.
                        ## Skip table if no queries are configured
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (Query_Field_Level_Netezza.count() == 0) & (Query_Measure_Netezza.count() == 0) & (Query_Agg_Netezza.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                            print(f"No field validation query or measure query or partition query configured. No detailed report generated for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f"No field validation query or measure query or partition query configured. No detailed report generated for {db}.{table_name}", style = 'List Bullet')
# Execute this line as part of the notebook's workflow logic.
                            continue

# Execute this line as part of the notebook's workflow logic.
                        '''
# Original notebook comment retained.
                        ## Retrieve date part used for table partition (YEAR, MONTH, DAY)
# Assign the result on the right-hand side to `PARTITION_ON` so it can be reused later.
                        PARTITION_ON=spark.sql(f"select PARTITION_ON from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_ON is not null")
# Assign the result on the right-hand side to `Partition_Type` so it can be reused later.
                        Partition_Type=PARTITION_ON.collect()[0][0].upper()
# Print a message or value to the notebook output for validation or debugging.
                        print('PARTITION TYPE:', Partition_Type)
# Execute this line as part of the notebook's workflow logic.
                        '''


# Original notebook comment retained.
                        ## PERFORM PARTITION COUNT VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (Query_Agg_Netezza.count() == 0) | (Query_Agg_Synapse.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                            print(f'No partition queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No partition count validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'No partition queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No partition count validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                            partition_count_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                        else:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                            query_PARTITION_CNT_NZ = Query_Agg_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                            query_PARTITION_CNT_SY = Query_Agg_Synapse.collect()[0][0]

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (query_PARTITION_CNT_NZ == '') | (query_PARTITION_CNT_SY == ''):
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                print('No Partition Query is configured. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(('No Partition Query is configured. No partition count validation performed'), style= 'List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Attempting to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Attempting to perform partition count validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                    query_PARTITION_CNT_NZ = query_PARTITION_CNT_NZ.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_partition_cnt_df` so it can be reused later.
                                    netezza_partition_cnt_df = readfromNetizza(Env, query_PARTITION_CNT_NZ, db)
# Assign the result on the right-hand side to `synapse_partition_cnt_df` so it can be reused later.
                                    synapse_partition_cnt_df = readfromSynapse(Env,query_PARTITION_CNT_SY)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df` so it can be reused later.
                                        partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df = find_Mismatched(netezza_partition_cnt_df,synapse_partition_cnt_df,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                        print(f"Successfully performed partition count validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"Successfully performed partition count validation for {db}.{table_name}", style = 'List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                        partition_count_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                        partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Failed to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Failed to perform partition count validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                    partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Bad partition query configured for {db}.{table_name}. Please correct the query. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Bad partition query configured for {db}.{table_name}. Please correct the query. No partition count validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')




# Execute this line as part of the notebook's workflow logic.
                        '''
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (PL_TL == '7GS'):
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if Query_Agg_Netezza.count()!=0:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                query_PARTITION_CNT_NZ = Query_Agg_Netezza.collect()[0][0]
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                query_PARTITION_CNT_NZ = ''
# Print a message or value to the notebook output for validation or debugging.
                                print('No Partition_Query_Netezza_Synapse_Seven_Years is configured')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('No Partition_Query_Netezza_Synapse_Seven_Years is configured', style='List Bullet')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if Partition_Type == 'YEAR' :
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                                query_PARTITION_CNT_SY= f"""select FORMAT({Partition_col}, 'yyyy') as {Partition_col}, count(*) as CNT from {db}.{table_name} group by FORMAT({Partition_col}, 'yyyy')"""
# Check the next condition if the previous condition was not met.
                            elif Partition_Type == 'MONTH' :
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                                query_PARTITION_CNT_SY=f"""select FORMAT({Partition_col}, 'yyyy MMM') as {Partition_col}, count(*) as CNT from {db}.{table_name} group by FORMAT({Partition_col}, 'yyyy MMM')"""
# Check the next condition if the previous condition was not met.
                            elif Partition_Type == 'DAY' :
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                                query_PARTITION_CNT_SY=f"""select FORMAT({Partition_col}, 'yyyy MMM dd') as {Partition_col}, count(*) as CNT from {db}.{table_name} group by FORMAT({Partition_col}, 'yyyy MMM dd')"""

# Run the fallback branch when the earlier conditions do not match.
                        else:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if Partition_Type == 'YEAR' :
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                query_PARTITION_CNT_NZ = f"""select to_char({Partition_col}, 'YYYY') as {Partition_col}, count(*) as CNT from {db_tmp}.{table_name} group by to_char({Partition_col}, 'YYYY')"""
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                                query_PARTITION_CNT_SY= f"""select FORMAT({Partition_col}, 'yyyy') as {Partition_col}, count(*) as CNT from {db}.{table_name} group by FORMAT({Partition_col}, 'yyyy')"""
# Check the next condition if the previous condition was not met.
                            elif Partition_Type == 'MONTH' :
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                query_PARTITION_CNT_NZ = f"""select to_char({Partition_col}, 'YYYY Mon') as {Partition_col}, count(*) as CNT from {db_tmp}.{table_name} group by to_char({Partition_col}, 'YYYY Mon')"""
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                                query_PARTITION_CNT_SY=f"""select FORMAT({Partition_col}, 'yyyy MMM') as {Partition_col}, count(*) as CNT from {db}.{table_name} group by FORMAT({Partition_col}, 'yyyy MMM')"""
# Check the next condition if the previous condition was not met.
                            elif Partition_Type == 'DAY' :
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                query_PARTITION_CNT_NZ = f"""select to_char({Partition_col}, 'YYYY Mon dd') as {Partition_col}, count(*) as CNT from {db_tmp}.{table_name} group by to_char({Partition_col}, 'YYYY Mon dd')"""
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                                query_PARTITION_CNT_SY=f"""select FORMAT({Partition_col}, 'yyyy MMM dd') as {Partition_col}, count(*) as CNT from {db}.{table_name} group by FORMAT({Partition_col}, 'yyyy MMM dd')"""

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (query_PARTITION_CNT_NZ == '') | (query_PARTITION_CNT_SY == ''):
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                            partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                            print('No Partition Query is configured. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(('No Partition Query is configured. No partition count validation performed'), style= 'List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                        else:
# Assign the result on the right-hand side to `netezza_partition_cnt_df` so it can be reused later.
                            netezza_partition_cnt_df = readfromNetizza(Env, query_PARTITION_CNT_NZ, db)
# Assign the result on the right-hand side to `synapse_partition_cnt_df` so it can be reused later.
                            synapse_partition_cnt_df = readfromSynapse(Env,query_PARTITION_CNT_SY)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:
# Assign the result on the right-hand side to `partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df` so it can be reused later.
                                partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df = find_Mismatched(netezza_partition_cnt_df,synapse_partition_cnt_df,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                print(f"Successfully performed partition count validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f"Successfully performed partition count validation for {db}.{table_name}", style = 'List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                partition_count_validation_status = 'True'
# Handle an error raised in the preceding try block.
                            except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Failed to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Failed to perform partition count validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'{e}', style='Intense Quote')
# Execute this line as part of the notebook's workflow logic.
                        '''




# Original notebook comment retained.
                        ## PERFORM MEASURE LEVEL VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (Query_Measure_Netezza.count() == 0) | (Query_Measure_Synapse.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                            print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                            measure_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                        else:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                            query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_measure_synapse` so it can be reused later.
                            query_measure_synapse = Query_Measure_Synapse.collect()[0][0]

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (query_measure_netezza == '') | (query_measure_synapse == ''):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                measure_validation_status = 'False' 
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Attempting to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Attempting to perform measure level validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                    query_measure_netezza = query_measure_netezza.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                    netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Assign the result on the right-hand side to `synapse_df_measure` so it can be reused later.
                                    synapse_df_measure=readfromSynapse(Env,query_measure_synapse)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff, measure_level_mismatched_record_df, measure_level_mismatched_detail_cnt, measure_level_mismatched_detail_df` so it can be reused later.
                                        measure_level_mismatched_cnt_diff, measure_level_mismatched_record_df, measure_level_mismatched_detail_cnt, measure_level_mismatched_detail_df = find_Mismatched(netezza_df_measure,synapse_df_measure,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                        print(f"Successfully performed measure level validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"Successfully performed measure level validation for {db}.{table_name}", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                        measure_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                        measure_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Failed to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Failed to perform measure level validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                    measure_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')



# Original notebook comment retained.
                        ## PERFORM PARTITION LEVEL VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (Query_Field_Level_Netezza.count()==0) | (Query_Field_Level_Synapse.count()==0):
# Print a message or value to the notebook output for validation or debugging.
                            print(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No partition level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No partition level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                            partition_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                        else:
# Assign the result on the right-hand side to `query_PARTITION_NZ` so it can be reused later.
                            query_PARTITION_NZ = Query_Field_Level_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_PARTITION_SY` so it can be reused later.
                            query_PARTITION_SY = Query_Field_Level_Synapse.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (query_PARTITION_NZ == '') | (query_PARTITION_SY == ''):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No partition level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No partition level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                partition_validation_status = 'False'  
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:
# Assign the result on the right-hand side to `query_PARTITION_NZ` so it can be reused later.
                                    query_PARTITION_NZ = query_PARTITION_NZ.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_partition` so it can be reused later.
                                    netezza_df_partition=readfromNetizza(Env,query_PARTITION_NZ,db)
# Assign the result on the right-hand side to `synapse_df_partition` so it can be reused later.
                                    synapse_df_partition=readfromSynapse(Env,query_PARTITION_SY)  

# Original notebook comment retained.
                                    ## Validate that the partition size is within the max row permitted limit
# Print a message or value to the notebook output for validation or debugging.
                                    print('Attempting to perform partition level validation. Need to confirm that the partition size is within acceptable limit')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph('Attempting to perform partition level validation. Need to confirm that the partition size is within acceptable limit', style = 'List Bullet')

# Assign the result on the right-hand side to `cnt_netezza_df_partition` so it can be reused later.
                                    cnt_netezza_df_partition = netezza_df_partition.limit(max_rows_permitted + 1).count()
# Assign the result on the right-hand side to `cnt_synapse_df_partition` so it can be reused later.
                                    cnt_synapse_df_partition = synapse_df_partition.limit(max_rows_permitted + 1).count()

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if (cnt_netezza_df_partition == 0) | (cnt_synapse_df_partition == 0):
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Partition for table {db}.{table_name} is empty either in Netezza or in Synapse. No partition validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Partition for table {db}.{table_name} is empty either in Netezza or in Synapse. No partition validation performed', style = 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                        partition_validation_status = 'False'
# Check the next condition if the previous condition was not met.
                                    elif (cnt_netezza_df_partition > max_rows_permitted) | (cnt_synapse_df_partition > max_rows_permitted):
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Partition for {db}.{table_name} is greater than {max_rows_permitted}. No partition validation performed. Please reconfigure the partition for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Partition for {db}.{table_name} is greater than {max_rows_permitted}. No partition validation performed. Please reconfigure the partition for {db}.{table_name}', style = 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                        partition_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Original notebook comment retained.
                                        ## Perform validation on the partition
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Partition for {db}.{table_name} is less than or equal to {max_rows_permitted}.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Partition for {db}.{table_name} is less than or equal to {max_rows_permitted}.', style = 'List Bullet')
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Original notebook comment retained.
                                            #partition_level_mismatched_cnt_diff, partition_level_mismatched_record_df, partition_level_mismatched_record_detail_cnt,partition_level_mismatched_record_detail_df, partition_level_missing_cnt, partition_level_missing_output_df = find_difference(netezza_df_partition, synapse_df_partition, primary_key)
# Assign the result on the right-hand side to `partition_level_mismatched_cnt_diff, partition_level_mismatched_record_df, partition_level_mismatched_record_detail_cnt,partition_level_mismatched_record_detail_df, partition_level_missing_cnt, partition_level_missing_output_df` so it can be reused later.
                                            partition_level_mismatched_cnt_diff, partition_level_mismatched_record_df, partition_level_mismatched_record_detail_cnt,partition_level_mismatched_record_detail_df, partition_level_missing_cnt, partition_level_missing_output_df = find_difference(trim_space(netezza_df_partition), trim_space(synapse_df_partition), primary_key)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed partition level validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed partition level validation for {db}.{table_name}", style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                            partition_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Failed to perform partiton level validation for {db}.{table_name}_(Netezza v. Synapse)')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Failed to perform partiton level validation for {db}.{table_name}_(Netezza v. Synapse)', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                            partition_validation_status = 'False'
# Handle an error raised in the preceding try block.
                                except Exception as e:
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                    partition_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Bad field validation query configured for {db}.{table_name}. Please correct the query. No partition level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Bad field validation query configured for {db}.{table_name}. Please correct the query. No partition level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')


# Original notebook comment retained.
                        ## Define excel writer
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                        try:
# Open a context-managed resource so it is handled safely and closed automatically after use.
                            with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Synapse).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                ## Write missing and mismatched information to sheets in excel workbook
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if partition_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `partition_level_missing_output_df.sort_values(by` so it can be reused later.
                                        partition_level_missing_output_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = f"missing_record_partition", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                    except:
# Assign the result on the right-hand side to `partition_level_missing_output_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        partition_level_missing_output_df.head(max_excel_row).to_excel(writer, sheet_name = f"missing_record_partition", startrow = 0, startcol=0, index = False)
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `partition_level_mismatched_record_df.sort_values(by` so it can be reused later.
                                        partition_level_mismatched_record_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = f"mismatched_record_partition", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_level_mismatched_record_detail_df.sort_values(by` so it can be reused later.
                                        partition_level_mismatched_record_detail_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = f"mismatched_detail_partition", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                    except:
# Assign the result on the right-hand side to `partition_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        partition_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = f"mismatched_record_partition", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_level_mismatched_record_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        partition_level_mismatched_record_detail_df.head(max_excel_row).to_excel(writer, sheet_name = f"mismatched_detail_partition", startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if measure_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.sort_values(by` so it can be reused later.
                                        measure_level_mismatched_record_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `measure_level_mismatched_detail_df.sort_values(by` so it can be reused later.
                                        measure_level_mismatched_detail_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                    except:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `measure_level_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        measure_level_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_detail", startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if partition_count_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `partition_count_mismatched_df.sort_values(by` so it can be reused later.
                                        partition_count_mismatched_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_count_mismatched_detail_df.sort_values(by` so it can be reused later.
                                        partition_count_mismatched_detail_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                    except:
# Assign the result on the right-hand side to `partition_count_mismatched_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        partition_count_mismatched_df.head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_count_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        partition_count_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_detail", startrow = 0, startcol=0, index = False)
# Print a message or value to the notebook output for validation or debugging.
                            print('Detailed report generated')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph('Detailed report generated', style = 'List Bullet')
# Assign the result on the right-hand side to `detailed_report_status` so it can be reused later.
                            detailed_report_status = 'True'
# Handle an error raised in the preceding try block.
                        except Exception as e:
# Assign the result on the right-hand side to `detailed_report_status` so it can be reused later.
                            detailed_report_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                            print('No detailed report generated')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph('No detailed report generated', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'{e}', style= 'Intense Quote')


# Original notebook comment retained.
                        ## Upload workbook to storage
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if detailed_report_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                workbook_path = path + f"{db}-{table_name}_(Netezza v. Synapse).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                shutil.copy2(f"{db}-{table_name}_(Netezza v. Synapse).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                os.remove(f"{db}-{table_name}_(Netezza v. Synapse).xlsx")

# Original notebook comment retained.
                                ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_table_name.append(table_name)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if partition_count_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_count_mismatched_cnt.append(str(partition_count_mismatched_cnt))
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_count_mismatched_cnt.append('')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if measure_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_measure_level_mismatched_cnt_diff.append('')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if partition_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_level_mismatched_cnt_diff.append(str(partition_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_level_missing_cnt.append(str(partition_level_missing_cnt))
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Synapse)", style='List Bullet')


# Handle an error raised in the preceding try block.
                            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Synapse)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'{e}', style='Intense Quote')


# Start a new indented code block for the statement above.
                    else :
# Original notebook comment retained.
                        ## This condition is to perform detailed validation for small and medium size tables
# Print a message or value to the notebook output for validation or debugging.
                        print(f'Attempting to perform detailed validation on {db}.{table_name}_(Netezza v. Synapse). Need to confirm it is a small to medium size table')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (PL_TL == '7GS'):

# Original notebook comment retained.
                            ## Retrieve measure query                        
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f"select distinct MEASURE_QUERY_NETEZZA_SEVEN_YEARS from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}'\
# This line is part of the SQL statement being built for the data comparison or extraction step.
                                                            and MEASURE_QUERY_NETEZZA_SEVEN_YEARS is not null")
# Assign the result on the right-hand side to `Query_Measure_Synapse` so it can be reused later.
                            Query_Measure_Synapse=spark.sql(f"select distinct MEASURE_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}'\
# This line is part of the SQL statement being built for the data comparison or extraction step.
                                                    and MEASURE_QUERY_SYNAPSE is not null")

# Original notebook comment retained.
                            ## Retrieve the retention column
# Execute this line as part of the notebook's workflow logic.
                            filterColumn=gold_df.select('Filter7').where((gold_df.TABLE_NAME==table_name)&(gold_df.DATABASE==db)).distinct().rdd.map(lambda x: x.Filter7).collect()
# Assign the result on the right-hand side to `filterColumn` so it can be reused later.
                            filterColumn=str(filterColumn)
# Assign the result on the right-hand side to `filterColumn` so it can be reused later.
                            filterColumn=filterColumn.replace("[","").replace("]","").replace("'","")
# Original notebook comment retained.
                            ## BELOW is fetch the minimum date from synapse to avoid mismatch on the day where data is not loaded in netizza
# Print a message or value to the notebook output for validation or debugging.
                            print(filterColumn)

# Original notebook comment retained.
                            ## Retrieve the retention value
# Assign the result on the right-hand side to `query_forValuefilter` so it can be reused later.
                            query_forValuefilter= f"""select CAST(min({filterColumn}) as varchar(10)) {filterColumn} from {db}.{table_name}""" 

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if filterColumn!='None':
# Assign the result on the right-hand side to `filterValue1` so it can be reused later.
                                filterValue1=readfromSynapse(Env,query_forValuefilter)
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                                filterValue=filterValue1.rdd.map(lambda x: x[0]).collect()
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                                filterValue=str(filterValue)
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                                filterValue=filterValue.replace("[","").replace("]","")
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                                filterValue='None'

# Original notebook comment retained.
                            ## Query netezza table based on retention values. Get netezza table for validation
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if filterValue!='None':
# Assign the result on the right-hand side to `Query_Netezza` so it can be reused later.
                                Query_Netezza = f"""select * from {db_tmp}.{table_name} where {filterColumn} >= {filterValue}""" 
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `Query_Netezza` so it can be reused later.
                                Query_Netezza = f"""select * from {db_tmp}.{table_name}"""
# Original notebook comment retained.
                            #below is filter for 7 years for netizza 
# Print a message or value to the notebook output for validation or debugging.
                            print(Query_Netezza)
# Assign the result on the right-hand side to `netezza_df` so it can be reused later.
                            netezza_df=readfromNetizza(Env,Query_Netezza,db)

# Original notebook comment retained.
                            ## Get synapse table for validation
# Assign the result on the right-hand side to `Query_Synapse` so it can be reused later.
                            Query_Synapse = f"""select * from {db}.{table_name}"""
# Assign the result on the right-hand side to `df_synapse` so it can be reused later.
                            df_synapse=readfromSynapse(Env,Query_Synapse)

# Run the fallback branch when the earlier conditions do not match.
                        else:

# Original notebook comment retained.
                            ## Retrieve measure query                        
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f"select distinct MEASURE_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}'\
# This line is part of the SQL statement being built for the data comparison or extraction step.
                                                            and MEASURE_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Synapse` so it can be reused later.
                            Query_Measure_Synapse=spark.sql(f"select distinct MEASURE_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}'\
# This line is part of the SQL statement being built for the data comparison or extraction step.
                                                    and MEASURE_QUERY_SYNAPSE is not null")

# Original notebook comment retained.
                            ## Query full netezza table. Get netezza table for validation
# Assign the result on the right-hand side to `Query_Netezza` so it can be reused later.
                            Query_Netezza = f"""select * from {db_tmp}.{table_name}"""
# Assign the result on the right-hand side to `netezza_df` so it can be reused later.
                            netezza_df=readfromNetizza(Env,Query_Netezza,db)

# Original notebook comment retained.
                            ## Get synapse table for validation
# Assign the result on the right-hand side to `Query_Synapse` so it can be reused later.
                            Query_Synapse = f"""select * from {db}.{table_name}"""
# Original notebook comment retained.
                            #print(Query_Synapse)
# Assign the result on the right-hand side to `df_synapse` so it can be reused later.
                            df_synapse=readfromSynapse(Env,Query_Synapse)




# Original notebook comment retained.
                        ## Confirm table is small or medium size (i.e. Not greater than 5 million rows)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                        try:
# Original notebook comment retained.
                            ## Retrieve row count
# Execute this line as part of the notebook's workflow logic.
                            cnt_netezza_df = int(report_dataframe_1.select(f'NETEZZA_TABLE_ROW_COUNT ({netezza_sub})').filter((col('NETEZZA_SCHEMA') == f'{db}') & (col('NETEZZA_TABLE') == f'{table_name}')).collect()[0][0]) #netezza_df.limit(max_rows_permitted + 1).count()
# Execute this line as part of the notebook's workflow logic.
                            cnt_df_synapse = int(report_dataframe_1.select('SYNAPSE_TABLE_ROW_COUNT').filter((col('SYNAPSE_SCHEMA') == f'{db}') & (col('SYNAPSE_TABLE') == f'{table_name}')).collect()[0][0])#df_synapse.limit(max_rows_permitted + 1).count()
# Handle an error raised in the preceding try block.
                        except:
# Print a message or value to the notebook output for validation or debugging.
                            print(f'{db}.{table_name} not available in summary report')
# Assign the result on the right-hand side to `cnt_netezza_df` so it can be reused later.
                            cnt_netezza_df = netezza_df.limit(max_rows_permitted + 1).count()
# Assign the result on the right-hand side to `cnt_df_synapse` so it can be reused later.
                            cnt_df_synapse = df_synapse.limit(max_rows_permitted + 1).count()

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (cnt_netezza_df == 0) | (cnt_df_synapse == 0):
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Table {db}.{table_name} is empty either in Netezza or in Synapse. No detailed validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Table {db}.{table_name} is empty either in Netezza or in Synapse. No detailed validation performed', style = 'List Bullet')

# Check the next condition if the previous condition was not met.
                        elif (cnt_netezza_df > max_rows_permitted) | (cnt_df_synapse > max_rows_permitted):
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Table {db}.{table_name}_(Netezza v. Synapse) is a big table, with no partition column configured. No table level validation is performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Table {db}.{table_name}_(Netezza v. Synapse) is a big table, with no partition column configured. No table level validation is performed', style='List Bullet')

# Print a message or value to the notebook output for validation or debugging.
                            print('Will attempt to perform measure level validation')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph('Attempting measure level validation for a big table with no partition configured', style= 'List Bullet')

# Original notebook comment retained.
                            ## PERFORM MEASURE LEVEL VALIDATION
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Measure_Netezza.count() == 0) | (Query_Measure_Synapse.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                measure_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_measure_synapse` so it can be reused later.
                                query_measure_synapse = Query_Measure_Synapse.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_measure_netezza == '') | (query_measure_synapse == ''):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                    measure_validation_status = 'False' 
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Measure queries configured. Attempting to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Measure queries configured. Attempting to perform measure level validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                        query_measure_netezza = query_measure_netezza.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                        netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Assign the result on the right-hand side to `synapse_df_measure` so it can be reused later.
                                        synapse_df_measure=readfromSynapse(Env,query_measure_synapse)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df` so it can be reused later.
                                            measure_level_mismatched_record_df = get_pandas(synapse_df_measure.subtract(netezza_df_measure))
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff` so it can be reused later.
                                            measure_level_mismatched_cnt_diff = len(measure_level_mismatched_record_df)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                            measure_validation_status = 'True'

# Original notebook comment retained.
                                            ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                            with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Synapse).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                                ## Write missing and mismatched information to sheets in excel workbook
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                                measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = f"measure_mismatched_record", startrow = 0, startcol=0, index = False)

# Original notebook comment retained.
                                            ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                            try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                                workbook_path = path + f"{db}-{table_name}_(Netezza v. Synapse).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                                shutil.copy2(f"{db}-{table_name}_(Netezza v. Synapse).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                                os.remove(f"{db}-{table_name}_(Netezza v. Synapse).xlsx")

# Original notebook comment retained.
                                                ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Synapse)", style='List Bullet')

# Handle an error raised in the preceding try block.
                                            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                                    print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                    doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Synapse)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                    doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Failed to perform detailed validation on {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Failed to perform detailed validation on {db}.{table_name}", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'{e}', style='Intense Quote')
# Execute this line as part of the notebook's workflow logic.
                                                continue
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')
# Execute this line as part of the notebook's workflow logic.
                            '''
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if Query_Measure_Netezza.count() != 0:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                    query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                    netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Original notebook comment retained.
                                    #print("QUERY MEASURE NETEZZA:", query_measure_netezza)                            

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if Query_Measure_Synapse.count() != 0:
# Assign the result on the right-hand side to `query_measure_synapse` so it can be reused later.
                                        query_measure_synapse = Query_Measure_Synapse.collect()[0][0]
# Assign the result on the right-hand side to `synapse_df_measure` so it can be reused later.
                                        synapse_df_measure=readfromSynapse(Env,query_measure_synapse)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df` so it can be reused later.
                                            measure_level_mismatched_record_df = get_pandas(synapse_df_measure.subtract(netezza_df_measure))
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff` so it can be reused later.
                                            measure_level_mismatched_cnt_diff = len(measure_level_mismatched_record_df)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                            measure_validation_status = 'True'

# Original notebook comment retained.
                                            ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                            with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Synapse).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                                ## Write missing and mismatched information to sheets in excel workbook
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                                measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = f"measure_mismatched_record", startrow = 0, startcol=0, index = False)

# Original notebook comment retained.
                                            ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                            try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                                workbook_path = path + f"{db}-{table_name}_(Netezza v. Synapse).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                                shutil.copy2(f"{db}-{table_name}_(Netezza v. Synapse).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                                os.remove(f"{db}-{table_name}_(Netezza v. Synapse).xlsx")

# Original notebook comment retained.
                                                ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Synapse)", style='List Bullet')

# Handle an error raised in the preceding try block.
                                            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                                    print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                    doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Synapse)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                    doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Failed to perform detailed validation on {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Failed to perform detailed validation on {db}.{table_name}", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Execute this line as part of the notebook's workflow logic.
                                            continue
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Print a message or value to the notebook output for validation or debugging.
                                        print(f"No measure query configured for {db}.{table_name} in Synapse")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"No measure query configured for {db}.{table_name} in Synapse", style='List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f"No measure query configured for {db}.{table_name} in Netezza")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"No measure query configured for {db}.{table_name} in Netezza", style='List Bullet')

# Handle an error raised in the preceding try block.
                            except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                measure_validation_status = 'False' 
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Failed to perform measure level validation for {db}.{table_name}. Re-examine the configured measure queries')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Failed to perform measure level validation for {db}.{table_name}. Re-examine the configured measure queries', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'{e}', style='Intense Quote')
# Execute this line as part of the notebook's workflow logic.
                            ''' 

# Run the fallback branch when the earlier conditions do not match.
                        else:
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Table size is less than or equal to {max_rows_permitted}. Performing detailed validation on full table {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Table size is less than or equal to {max_rows_permitted}. Performing detailed validation on full table {db}.{table_name}', style = 'List Bullet')
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:
# Original notebook comment retained.
                                #table_level_mismatched_cnt_diff, table_level_mismatched_record_df, table_level_mismatched_record_detail_cnt, table_level_mismatched_record_detail_df, table_level_missing_cnt, table_level_missing_output_df = find_difference(netezza_df, df_synapse, primary_key)
# Assign the result on the right-hand side to `table_level_mismatched_cnt_diff, table_level_mismatched_record_df, table_level_mismatched_record_detail_cnt, table_level_mismatched_record_detail_df, table_level_missing_cnt, table_level_missing_output_df` so it can be reused later.
                                table_level_mismatched_cnt_diff, table_level_mismatched_record_df, table_level_mismatched_record_detail_cnt, table_level_mismatched_record_detail_df, table_level_missing_cnt, table_level_missing_output_df = find_difference(trim_space(netezza_df), trim_space(df_synapse), primary_key)
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Successfully performed full table validation on {db}.{table_name}, comparing Netezza to Synapse table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Successfully performed full table validation on {db}.{table_name}, comparing Netezza to Synapse table', style='List Bullet')

# Original notebook comment retained.
                                ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Synapse).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                    ## Write missing and mismatched information to sheets in excel workbook
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `table_level_missing_output_df.sort_values(by` so it can be reused later.
                                        table_level_missing_output_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = "missing_record", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                    except:
# Assign the result on the right-hand side to `table_level_missing_output_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        table_level_missing_output_df.head(max_excel_row).to_excel(writer, sheet_name = "missing_record", startrow = 0, startcol=0, index = False)
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `table_level_mismatched_record_df.sort_values(by` so it can be reused later.
                                        table_level_mismatched_record_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = "mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `table_level_mismatched_record_detail_df.sort_values(by` so it can be reused later.
                                        table_level_mismatched_record_detail_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = "mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                    except:
# Assign the result on the right-hand side to `table_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        table_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = "mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `table_level_mismatched_record_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        table_level_mismatched_record_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "mismatched_detail", startrow = 0, startcol=0, index = False)


# Original notebook comment retained.
                                ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                    workbook_path = path + f"{db}-{table_name}_(Netezza v. Synapse).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                    shutil.copy2(f"{db}-{table_name}_(Netezza v. Synapse).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                    os.remove(f"{db}-{table_name}_(Netezza v. Synapse).xlsx")

# Original notebook comment retained.
                                    ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_measure_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_table_level_mismatched_cnt_diff.append(str(table_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_table_level_missing_cnt.append(str(table_level_missing_cnt))
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                    print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Synapse)", style='List Bullet')

# Handle an error raised in the preceding try block.
                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Synapse)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')


# Handle an error raised in the preceding try block.
                            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Synapse table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Synapse table', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                print(f'ERROR OCCURRED: Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Synapse table')
# Add a paragraph to the Word document so the log captures another message or detail.
                doc.add_paragraph(f'ERROR OCCURRED: Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Synapse table', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                doc.add_paragraph(f'{e}', style='Intense Quote')


# Original notebook comment retained.
    ## Upload log file
# Execute this line as part of the notebook's workflow logic.
    save_log(doc)

# Handle an error raised in the preceding try block.
except Exception as e:
# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph("Data Validation Run Failed", style="List Bullet")
# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph('')
# Print a message or value to the notebook output for validation or debugging.
    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph(f'{e}', style='Intense Quote')

# Original notebook comment retained.
    ## Upload log file
# Execute this line as part of the notebook's workflow logic.
    save_log(doc)

# Original notebook comment retained.
    ## Exit the notebook run
# Execute this line as part of the notebook's workflow logic.
    dbutils.notebook.exit("Data Validation Run Failed") 




## Command cell 33

This section corresponds to command 33 from the original Databricks notebook.


In [ ]:
# Original notebook comment retained.
#Netezza Tables with  Synapse Comparsion.
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.functions import *.
from pyspark.sql.functions import *
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.types import DecimalType, IntegerType, DateType, StructType, TimestampType.
from pyspark.sql.types import DecimalType, IntegerType, DateType, StructType, TimestampType
# Import specific objects from a module so they can be used directly in later code: from datetime import date, timedelta, datetime.
from datetime import date, timedelta, datetime
# Import specific objects from a module so they can be used directly in later code: from delta.tables import *.
from delta.tables import *

# Original notebook comment retained.
# Header for log document
# Add a heading to the Word document being used as a log or report.
doc.add_heading('KEY TYPE UNIQUE = N', 3)
# Assign the result on the right-hand side to `container` so it can be reused later.
container = 'synapse'

# Begin a protected block so the notebook can handle runtime errors more gracefully.
try:

# Start a loop so the same logic is applied repeatedly across multiple items.
    for db in relevant_gold_db:

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
        if Env == 'QA':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
            db_tmp = f"GDC_{db}"
# Check the next condition if the previous condition was not met.
        elif Env == 'PROD_DR':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
            db_tmp = f"{db}_DR"
# Run the fallback branch when the earlier conditions do not match.
        else:
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
            db_tmp = db
# Print a message or value to the notebook output for validation or debugging.
        print('DATABASE: ', db)


# Start a loop so the same logic is applied repeatedly across multiple items.
        for table_dict in [v for d in validation_tables_info_2 for k,v in d.items() if k == db][0]:
# Original notebook comment retained.
            #print(table_dict)
# Assign the result on the right-hand side to `table_name` so it can be reused later.
            table_name=list(table_dict.keys())[0]
# Print a message or value to the notebook output for validation or debugging.
            print('TABLE:',table_name)


# Begin a protected block so the notebook can handle runtime errors more gracefully.
            try:

# Assign the result on the right-hand side to `Partition_column` so it can be reused later.
                Partition_column=spark.sql(f"select distinct PARTITION_COLUMN from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_COLUMN is not null")
# Assign the result on the right-hand side to `KEY_TYPE` so it can be reused later.
                KEY_TYPE=spark.sql(f"select distinct KEY_TYPE_UNIQUE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and KEY_TYPE_UNIQUE is not null")
# Assign the result on the right-hand side to `KEY_TYPES` so it can be reused later.
                KEY_TYPES=KEY_TYPE.collect()[0][0]



# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                if KEY_TYPES == "N" :
# Assign the result on the right-hand side to `tab_sy_cnt +` so it can be reused later.
                    tab_sy_cnt += 1
# Print a message or value to the notebook output for validation or debugging.
                    print(tab_sy_cnt)

# Add a heading to the Word document being used as a log or report.
                    doc.add_heading(f"{db}.{table_name}", 4)

# Original notebook comment retained.
                    ## Check if a partition column is configured for this table
# Print a message or value to the notebook output for validation or debugging.
                    print(f'Checking if a partition column is configured for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'Checking if a partition column is configured for {db}.{table_name}', style='List Bullet')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                    if Partition_column.count() == 0:
# Assign the result on the right-hand side to `Partition_col` so it can be reused later.
                        Partition_col = None
# Print a message or value to the notebook output for validation or debugging.
                        print(f'No partition column configured for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                        doc.add_paragraph(f'No partition column configured for {db}.{table_name}', style='List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                    else:
# Assign the result on the right-hand side to `Partition_col` so it can be reused later.
                        Partition_col =Partition_column.collect()[0][0]
# Print a message or value to the notebook output for validation or debugging.
                        print(f'Partition column configured for {db}.{table_name} is {Partition_col}')
# Add a paragraph to the Word document so the log captures another message or detail.
                        doc.add_paragraph(f'Partition column configured for {db}.{table_name} is {Partition_col}', style='List Bullet')


# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                    if ((Partition_col != None) and (Partition_col != 'NA') ):
# Original notebook comment retained.
                        ## Retrieve partition, measure and field validation queries from metadata table
# Print a message or value to the notebook output for validation or debugging.
                        print(f'Retrieving Partition, Measure and Field Validation Queries from Fwk_Gold_Tables for {db}.{table_name}')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (PL_TL == '7GS'):  
# Assign the result on the right-hand side to `Query_Agg_Netezza` so it can be reused later.
                            Query_Agg_Netezza=spark.sql(f"select distinct PARTITON_QUERY_NETEZZA_SYNAPSE_SEVEN_YEARS from gold_df \
# Assign the result on the right-hand side to `where TABLE_NAME` so it can be reused later.
                                                    where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITON_QUERY_NETEZZA_SYNAPSE_SEVEN_YEARS is not null")
# Assign the result on the right-hand side to `Query_Agg_Synapse` so it can be reused later.
                            Query_Agg_Synapse=spark.sql(f"select distinct PARTITION_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}'\
# This line is part of the SQL statement being built for the data comparison or extraction step.
                                                    and PARTITION_QUERY_SYNAPSE is not null") 
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f"select distinct MEASURE_QUERY_NETEZZA_SEVEN_YEARS from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}'\
# This line is part of the SQL statement being built for the data comparison or extraction step.
                                                            and MEASURE_QUERY_NETEZZA_SEVEN_YEARS is not null")
# Assign the result on the right-hand side to `Query_Measure_Synapse` so it can be reused later.
                            Query_Measure_Synapse=spark.sql(f"select distinct MEASURE_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}'\
# This line is part of the SQL statement being built for the data comparison or extraction step.
                                                    and MEASURE_QUERY_SYNAPSE is not null")
# Start a new indented code block for the statement above.
                        else :
# Original notebook comment retained.
                            #below is for NON transactional and transactional where we have full data in synapse as well
# Assign the result on the right-hand side to `Query_Agg_Netezza` so it can be reused later.
                            Query_Agg_Netezza=spark.sql(f"select distinct PARTITION_QUERY_NETEZZA from gold_df where TABLE_NAME = '{table_name}' and DATABASE ='{db}' and PARTITION_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Agg_Synapse` so it can be reused later.
                            Query_Agg_Synapse=spark.sql(f"select distinct PARTITION_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_QUERY_SYNAPSE is not null") 
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f" select distinct MEASURE_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Synapse` so it can be reused later.
                            Query_Measure_Synapse=spark.sql(f" select distinct MEASURE_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_SYNAPSE is not null")


# Assign the result on the right-hand side to `Query_Field_Level_Netezza` so it can be reused later.
                        Query_Field_Level_Netezza=spark.sql(f"select distinct FIELD_VALIDATION_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and FIELD_VALIDATION_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Field_Level_Synapse` so it can be reused later.
                        Query_Field_Level_Synapse=spark.sql(f"select distinct FIELD_VALIDATION_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and FIELD_VALIDATION_QUERY_SYNAPSE is not null")   

# Execute this line as part of the notebook's workflow logic.
                        '''
# Original notebook comment retained.
                        ## Retrieve dataframe of the field used for validation (YEAR, MONTH, DAY)
# Assign the result on the right-hand side to `PARTITION_ON` so it can be reused later.
                        PARTITION_ON=spark.sql(f"select PARTITION_ON from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_ON is not null")
# Assign the result on the right-hand side to `Partition_Type` so it can be reused later.
                        Partition_Type=PARTITION_ON.collect()[0][0].upper()
# Print a message or value to the notebook output for validation or debugging.
                        print('PARTITON TYPE:', Partition_Type)
# Execute this line as part of the notebook's workflow logic.
                        '''

# Original notebook comment retained.
                        ## Skip table if no queries are configured
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (Query_Field_Level_Netezza.count() == 0) & (Query_Measure_Netezza.count() == 0) & (Query_Agg_Netezza.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                            print(f"No field validation query or measure query configured. No detailed report generated for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f"No field validation query or measure query configured. No detailed report generated for {db}.{table_name}", style = 'List Bullet')
# Execute this line as part of the notebook's workflow logic.
                            continue

# Original notebook comment retained.
                        ## PERFORM PARTITION COUNT VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (Query_Agg_Netezza.count() == 0) | (Query_Agg_Synapse.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                            print(f'No partition queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No partition count validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'No partition queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No partition count validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                            partition_count_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                        else:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                            query_PARTITION_CNT_NZ = Query_Agg_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                            query_PARTITION_CNT_SY = Query_Agg_Synapse.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (query_PARTITION_CNT_NZ == '') | (query_PARTITION_CNT_SY == ''):
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                print('No Partition Query is configured. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(('No Partition Query is configured. No partition count validation performed'), style= 'List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Attempting to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Attempting to perform partition count validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                    query_PARTITION_CNT_NZ = query_PARTITION_CNT_NZ.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_partition_cnt_df` so it can be reused later.
                                    netezza_partition_cnt_df = readfromNetizza(Env, query_PARTITION_CNT_NZ, db)
# Assign the result on the right-hand side to `synapse_partition_cnt_df` so it can be reused later.
                                    synapse_partition_cnt_df = readfromSynapse(Env,query_PARTITION_CNT_SY)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df` so it can be reused later.
                                        partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df = find_Mismatched(netezza_partition_cnt_df,synapse_partition_cnt_df,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                        print(f"Successfully performed partition count validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"Successfully performed partition count validation for {db}.{table_name}", style = 'List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                        partition_count_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                        partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Failed to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Failed to perform partition count validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                    partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Bad partition query configured for {db}.{table_name}. Please correct the query. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Bad partition query configured for {db}.{table_name}. Please correct the query. No partition count validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')




# Execute this line as part of the notebook's workflow logic.
                        '''
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (PL_TL == '7GS'): 
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                            query_PARTITION_CNT_NZ = Query_Agg_Netezza.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if Partition_Type == 'YEAR' :
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                                query_PARTITION_CNT_SY= f"""select FORMAT({Partition_col}, 'yyyy') as {Partition_col}, count(*) as CNT from {db}.{table_name} group by FORMAT({Partition_col}, 'yyyy')"""
# Check the next condition if the previous condition was not met.
                            elif Partition_Type == 'MONTH' :
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                                query_PARTITION_CNT_SY=f"""select FORMAT({Partition_col}, 'yyyy MMM') as {Partition_col}, count(*) as CNT from {db}.{table_name} group by FORMAT({Partition_col}, 'yyyy MMM')"""
# Check the next condition if the previous condition was not met.
                            elif Partition_Type == 'DAY' :
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                                query_PARTITION_CNT_SY=f"""select FORMAT({Partition_col}, 'yyyy MMM dd') as {Partition_col}, count(*) as CNT from {db}.{table_name} group by FORMAT({Partition_col}, 'yyyy MMM dd')"""
# Run the fallback branch when the earlier conditions do not match.
                        else:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if Partition_Type == 'YEAR' :
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                query_PARTITION_CNT_NZ = f"""select to_char({Partition_col}, 'YYYY') as {Partition_col}, count(*) as CNT from {db_tmp}.{table_name} group by to_char({Partition_col}, 'YYYY')"""
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                                query_PARTITION_CNT_SY= f"""select FORMAT({Partition_col}, 'yyyy') as {Partition_col}, count(*) as CNT from {db}.{table_name} group by FORMAT({Partition_col}, 'yyyy')"""
# Check the next condition if the previous condition was not met.
                            elif Partition_Type == 'MONTH' :
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                query_PARTITION_CNT_NZ = f"""select to_char({Partition_col}, 'YYYY Mon') as {Partition_col}, count(*) as CNT from {db_tmp}.{table_name} group by to_char({Partition_col}, 'YYYY Mon')"""
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                                query_PARTITION_CNT_SY=f"""select FORMAT({Partition_col}, 'yyyy MMM') as {Partition_col}, count(*) as CNT from {db}.{table_name} group by FORMAT({Partition_col}, 'yyyy MMM')"""
# Check the next condition if the previous condition was not met.
                            elif Partition_Type == 'DAY' :
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                query_PARTITION_CNT_NZ = f"""select to_char({Partition_col}, 'YYYY Mon dd') as {Partition_col}, count(*) as CNT from {db_tmp}.{table_name} group by to_char({Partition_col}, 'YYYY Mon dd')"""
# Assign the result on the right-hand side to `query_PARTITION_CNT_SY` so it can be reused later.
                                query_PARTITION_CNT_SY=f"""select FORMAT({Partition_col}, 'yyyy MMM dd') as {Partition_col}, count(*) as CNT from {db}.{table_name} group by FORMAT({Partition_col}, 'yyyy MMM dd')"""

# Assign the result on the right-hand side to `netezza_partition_cnt_df` so it can be reused later.
                        netezza_partition_cnt_df = readfromNetizza(Env, query_PARTITION_CNT_NZ, db)
# Assign the result on the right-hand side to `synapse_partition_cnt_df` so it can be reused later.
                        synapse_partition_cnt_df = readfromSynapse(Env,query_PARTITION_CNT_SY)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                        try:
# Assign the result on the right-hand side to `partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df` so it can be reused later.
                            partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df = find_Mismatched(netezza_partition_cnt_df,synapse_partition_cnt_df,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                            print(f"Successfully performed partition count validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f"Successfully performed partition count validation for {db}.{table_name}", style = 'List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                            partition_count_validation_status = 'True'
# Handle an error raised in the preceding try block.
                        except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                            partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Failed to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Failed to perform partition count validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Execute this line as part of the notebook's workflow logic.
                        '''


# Original notebook comment retained.
                        ## PERFORM MEASURE LEVEL VALIDATION
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (Query_Measure_Netezza.count() == 0) | (Query_Measure_Synapse.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                            print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                            measure_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                        else:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                            query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_measure_synapse` so it can be reused later.
                            query_measure_synapse = Query_Measure_Synapse.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (query_measure_netezza == '') | (query_measure_synapse == ''):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                measure_validation_status = 'False' 
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Attempting to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Attempting to perform measure level validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                    query_measure_netezza = query_measure_netezza.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                    netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Assign the result on the right-hand side to `synapse_df_measure` so it can be reused later.
                                    synapse_df_measure=readfromSynapse(Env,query_measure_synapse)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff, measure_level_mismatched_record_df, measure_level_mismatched_detail_cnt, measure_level_mismatched_detail_df` so it can be reused later.
                                        measure_level_mismatched_cnt_diff, measure_level_mismatched_record_df, measure_level_mismatched_detail_cnt, measure_level_mismatched_detail_df = find_Mismatched(netezza_df_measure,synapse_df_measure,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                        print(f"Successfully performed measure level validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"Successfully performed measure level validation for {db}.{table_name}", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                        measure_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                        measure_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Failed to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Failed to perform measure level validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                    measure_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')




# Original notebook comment retained.
                        ## PERFORM PARTITION LEVEL VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (Query_Field_Level_Netezza.count()==0) | (Query_Field_Level_Synapse.count()==0):
# Print a message or value to the notebook output for validation or debugging.
                            print(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No partition level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No partition level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                            partition_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                        else:
# Assign the result on the right-hand side to `query_PARTITION_NZ` so it can be reused later.
                            query_PARTITION_NZ = Query_Field_Level_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_PARTITION_SY` so it can be reused later.
                            query_PARTITION_SY = Query_Field_Level_Synapse.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (query_PARTITION_NZ == '') | (query_PARTITION_SY == ''):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No partition level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No partition level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                partition_validation_status = 'False'  
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:
# Assign the result on the right-hand side to `query_PARTITION_NZ` so it can be reused later.
                                    query_PARTITION_NZ = query_PARTITION_NZ.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_partition` so it can be reused later.
                                    netezza_df_partition=readfromNetizza(Env,query_PARTITION_NZ,db)
# Assign the result on the right-hand side to `synapse_df_partition` so it can be reused later.
                                    synapse_df_partition=readfromSynapse(Env,query_PARTITION_SY)  

# Original notebook comment retained.
                                    ## Validate that the partition size is within the max row permitted limit
# Print a message or value to the notebook output for validation or debugging.
                                    print('Attempting to perform partition level validation. Need to confirm that the partition size is within acceptable limit')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph('Attempting to perform partition level validation. Need to confirm that the partition size is within acceptable limit', style = 'List Bullet')

# Assign the result on the right-hand side to `cnt_netezza_df_partition` so it can be reused later.
                                    cnt_netezza_df_partition = netezza_df_partition.limit(max_rows_permitted + 1).count()
# Assign the result on the right-hand side to `cnt_synapse_df_partition` so it can be reused later.
                                    cnt_synapse_df_partition = synapse_df_partition.limit(max_rows_permitted + 1).count()

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if (cnt_netezza_df_partition == 0) | (cnt_synapse_df_partition == 0):
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Partition for table {db}.{table_name} is empty either in Netezza or in Synapse. No partition validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Partition for table {db}.{table_name} is empty either in Netezza or in Synapse. No partition validation performed', style = 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                        partition_validation_status = 'False'
# Check the next condition if the previous condition was not met.
                                    elif (cnt_netezza_df_partition > max_rows_permitted) | (cnt_synapse_df_partition > max_rows_permitted):
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Partition for {db}.{table_name} is greater than {max_rows_permitted}. No partition validation performed. Please reconfigure the partition for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Partition for {db}.{table_name} is greater than {max_rows_permitted}. No partition validation performed. Please reconfigure the partition for {db}.{table_name}', style = 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                        partition_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Original notebook comment retained.
                                        ## Perform validation on the partition
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Partition for {db}.{table_name} is less than or equal to {max_rows_permitted}.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Partition for {db}.{table_name} is less than or equal to {max_rows_permitted}.', style = 'List Bullet')
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Original notebook comment retained.
                                            #partition_level_diff = get_pandas(netezza_df_partition.subtract(synapse_df_partition))
# Assign the result on the right-hand side to `partition_level_diff` so it can be reused later.
                                            partition_level_diff = get_pandas(trim_space(netezza_df_partition).subtract(trim_space(synapse_df_partition)))
# Assign the result on the right-hand side to `partition_level_diff_cnt` so it can be reused later.
                                            partition_level_diff_cnt = len(partition_level_diff)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed partition level validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed partition level validation for {db}.{table_name}", style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                            partition_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                            partition_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Failed to perform partiton level validation for {db}.{table_name}_(Netezza v. Synapse)')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Failed to perform partiton level validation for {db}.{table_name}_(Netezza v. Synapse)', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                except Exception as e:
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                    partition_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Bad field validation query configured for {db}.{table_name}. Please correct the query. No partition level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Bad field validation query configured for {db}.{table_name}. Please correct the query. No partition level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')


# Original notebook comment retained.
                        ## Define excel writer
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                        try:
# Open a context-managed resource so it is handled safely and closed automatically after use.
                            with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Synapse).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                ## Write missing and mismatched information to sheets in excel workbook
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if partition_validation_status == 'True':
# Assign the result on the right-hand side to `partition_level_diff.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                    partition_level_diff.head(max_excel_row).to_excel(writer, sheet_name = f"difference_record_partition", startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if measure_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.sort_values(by` so it can be reused later.
                                        measure_level_mismatched_record_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `measure_level_mismatched_detail_df.sort_values(by` so it can be reused later.
                                        measure_level_mismatched_detail_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                    except:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `measure_level_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        measure_level_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_detail", startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if partition_count_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `partition_count_mismatched_df.sort_values(by` so it can be reused later.
                                        partition_count_mismatched_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_count_mismatched_detail_df.sort_values(by` so it can be reused later.
                                        partition_count_mismatched_detail_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                    except:
# Assign the result on the right-hand side to `partition_count_mismatched_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        partition_count_mismatched_df.head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_count_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        partition_count_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_detail", startrow = 0, startcol=0, index = False)
# Print a message or value to the notebook output for validation or debugging.
                            print('Detailed report generated')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph('Detailed report generated', style = 'List Bullet')
# Assign the result on the right-hand side to `detailed_report_status` so it can be reused later.
                            detailed_report_status = 'True'
# Handle an error raised in the preceding try block.
                        except Exception as e:
# Assign the result on the right-hand side to `detailed_report_status` so it can be reused later.
                            detailed_report_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                            print('No detailed report generated')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph('No detailed report generated', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'{e}', style= 'Intense Quote')



# Original notebook comment retained.
                        ## Upload workbook to storage
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if detailed_report_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                workbook_path = path + f"{db}-{table_name}_(Netezza v. Synapse).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                shutil.copy2(f"{db}-{table_name}_(Netezza v. Synapse).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                os.remove(f"{db}-{table_name}_(Netezza v. Synapse).xlsx")

# Original notebook comment retained.
                                ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_table_name.append(table_name)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if partition_count_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_count_mismatched_cnt.append(str(partition_count_mismatched_cnt))
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_count_mismatched_cnt.append('')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if measure_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_measure_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_table_level_missing_cnt.append('')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if partition_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_level_diff_cnt.append(str(partition_level_diff_cnt))
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                netezza_synapse_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                print(f"Successfully uploaded detailed report for {db}-{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f"Successfully uploaded detailed report for {db}-{table_name}_(Netezza v. Synapse)", style='List Bullet')


# Handle an error raised in the preceding try block.
                            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Synapse)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'{e}', style='Intense Quote')





# Start a new indented code block for the statement above.
                    else :
# Original notebook comment retained.
                        ## This condition is to perform detailed validation for small and medium size tables
# Print a message or value to the notebook output for validation or debugging.
                        print(f'Attempting to perform detailed validation on {db}.{table_name}_(Netezza v. Synapse). Need to confirm it is a small to medium size table')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (PL_TL == '7GS'):

# Original notebook comment retained.
                            ## Retrieve measure level query
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f"select distinct MEASURE_QUERY_NETEZZA_SEVEN_YEARS from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}'\
# This line is part of the SQL statement being built for the data comparison or extraction step.
                                                            and MEASURE_QUERY_NETEZZA_SEVEN_YEARS is not null")
# Assign the result on the right-hand side to `Query_Measure_Synapse` so it can be reused later.
                            Query_Measure_Synapse=spark.sql(f"select distinct MEASURE_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}'\
# This line is part of the SQL statement being built for the data comparison or extraction step.
                                                    and MEASURE_QUERY_SYNAPSE is not null")

# Original notebook comment retained.
                            ## Retrieve the retention column
# Execute this line as part of the notebook's workflow logic.
                            filterColumn=gold_df.select('Filter7').where((gold_df.TABLE_NAME==table_name)&(gold_df.DATABASE==db)).distinct().rdd.map(lambda x: x.Filter7).collect()
# Assign the result on the right-hand side to `filterColumn` so it can be reused later.
                            filterColumn=str(filterColumn)
# Assign the result on the right-hand side to `filterColumn` so it can be reused later.
                            filterColumn=filterColumn.replace("[","").replace("]","").replace("'","")
# Original notebook comment retained.
                            ## BELOW is fetch the minimum date from synapse to avoid mismatch on the day where data is not loaded in netizza
# Print a message or value to the notebook output for validation or debugging.
                            print(filterColumn)

# Original notebook comment retained.
                            ## Retrieve the retention value
# Assign the result on the right-hand side to `query_forValuefilter` so it can be reused later.
                            query_forValuefilter= f"""select CAST(min({filterColumn}) as varchar(10)) {filterColumn} from {db}.{table_name}""" 

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if filterColumn!='None':
# Assign the result on the right-hand side to `filterValue1` so it can be reused later.
                                filterValue1=readfromSynapse(Env,query_forValuefilter)
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                                filterValue=filterValue1.rdd.map(lambda x: x[0]).collect()
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                                filterValue=str(filterValue)
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                                filterValue=filterValue.replace("[","").replace("]","")
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `filterValue` so it can be reused later.
                                filterValue='None'

# Original notebook comment retained.
                            ## Query netezza table based on retention values. Get netezza table for validation
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if filterValue!='None':
# Assign the result on the right-hand side to `Query_Netezza` so it can be reused later.
                                Query_Netezza = f"""select * from {db_tmp}.{table_name} where {filterColumn} >= {filterValue}""" 
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `Query_Netezza` so it can be reused later.
                                Query_Netezza = f"""select * from {db_tmp}.{table_name}"""
# Original notebook comment retained.
                            #below is filter for 7 years for netizza 
# Print a message or value to the notebook output for validation or debugging.
                            print(Query_Netezza)
# Assign the result on the right-hand side to `netezza_df` so it can be reused later.
                            netezza_df=readfromNetizza(Env,Query_Netezza,db)

# Original notebook comment retained.
                            ## Get synapse table for validation
# Assign the result on the right-hand side to `Query_Synapse` so it can be reused later.
                            Query_Synapse = f"""select * from {db}.{table_name}"""
# Assign the result on the right-hand side to `df_synapse` so it can be reused later.
                            df_synapse=readfromSynapse(Env,Query_Synapse)

# Run the fallback branch when the earlier conditions do not match.
                        else:
# Original notebook comment retained.
                            ## Retrieve measure level query
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f" select distinct MEASURE_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Synapse` so it can be reused later.
                            Query_Measure_Synapse=spark.sql(f" select distinct MEASURE_QUERY_SYNAPSE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_SYNAPSE is not null")

# Print a message or value to the notebook output for validation or debugging.
                            print("else Full load Non Transactional ")
# Original notebook comment retained.
                            ## Get netezza table for validation
# Assign the result on the right-hand side to `Query_Netezza` so it can be reused later.
                            Query_Netezza = f"""select * from {db_tmp}.{table_name}"""
# Assign the result on the right-hand side to `netezza_df` so it can be reused later.
                            netezza_df=readfromNetizza(Env,Query_Netezza,db)

# Original notebook comment retained.
                            ## Get synapse table for validation
# Assign the result on the right-hand side to `Query_Synapse` so it can be reused later.
                            Query_Synapse = f"""select * from {db}.{table_name}"""
# Original notebook comment retained.
                            #print(Query_Synapse)
# Assign the result on the right-hand side to `df_synapse` so it can be reused later.
                            df_synapse=readfromSynapse(Env,Query_Synapse)


# Original notebook comment retained.
                        ## Confirm table is small or medium size (i.e. Not greater than 5 million rows)
# Original notebook comment retained.
                        #max_rows_permitted = 5000000

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                        try:
# Original notebook comment retained.
                            ## Retrieve row count
# Execute this line as part of the notebook's workflow logic.
                            cnt_netezza_df = int(report_dataframe_1.select(f'NETEZZA_TABLE_ROW_COUNT ({netezza_sub})').filter((col('NETEZZA_SCHEMA') == f'{db}') & (col('NETEZZA_TABLE') == f'{table_name}')).collect()[0][0])   #netezza_df.limit(max_rows_permitted + 1).count()
# Execute this line as part of the notebook's workflow logic.
                            cnt_df_synapse = int(report_dataframe_1.select('SYNAPSE_TABLE_ROW_COUNT').filter((col('SYNAPSE_SCHEMA') == f'{db}') & (col('SYNAPSE_TABLE') == f'{table_name}')).collect()[0][0])  #df_synapse.limit(max_rows_permitted + 1).count()
# Handle an error raised in the preceding try block.
                        except:
# Print a message or value to the notebook output for validation or debugging.
                            print(f'{db}.{table_name} not available in summary report')
# Assign the result on the right-hand side to `cnt_netezza_df` so it can be reused later.
                            cnt_netezza_df = netezza_df.limit(max_rows_permitted + 1).count()
# Assign the result on the right-hand side to `cnt_df_synapse` so it can be reused later.
                            cnt_df_synapse = df_synapse.limit(max_rows_permitted + 1).count()


# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if (cnt_netezza_df == 0) | (cnt_df_synapse == 0):
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Table {db}.{table_name} is empty either in Netezza or in Synapse. No detailed validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Table {db}.{table_name} is empty either in Netezza or in Synapse. No detailed validation performed', style = 'List Bullet')

# Check the next condition if the previous condition was not met.
                        elif (cnt_netezza_df > max_rows_permitted) | (cnt_df_synapse > max_rows_permitted):
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Table {db}.{table_name}_(Netezza v. Synapse) is a big table, with no partition column configures. No table level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Table {db}.{table_name}_(Netezza v. Synapse) is a big table, with no partition column configures. No table level validation performed', style='List Bullet')

# Print a message or value to the notebook output for validation or debugging.
                            print('Will attempt to perform measure level validation')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph('Attempting measure level validation for a big table with no partition configured', style= 'List Bullet')

# Original notebook comment retained.
                            ## PERFORM MEASURE LEVEL VALIDATION
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Measure_Netezza.count() == 0) | (Query_Measure_Synapse.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                measure_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_measure_synapse` so it can be reused later.
                                query_measure_synapse = Query_Measure_Synapse.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_measure_netezza == '') | (query_measure_synapse == ''):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or Synapse. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                    measure_validation_status = 'False' 
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Measure queries configured. Attempting to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Measure queries configured. Attempting to perform measure level validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                        query_measure_netezza = query_measure_netezza.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                        netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Assign the result on the right-hand side to `synapse_df_measure` so it can be reused later.
                                        synapse_df_measure=readfromSynapse(Env,query_measure_synapse)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df` so it can be reused later.
                                            measure_level_mismatched_record_df = get_pandas(synapse_df_measure.subtract(netezza_df_measure))
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff` so it can be reused later.
                                            measure_level_mismatched_cnt_diff = len(measure_level_mismatched_record_df)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                            measure_validation_status = 'True'

# Original notebook comment retained.
                                            ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                            with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Synapse).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                                ## Write missing and mismatched information to sheets in excel workbook
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                                measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = f"measure_mismatched_record", startrow = 0, startcol=0, index = False)

# Original notebook comment retained.
                                            ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                            try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                                workbook_path = path + f"{db}-{table_name}_(Netezza v. Synapse).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                                shutil.copy2(f"{db}-{table_name}_(Netezza v. Synapse).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                                os.remove(f"{db}-{table_name}_(Netezza v. Synapse).xlsx")

# Original notebook comment retained.
                                                ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                netezza_synapse_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Successfully uploaded detailed report for {db}-{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Successfully uploaded detailed report for {db}-{table_name}_(Netezza v. Synapse)", style='List Bullet')

# Handle an error raised in the preceding try block.
                                            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Failed to upload detailed report for {db}-{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Failed to upload detailed report for {db}-{table_name}_(Netezza v. Synapse)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Failed to perform detailed validation on {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Failed to perform detailed validation on {db}.{table_name}", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Execute this line as part of the notebook's workflow logic.
                                            continue                                   
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')



# Run the fallback branch when the earlier conditions do not match.
                        else:
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Table size is less than or equal to {max_rows_permitted}. Performing detailed validation on full table {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Table size is less than or equal to {max_rows_permitted}. Performing detailed validation on full table {db}.{table_name}', style = 'List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:
# Original notebook comment retained.
                                #table_level_diff = get_pandas(netezza_df.subtract(df_synapse))
# Assign the result on the right-hand side to `table_level_diff` so it can be reused later.
                                table_level_diff = get_pandas(trim_space(netezza_df).subtract(trim_space(df_synapse)))
# Assign the result on the right-hand side to `table_level_diff_cnt` so it can be reused later.
                                table_level_diff_cnt = len(table_level_diff)
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Successfully performed full table validation on {db}.{table_name}, comparing Netezza to Synapse table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Successfully performed full table validation on {db}.{table_name}, comparing Netezza to Synapse table', style='List Bullet')

# Original notebook comment retained.
                                ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Synapse).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                    ## Write missing and mismatched information to sheets in excel workbook
# Assign the result on the right-hand side to `table_level_diff.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                    table_level_diff.head(max_excel_row).to_excel(writer, sheet_name = "difference_record", startrow = 0, startcol=0, index = False)


# Original notebook comment retained.
                                ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                    workbook_path = path + f"{db}-{table_name}_(Netezza v. Synapse).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                    shutil.copy2(f"{db}-{table_name}_(Netezza v. Synapse).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                    os.remove(f"{db}-{table_name}_(Netezza v. Synapse).xlsx")

# Original notebook comment retained.
                                    ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_measure_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_table_level_diff_cnt.append(table_level_diff_cnt)
# Execute this line as part of the notebook's workflow logic.
                                    netezza_synapse_detailed_report_path.append(workbook_path)   

# Print a message or value to the notebook output for validation or debugging.
                                    print(f"Successfully uploaded detailed report for {db}-{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"Successfully uploaded detailed report for {db}-{table_name}_(Netezza v. Synapse)", style='List Bullet')


# Handle an error raised in the preceding try block.
                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f"Failed to upload detailed report for {db}-{table_name}_(Netezza v. Synapse)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"Failed to upload detailed report for {db}-{table_name}_(Netezza v. Synapse)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
                            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Synapse table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Synapse table', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                print(f'ERROR OCCURRED: Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Synapse table')
# Add a paragraph to the Word document so the log captures another message or detail.
                doc.add_paragraph(f'ERROR OCCURRED: Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Synapse table', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                doc.add_paragraph(f'{e}', style='Intense Quote')
# Original notebook comment retained.
                #continue                                                       

# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph()

# Original notebook comment retained.
    ## Upload log file
# Execute this line as part of the notebook's workflow logic.
    save_log(doc)

# Handle an error raised in the preceding try block.
except Exception as e:
# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph("Data Validation Run Failed", style="Intense Quote")
# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph('')
# Print a message or value to the notebook output for validation or debugging.
    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph(f'{e}', style='Intense Quote')

# Original notebook comment retained.
    ## Upload log file
# Execute this line as part of the notebook's workflow logic.
    save_log(doc)

# Original notebook comment retained.
    ## Exit the notebook run
# Execute this line as part of the notebook's workflow logic.
    dbutils.notebook.exit("Data Validation Run Failed") 





## Command cell 34

This section corresponds to command 34 from the original Databricks notebook.


In [ ]:
# Assign the result on the right-hand side to `netezza_gold_database_name` so it can be reused later.
netezza_gold_database_name = []
# Assign the result on the right-hand side to `netezza_gold_table_name` so it can be reused later.
netezza_gold_table_name = []
# Assign the result on the right-hand side to `netezza_gold_partition_count_mismatched_cnt` so it can be reused later.
netezza_gold_partition_count_mismatched_cnt = []
# Assign the result on the right-hand side to `netezza_gold_measure_level_mismatched_cnt_diff` so it can be reused later.
netezza_gold_measure_level_mismatched_cnt_diff = []
# Assign the result on the right-hand side to `netezza_gold_partition_level_mismatched_cnt_diff` so it can be reused later.
netezza_gold_partition_level_mismatched_cnt_diff = []
# Assign the result on the right-hand side to `netezza_gold_partition_level_missing_cnt` so it can be reused later.
netezza_gold_partition_level_missing_cnt = []
# Assign the result on the right-hand side to `netezza_gold_table_level_mismatched_cnt_diff` so it can be reused later.
netezza_gold_table_level_mismatched_cnt_diff = []
# Assign the result on the right-hand side to `netezza_gold_table_level_missing_cnt` so it can be reused later.
netezza_gold_table_level_missing_cnt = []
# Assign the result on the right-hand side to `netezza_gold_measure_level_diff_cnt` so it can be reused later.
netezza_gold_measure_level_diff_cnt = []
# Assign the result on the right-hand side to `netezza_gold_partition_level_diff_cnt` so it can be reused later.
netezza_gold_partition_level_diff_cnt = []
# Assign the result on the right-hand side to `netezza_gold_table_level_diff_cnt` so it can be reused later.
netezza_gold_table_level_diff_cnt = []
# Assign the result on the right-hand side to `netezza_gold_detailed_report_path` so it can be reused later.
netezza_gold_detailed_report_path = []

## Command cell 35

This section corresponds to command 35 from the original Databricks notebook.


In [ ]:
# Assign the result on the right-hand side to `tab_gd_cnt` so it can be reused later.
tab_gd_cnt = 0

## Command cell 36

This section corresponds to command 36 from the original Databricks notebook.


In [ ]:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if (PL_TL != '7GS') & (Load_Type == 'Historical'): 

# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.functions import *.
    from pyspark.sql.functions import *
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.types import DecimalType, IntegerType, DateType, StructType, TimestampType.
    from pyspark.sql.types import DecimalType, IntegerType, DateType, StructType, TimestampType
# Import specific objects from a module so they can be used directly in later code: from datetime import date, timedelta, datetime.
    from datetime import date, timedelta, datetime
# Import specific objects from a module so they can be used directly in later code: from delta.tables import *.
    from delta.tables import *

# Original notebook comment retained.
    # Header for log document
# Add a heading to the Word document being used as a log or report.
    doc.add_heading('Log From Detailed Report Generation Comparing Netezza v. Gold', 2)
# Add a heading to the Word document being used as a log or report.
    doc.add_heading('KEY TYPE UNIQUE = Y',3)
# Assign the result on the right-hand side to `container` so it can be reused later.
    container = 'gold'

# Begin a protected block so the notebook can handle runtime errors more gracefully.
    try:

# Start a loop so the same logic is applied repeatedly across multiple items.
        for db in relevant_gold_db:

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
            if Env == 'QA':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = f"GDC_{db}"
# Check the next condition if the previous condition was not met.
            elif Env == 'PROD_DR':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = f"{db}_DR"
# Run the fallback branch when the earlier conditions do not match.
            else:
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = db
# Print a message or value to the notebook output for validation or debugging.
            print('DATABASE: ', db)


# Start a loop so the same logic is applied repeatedly across multiple items.
            for table_dict in [v for d in validation_tables_info_2 for k,v in d.items() if k == db][0]:
# Print a message or value to the notebook output for validation or debugging.
                print(table_dict)
# Assign the result on the right-hand side to `table_name` so it can be reused later.
                table_name=list(table_dict.keys())[0]
# Print a message or value to the notebook output for validation or debugging.
                print('TABLE NAME: ',table_name)   

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                try:      
# Assign the result on the right-hand side to `partition_key` so it can be reused later.
                    partition_key=list(table_dict.values())[0][0]
# Print a message or value to the notebook output for validation or debugging.
                    print(type(partition_key))
# Assign the result on the right-hand side to `primary_key` so it can be reused later.
                    primary_key=list(table_dict.values())[0]
# Print a message or value to the notebook output for validation or debugging.
                    print('PRIMARY KEY: ',primary_key)
# Assign the result on the right-hand side to `Partition_column` so it can be reused later.
                    Partition_column=spark.sql(f"select distinct PARTITION_COLUMN from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_COLUMN is not null")
# Assign the result on the right-hand side to `KEY_TYPE` so it can be reused later.
                    KEY_TYPE=spark.sql(f"select distinct KEY_TYPE_UNIQUE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and KEY_TYPE_UNIQUE is not null")
# Assign the result on the right-hand side to `KEY_TYPES` so it can be reused later.
                    KEY_TYPES=KEY_TYPE.collect()[0][0]
# Print a message or value to the notebook output for validation or debugging.
                    print('KEY_TYPES: ',KEY_TYPES)



# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                    if KEY_TYPES == "Y" :
# Assign the result on the right-hand side to `tab_gd_cnt +` so it can be reused later.
                        tab_gd_cnt += 1
# Print a message or value to the notebook output for validation or debugging.
                        print(tab_gd_cnt)

# Add a heading to the Word document being used as a log or report.
                        doc.add_heading(f"{db}.{table_name}", 4)

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if Partition_column.count() == 0:
# Assign the result on the right-hand side to `Partition_col` so it can be reused later.
                            Partition_col = None
# Print a message or value to the notebook output for validation or debugging.
                            print(f'No partition column configured for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'No partition column configured for {db}.{table_name}', style='List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                        else:
# Assign the result on the right-hand side to `Partition_col` so it can be reused later.
                            Partition_col =Partition_column.collect()[0][0]
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Partition column configured for {db}.{table_name} is {Partition_col}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Partition column configured for {db}.{table_name} is {Partition_col}', style='List Bullet')


# Original notebook comment retained.
                        ## The ADLS gold table
# Read data from storage into a DataFrame or Python object.
                        Data_Lake_df=spark.read.format("delta").option("header","true").load("/mnt/gold/"+db+"/ADMIN/"+table_name)
# Execute this line as part of the notebook's workflow logic.
                        Data_Lake_df.createOrReplaceTempView("Data_Lake_df")

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if ((Partition_col != None) and (Partition_col != 'NA') ):
# Original notebook comment retained.
                            ## Retrieve dataframe of the queries for row count per partition for Netezza tables, and Retrieve dataframe of query for generating a measure from Netezza and ADLS_Gold
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Retrieving Partition, Measure and Field Validation Queries from Fwk_Gold_Tables for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Retrieving Partition, Measure and Field Validation Queries from Fwk_Gold_Tables for {db}.{table_name}', style='List Bullet')

# Assign the result on the right-hand side to `Query_Agg_Netezza` so it can be reused later.
                            Query_Agg_Netezza=spark.sql(f"select distinct PARTITION_QUERY_NETEZZA from gold_df where TABLE_NAME = '{table_name}' and DATABASE ='{db}' and PARTITION_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f" select distinct MEASURE_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Gold` so it can be reused later.
                            Query_Measure_Gold=spark.sql(f" select distinct ADLS_QUERY_FOR_MEASURE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and ADLS_QUERY_FOR_MEASURE is not null")

# Assign the result on the right-hand side to `Query_Field_Level_Netezza` so it can be reused later.
                            Query_Field_Level_Netezza=spark.sql(f"select distinct FIELD_VALIDATION_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and FIELD_VALIDATION_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Field_Level_Gold` so it can be reused later.
                            Query_Field_Level_Gold=spark.sql(f"select distinct ADLS_QUERY_FOR_FIELD from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and ADLS_QUERY_FOR_FIELD is not null")  



# Original notebook comment retained.
                            ## Retrieve dataframe of the field used for validation (YEAR, MONTH, DAY)
# Assign the result on the right-hand side to `PARTITION_ON` so it can be reused later.
                            PARTITION_ON=spark.sql(f"select PARTITION_ON from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_ON is not null")
# Assign the result on the right-hand side to `Partition_Type` so it can be reused later.
                            Partition_Type=PARTITION_ON.collect()[0][0].upper()
# Print a message or value to the notebook output for validation or debugging.
                            print('PARTITION TYPE:', Partition_Type)



# Original notebook comment retained.
                            ## Skip table if no queries are configured
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Field_Level_Netezza.count() == 0) & (Query_Measure_Netezza.count() == 0) & (Query_Agg_Netezza.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f"No field validation query or measure query or partition query configured. No detailed report generated for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f"No field validation query or measure query or partition query configured. No detailed report generated for {db}.{table_name}", style = 'List Bullet')
# Execute this line as part of the notebook's workflow logic.
                                continue


# Original notebook comment retained.
                            ## PERFORM PARTITION COUNT VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Agg_Netezza.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No partition queries configured for {db}.{table_name} in Fwk_Gold_Tables for Netezza. No partition count validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No partition queries configured for {db}.{table_name} in Fwk_Gold_Tables for Netezza. No partition count validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                partition_count_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                query_PARTITION_CNT_NZ = Query_Agg_Netezza.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_PARTITION_CNT_NZ == ''):
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                    partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                    print('No Partition Query is configured. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(('No Partition Query is configured. No partition count validation performed'), style= 'List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Attempting to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Attempting to perform partition count validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                        query_PARTITION_CNT_NZ = query_PARTITION_CNT_NZ.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_partition_cnt_df` so it can be reused later.
                                        netezza_partition_cnt_df = readfromNetizza(Env, query_PARTITION_CNT_NZ, db)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                        if Partition_Type == 'YEAR' :
# Assign the result on the right-hand side to `Data_Lake_df_PARTITION_CNT` so it can be reused later.
                                            Data_Lake_df_PARTITION_CNT=spark.sql(f"select DATE_FORMAT({Partition_col}, 'yyyy') as {Partition_col}, count(*) as CNT from Data_Lake_df group by DATE_FORMAT({Partition_col}, 'yyyy')")
# Check the next condition if the previous condition was not met.
                                        elif Partition_Type == 'MONTH' :
# Assign the result on the right-hand side to `Data_Lake_df_PARTITION_CNT` so it can be reused later.
                                            Data_Lake_df_PARTITION_CNT=spark.sql(f"select DATE_FORMAT({Partition_col}, 'yyyy MMM') as {Partition_col}, count(*) as CNT from Data_Lake_df group by DATE_FORMAT({Partition_col}, 'yyyy MMM')")
# Check the next condition if the previous condition was not met.
                                        elif Partition_Type == 'DAY' :
# Assign the result on the right-hand side to `Data_Lake_df_PARTITION_CNT` so it can be reused later.
                                            Data_Lake_df_PARTITION_CNT=spark.sql(f"select DATE_FORMAT({Partition_col}, 'yyyy MMM dd') as {Partition_col}, count(*) as CNT from Data_Lake_df group by DATE_FORMAT({Partition_col}, 'yyyy MMM dd')")

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df` so it can be reused later.
                                            partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df = find_Mismatched(netezza_partition_cnt_df,Data_Lake_df_PARTITION_CNT,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed partition count validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed partition count validation for {db}.{table_name}", style = 'List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                            partition_count_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                            partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Failed to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Failed to perform partition count validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                        partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad partition query configured for {db}.{table_name}. Please correct the query. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad partition query configured for {db}.{table_name}. Please correct the query. No partition count validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')



# Original notebook comment retained.
                            ## PERFORM MEASURE LEVEL VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Measure_Netezza.count() == 0) | (Query_Measure_Gold.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                measure_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_measure_gold` so it can be reused later.
                                query_measure_gold = Query_Measure_Gold.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_measure_netezza == '') | (query_measure_gold == ''):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                    measure_validation_status = 'False' 
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Attempting to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Attempting to perform measure level validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                        query_measure_netezza = query_measure_netezza.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                        netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Assign the result on the right-hand side to `adls_gold_df_measure` so it can be reused later.
                                        adls_gold_df_measure = spark.sql(f"{query_measure_gold}")

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff, measure_level_mismatched_record_df, measure_level_mismatched_detail_cnt, measure_level_mismatched_detail_df` so it can be reused later.
                                            measure_level_mismatched_cnt_diff, measure_level_mismatched_record_df, measure_level_mismatched_detail_cnt, measure_level_mismatched_detail_df = find_Mismatched(netezza_df_measure,adls_gold_df_measure,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed measure level validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed measure level validation for {db}.{table_name}", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                            measure_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                            measure_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Failed to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Failed to perform measure level validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                        measure_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')



# Original notebook comment retained.
                            ## PERFORM PARTITION LEVEL VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Field_Level_Netezza.count()==0) | (Query_Field_Level_Gold.count()==0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                partition_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_PARTITION_NZ` so it can be reused later.
                                query_PARTITION_NZ = Query_Field_Level_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_PARTITION_Gold` so it can be reused later.
                                query_PARTITION_Gold = Query_Field_Level_Gold.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_PARTITION_NZ == '') | (query_PARTITION_Gold == ''):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                    partition_validation_status = 'False'  
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_PARTITION_NZ` so it can be reused later.
                                        query_PARTITION_NZ = query_PARTITION_NZ.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_partition` so it can be reused later.
                                        netezza_df_partition=readfromNetizza(Env,query_PARTITION_NZ,db)
# Assign the result on the right-hand side to `Data_Lake_df_1` so it can be reused later.
                                        Data_Lake_df_1=spark.sql(query_PARTITION_Gold)  

# Original notebook comment retained.
                                        ## Validate that the partition size is within the max row permitted limit
# Print a message or value to the notebook output for validation or debugging.
                                        print('Attempting to perform partition level validation. Need to confirm that the partition size is within acceptable limit')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph('Attempting to perform partition level validation. Need to confirm that the partition size is within acceptable limit', style = 'List Bullet')

# Assign the result on the right-hand side to `cnt_netezza_df_partition` so it can be reused later.
                                        cnt_netezza_df_partition = netezza_df_partition.limit(max_rows_permitted + 1).count()
# Assign the result on the right-hand side to `cnt_Data_Lake_df_1` so it can be reused later.
                                        cnt_Data_Lake_df_1 = Data_Lake_df_1.limit(max_rows_permitted + 1).count()

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                        if (cnt_netezza_df_partition == 0) | (cnt_Data_Lake_df_1 == 0):
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Partition for table {db}.{table_name} is empty either in Netezza or in Gold. No partition validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Partition for table {db}.{table_name} is empty either in Netezza or in Gold. No partition validation performed', style = 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                            partition_validation_status = 'False'

# Check the next condition if the previous condition was not met.
                                        elif (cnt_netezza_df_partition > max_rows_permitted) | (cnt_Data_Lake_df_1 > max_rows_permitted):
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Partition for {db}.{table_name} is greater than {max_rows_permitted}. No partition validation performed. Please reconfigure the partition for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Partition for {db}.{table_name} is greater than {max_rows_permitted}. No partition validation performed. Please reconfigure the partition for {db}.{table_name}', style = 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                            partition_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                                        else:
# Original notebook comment retained.
                                            ## Perform validation on the partition
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Partition for {db}.{table_name} is less than or equal to {max_rows_permitted}.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Partition for {db}.{table_name} is less than or equal to {max_rows_permitted}.', style = 'List Bullet')
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                            try:
# Original notebook comment retained.
                                                #partition_level_mismatched_cnt_diff, partition_level_mismatched_record_df, partition_level_mismatched_record_detail_cnt, partition_level_mismatched_record_detail_df, partition_level_missing_cnt, partition_level_missing_output_df = find_difference(netezza_df_partition, Data_Lake_df_1, primary_key)
# Assign the result on the right-hand side to `partition_level_mismatched_cnt_diff, partition_level_mismatched_record_df, partition_level_mismatched_record_detail_cnt, partition_level_mismatched_record_detail_df, partition_level_missing_cnt, partition_level_missing_output_df` so it can be reused later.
                                                partition_level_mismatched_cnt_diff, partition_level_mismatched_record_df, partition_level_mismatched_record_detail_cnt, partition_level_mismatched_record_detail_df, partition_level_missing_cnt, partition_level_missing_output_df = find_difference(trim_space(netezza_df_partition), trim_space(Data_Lake_df_1), primary_key)
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Successfully performed partition level validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Successfully performed partition level validation for {db}.{table_name}", style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                                partition_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                            except Exception as e:
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                                partition_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                                print(f'Failed to perform partiton level validation for {db}.{table_name}_(Netezza v. Gold)')
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'Failed to perform partiton level validation for {db}.{table_name}_(Netezza v. Gold)', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                        partition_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad field validation query configured for {db}.{table_name}. Please correct the query. No partition level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad field validation query configured for {db}.{table_name}. Please correct the query. No partition level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')



# Original notebook comment retained.
                            ## Define excel writer
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Gold).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                    ## Write missing and mismatched information to sheets in excel workbook
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `partition_level_missing_output_df.sort_values(by` so it can be reused later.
                                            partition_level_missing_output_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = f"missing_record_partition", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `partition_level_missing_output_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_level_missing_output_df.head(max_excel_row).to_excel(writer, sheet_name = f"missing_record_partition", startrow = 0, startcol=0, index = False)
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `partition_level_mismatched_record_df.sort_values(by` so it can be reused later.
                                            partition_level_mismatched_record_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = f"mismatched_record_partition", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_level_mismatched_record_detail_df.sort_values(by` so it can be reused later.
                                            partition_level_mismatched_record_detail_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = f"mismatched_detail_partition", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `partition_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = f"mismatched_record_partition", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_level_mismatched_record_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_level_mismatched_record_detail_df.head(max_excel_row).to_excel(writer, sheet_name = f"mismatched_detail_partition", startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if measure_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.sort_values(by` so it can be reused later.
                                            measure_level_mismatched_record_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `measure_level_mismatched_detail_df.sort_values(by` so it can be reused later.
                                            measure_level_mismatched_detail_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `measure_level_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            measure_level_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_detail", startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_count_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `partition_count_mismatched_df.sort_values(by` so it can be reused later.
                                            partition_count_mismatched_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_count_mismatched_detail_df.sort_values(by` so it can be reused later.
                                            partition_count_mismatched_detail_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `partition_count_mismatched_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_count_mismatched_df.head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_count_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_count_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_detail", startrow = 0, startcol=0, index = False)
# Print a message or value to the notebook output for validation or debugging.
                                print('Detailed report generated')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('Detailed report generated', style = 'List Bullet')
# Assign the result on the right-hand side to `detailed_report_status` so it can be reused later.
                                detailed_report_status = 'True'
# Handle an error raised in the preceding try block.
                            except Exception as e:
# Assign the result on the right-hand side to `detailed_report_status` so it can be reused later.
                                detailed_report_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                print('No detailed report generated')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('No detailed report generated', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'{e}', style= 'Intense Quote')


# Original notebook comment retained.
                            ## Upload workbook to storage
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if detailed_report_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                    workbook_path = path + f"{db}-{table_name}_(Netezza v. Gold).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                    shutil.copy2(f"{db}-{table_name}_(Netezza v. Gold).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                    os.remove(f"{db}-{table_name}_(Netezza v. Gold).xlsx")

# Original notebook comment retained.
                                    ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_table_name.append(table_name)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_count_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_count_mismatched_cnt.append(str(partition_count_mismatched_cnt))
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_count_mismatched_cnt.append('')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if measure_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_measure_level_mismatched_cnt_diff.append('')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_level_mismatched_cnt_diff.append(str(partition_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_level_missing_cnt.append(str(partition_level_missing_cnt))
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                    print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Gold)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Gold)", style='List Bullet')


# Handle an error raised in the preceding try block.
                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Gold)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Gold)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')



# Start a new indented code block for the statement above.
                        else :
# Original notebook comment retained.
                            ## This condition is to perform detailed validation for small and medium size tables
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Attempting to perform detailed validation on {db}.{table_name}. Need to confirm it is a small to medium size table')


# Original notebook comment retained.
                            ## Retrieve measure level query
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f" select distinct MEASURE_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Gold` so it can be reused later.
                            Query_Measure_Gold=spark.sql(f" select distinct ADLS_QUERY_FOR_MEASURE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and ADLS_QUERY_FOR_MEASURE is not null")

# Print a message or value to the notebook output for validation or debugging.
                            print("else Full load Non Transactional ")
# Original notebook comment retained.
                            ## Get FULL netezza table for validation
# Assign the result on the right-hand side to `Query_Netezza` so it can be reused later.
                            Query_Netezza = f"""select * from {db_tmp}.{table_name}"""
# Assign the result on the right-hand side to `netezza_df` so it can be reused later.
                            netezza_df=readfromNetizza(Env,Query_Netezza,db)



# Original notebook comment retained.
                            ## Confirm table is small or medium size (i.e. Not greater than 5 million rows)


# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:
# Original notebook comment retained.
                                ## Retrieve row count
# Execute this line as part of the notebook's workflow logic.
                                cnt_netezza_df = int(report_dataframe_1.select(f'NETEZZA_TABLE_ROW_COUNT ({netezza_sub})').filter((col('NETEZZA_SCHEMA') == f'{db}') & (col('NETEZZA_TABLE') == f'{table_name}')).collect()[0][0]) #netezza_df.limit(max_rows_permitted + 1).count()
# Execute this line as part of the notebook's workflow logic.
                                cnt_df_ADLS_gold = int(report_dataframe_1.select('GOLD_TABLE_ROW_COUNT').filter((col('SYNAPSE_SCHEMA') == f'{db}') & (col('SYNAPSE_TABLE') == f'{table_name}')).collect()[0][0])#df_synapse.limit(max_rows_permitted + 1).count()
# Handle an error raised in the preceding try block.
                            except:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'{db}.{table_name} not available in summary report')
# Assign the result on the right-hand side to `cnt_netezza_df` so it can be reused later.
                                cnt_netezza_df = netezza_df.limit(max_rows_permitted + 1).count()
# Assign the result on the right-hand side to `cnt_df_ADLS_gold` so it can be reused later.
                                cnt_df_ADLS_gold = Data_Lake_df.limit(max_rows_permitted + 1).count()



# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (cnt_netezza_df == 0) | (cnt_df_ADLS_gold == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Table {db}.{table_name} is empty either in Netezza or in Gold. No detailed validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Table {db}.{table_name} is empty either in Netezza or in Gold. No detailed validation performed', style = 'List Bullet')

# Check the next condition if the previous condition was not met.
                            elif (cnt_netezza_df > max_rows_permitted) | (cnt_df_ADLS_gold > max_rows_permitted):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Table {db}.{table_name} is a big table, with no partition column configures. No table level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Table {db}.{table_name} is a big table, with no partition column configures. No table level validation performed', style='List Bullet')

# Print a message or value to the notebook output for validation or debugging.
                                print('Will attempt to perform measure level validation')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('Attempting measure level validation for a big table with no partition configured', style= 'List Bullet')

# Original notebook comment retained.
                                ## PERFORM MEASURE LEVEL VALIDATION
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (Query_Measure_Netezza.count() == 0) | (Query_Measure_Gold.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                    measure_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                    query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_measure_gold` so it can be reused later.
                                    query_measure_gold = Query_Measure_Gold.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if (query_measure_netezza == '') | (query_measure_gold == ''):
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                        measure_validation_status = 'False' 
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Measure queries configured. Attempting to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Measure queries configured. Attempting to perform measure level validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                            query_measure_netezza = query_measure_netezza.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                            netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Assign the result on the right-hand side to `adls_gold_df_measure` so it can be reused later.
                                            adls_gold_df_measure = spark.sql(f"{query_measure_gold}")

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                            try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df` so it can be reused later.
                                                measure_level_mismatched_record_df = get_pandas(adls_gold_df_measure.subtract(netezza_df_measure))
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff` so it can be reused later.
                                                measure_level_mismatched_cnt_diff = len(measure_level_mismatched_record_df)
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                                measure_validation_status = 'True'


# Original notebook comment retained.
                                                ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                                with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Gold).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                                    ## Write missing and mismatched information to sheets in excel workbook
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                                    measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = f"measure_mismatched_record", startrow = 0, startcol=0, index = False)

# Original notebook comment retained.
                                                ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                                try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                                    workbook_path = path + f"{db}-{table_name}_(Netezza v. Gold).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                                    shutil.copy2(f"{db}-{table_name}_(Netezza v. Gold).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                                    os.remove(f"{db}-{table_name}_(Netezza v. Gold).xlsx")

# Original notebook comment retained.
                                                    ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                                    print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Gold)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                    doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Gold)", style='List Bullet')

# Handle an error raised in the preceding try block.
                                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                                        print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Gold)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                        doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Gold)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                        doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
                                            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Failed to perform detailed validation on {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Failed to perform detailed validation on {db}.{table_name}", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'{e}', style='Intense Quote')
# Execute this line as part of the notebook's workflow logic.
                                                continue

# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')


# Run the fallback branch when the earlier conditions do not match.
                            else:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Table size is less than or equal to {max_rows_permitted}. Performing detailed validation on full table {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Table size is less than or equal to {max_rows_permitted}. Performing detailed validation on full table {db}.{table_name}', style = 'List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:
# Original notebook comment retained.
                                    #table_level_mismatched_cnt_diff, table_level_mismatched_record_df, table_level_mismatched_record_detail_cnt, table_level_mismatched_record_detail_df, table_level_missing_cnt, table_level_missing_output_df = find_difference(netezza_df, Data_Lake_df, primary_key)
# Assign the result on the right-hand side to `table_level_mismatched_cnt_diff, table_level_mismatched_record_df, table_level_mismatched_record_detail_cnt, table_level_mismatched_record_detail_df, table_level_missing_cnt, table_level_missing_output_df` so it can be reused later.
                                    table_level_mismatched_cnt_diff, table_level_mismatched_record_df, table_level_mismatched_record_detail_cnt, table_level_mismatched_record_detail_df, table_level_missing_cnt, table_level_missing_output_df = find_difference(trim_space(netezza_df), trim_space(Data_Lake_df), primary_key)
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Successfully performed full table validation on {db}.{table_name}, comparing Netezza to Gold table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Successfully performed full table validation on {db}.{table_name}, comparing Netezza to Gold table', style='List Bullet')

# Original notebook comment retained.
                                    ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                    with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Gold).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                        ## Write missing and mismatched information to sheets in excel workbook
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `table_level_missing_output_df.sort_values(by` so it can be reused later.
                                            table_level_missing_output_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = "missing_record", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `table_level_missing_output_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            table_level_missing_output_df.head(max_excel_row).to_excel(writer, sheet_name = "missing_record", startrow = 0, startcol=0, index = False)
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `table_level_mismatched_record_df.sort_values(by` so it can be reused later.
                                            table_level_mismatched_record_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = "mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `table_level_mismatched_record_detail_df.sort_values(by` so it can be reused later.
                                            table_level_mismatched_record_detail_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = "mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `table_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            table_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = "mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `table_level_mismatched_record_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            table_level_mismatched_record_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "mismatched_detail", startrow = 0, startcol=0, index = False)


# Original notebook comment retained.
                                    ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                        workbook_path = path + f"{db}-{table_name}_(Netezza v. Gold).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                        shutil.copy2(f"{db}-{table_name}_(Netezza v. Gold).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                        os.remove(f"{db}-{table_name}_(Netezza v. Gold).xlsx")

# Original notebook comment retained.
                                        ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_measure_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_table_level_mismatched_cnt_diff.append(str(table_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_table_level_missing_cnt.append(str(table_level_missing_cnt))
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                        print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Gold)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Gold)", style='List Bullet')


# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                        print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Gold)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Gold)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')


# Handle an error raised in the preceding try block.
                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Gold table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Gold table', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')


# Handle an error raised in the preceding try block.
                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                    print(f'ERROR OCCURRED: Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Gold table')
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'ERROR OCCURRED: Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Gold table', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'{e}', style='Intense Quote')
# Original notebook comment retained.
                    #continue


# Original notebook comment retained.
        ## Upload log file
# Execute this line as part of the notebook's workflow logic.
        save_log(doc)

# Handle an error raised in the preceding try block.
    except Exception as e:
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph("Data Validation Run Failed", style="List Bullet")
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph('')
# Print a message or value to the notebook output for validation or debugging.
        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph(f'{e}', style='Intense Quote')

# Original notebook comment retained.
        ## Upload log file
# Execute this line as part of the notebook's workflow logic.
        save_log(doc)

# Original notebook comment retained.
        ## Exit the notebook run
# Execute this line as part of the notebook's workflow logic.
        dbutils.notebook.exit("Data Validation Run Failed") 







## Command cell 37

This section corresponds to command 37 from the original Databricks notebook.


In [ ]:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if (PL_TL != '7GS') & (Load_Type == 'Historical'): 

# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.functions import *.
    from pyspark.sql.functions import *
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.types import DecimalType, IntegerType, DateType, StructType, TimestampType.
    from pyspark.sql.types import DecimalType, IntegerType, DateType, StructType, TimestampType
# Import specific objects from a module so they can be used directly in later code: from datetime import date, timedelta, datetime.
    from datetime import date, timedelta, datetime
# Import specific objects from a module so they can be used directly in later code: from delta.tables import *.
    from delta.tables import *

# Original notebook comment retained.
    # Header for log document
# Add a heading to the Word document being used as a log or report.
    doc.add_heading('KEY TYPE UNIQUE = N',3)
# Assign the result on the right-hand side to `container` so it can be reused later.
    container = 'gold'

# Begin a protected block so the notebook can handle runtime errors more gracefully.
    try:

# Start a loop so the same logic is applied repeatedly across multiple items.
        for db in relevant_gold_db:

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
            if Env == 'QA':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = f"GDC_{db}"
# Check the next condition if the previous condition was not met.
            elif Env == 'PROD_DR':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = f"{db}_DR"
# Run the fallback branch when the earlier conditions do not match.
            else:
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = db
# Print a message or value to the notebook output for validation or debugging.
            print('DATABASE: ', db)


# Start a loop so the same logic is applied repeatedly across multiple items.
            for table_dict in [v for d in validation_tables_info_2 for k,v in d.items() if k == db][0]:
# Print a message or value to the notebook output for validation or debugging.
                print(table_dict)
# Assign the result on the right-hand side to `table_name` so it can be reused later.
                table_name=list(table_dict.keys())[0]
# Print a message or value to the notebook output for validation or debugging.
                print('TABLE NAME: ',table_name)    

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                try:    

# Assign the result on the right-hand side to `Partition_column` so it can be reused later.
                    Partition_column=spark.sql(f"select distinct PARTITION_COLUMN from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_COLUMN is not null")
# Assign the result on the right-hand side to `KEY_TYPE` so it can be reused later.
                    KEY_TYPE=spark.sql(f"select distinct KEY_TYPE_UNIQUE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and KEY_TYPE_UNIQUE is not null")
# Assign the result on the right-hand side to `KEY_TYPES` so it can be reused later.
                    KEY_TYPES=KEY_TYPE.collect()[0][0]
# Print a message or value to the notebook output for validation or debugging.
                    print('KEY_TYPES: ',KEY_TYPES)



# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                    if KEY_TYPES == "N" :
# Assign the result on the right-hand side to `tab_gd_cnt +` so it can be reused later.
                        tab_gd_cnt += 1
# Print a message or value to the notebook output for validation or debugging.
                        print(tab_gd_cnt)

# Add a heading to the Word document being used as a log or report.
                        doc.add_heading(f"{db}.{table_name}", 4)

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if Partition_column.count() == 0:
# Assign the result on the right-hand side to `Partition_col` so it can be reused later.
                            Partition_col = None
# Print a message or value to the notebook output for validation or debugging.
                            print(f'No partition column configured for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'No partition column configured for {db}.{table_name}', style='List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                        else:
# Assign the result on the right-hand side to `Partition_col` so it can be reused later.
                            Partition_col =Partition_column.collect()[0][0]
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Partition column configured for {db}.{table_name} is {Partition_col}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Partition column configured for {db}.{table_name} is {Partition_col}', style='List Bullet')

# Original notebook comment retained.
                        ## The ADLS gold table
# Read data from storage into a DataFrame or Python object.
                        Data_Lake_df=spark.read.format("delta").option("header","true").load("/mnt/gold/"+db+"/ADMIN/"+table_name)
# Execute this line as part of the notebook's workflow logic.
                        Data_Lake_df.createOrReplaceTempView("Data_Lake_df")

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if ((Partition_col != None) and (Partition_col != 'NA') ):
# Original notebook comment retained.
                            ## Retrieve dataframe of the queries for row count per partition for Netezza tables, and Retrieve dataframe of query for generating a measure from Netezza and ADLS_Gold
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Retrieving Partition, Measure and Field Validation Queries from Fwk_Gold_Tables for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Retrieving Partition, Measure and Field Validation Queries from Fwk_Gold_Tables for {db}.{table_name}', style='List Bullet')

# Assign the result on the right-hand side to `Query_Agg_Netezza` so it can be reused later.
                            Query_Agg_Netezza=spark.sql(f"select distinct PARTITION_QUERY_NETEZZA from gold_df where TABLE_NAME = '{table_name}' and DATABASE ='{db}' and PARTITION_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f" select distinct MEASURE_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Gold` so it can be reused later.
                            Query_Measure_Gold=spark.sql(f" select distinct ADLS_QUERY_FOR_MEASURE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and ADLS_QUERY_FOR_MEASURE is not null")

# Assign the result on the right-hand side to `Query_Field_Level_Netezza` so it can be reused later.
                            Query_Field_Level_Netezza=spark.sql(f"select distinct FIELD_VALIDATION_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and FIELD_VALIDATION_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Field_Level_Gold` so it can be reused later.
                            Query_Field_Level_Gold=spark.sql(f"select distinct ADLS_QUERY_FOR_FIELD from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and ADLS_QUERY_FOR_FIELD is not null")  


# Original notebook comment retained.
                            ## Retrieve dataframe of the field used for validation (YEAR, MONTH, DAY)
# Assign the result on the right-hand side to `PARTITION_ON` so it can be reused later.
                            PARTITION_ON=spark.sql(f"select PARTITION_ON from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_ON is not null")
# Assign the result on the right-hand side to `Partition_Type` so it can be reused later.
                            Partition_Type=PARTITION_ON.collect()[0][0].upper()
# Print a message or value to the notebook output for validation or debugging.
                            print('PARTITION TYPE:', Partition_Type)

# Original notebook comment retained.
                            ## Skip table if no queries are configured
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Field_Level_Netezza.count() == 0) & (Query_Measure_Netezza.count() == 0) & (Query_Agg_Netezza.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f"No field validation query or measure query or partition query configured. No detailed report generated for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f"No field validation query or measure query or partition query configured. No detailed report generated for {db}.{table_name}", style = 'List Bullet')
# Execute this line as part of the notebook's workflow logic.
                                continue


# Original notebook comment retained.
                            ## Get the various queries, partition field and partition value
# Execute this line as part of the notebook's workflow logic.
                            '''
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if Query_Agg_Netezza.count()!=0:
# Assign the result on the right-hand side to `query_table_load_netezza` so it can be reused later.
                                query_table_load_netezza =Query_Agg_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `netezza_df` so it can be reused later.
                                netezza_df=readfromNetizza(Env,query_table_load_netezza,db)
# Execute this line as part of the notebook's workflow logic.
                            '''

# Original notebook comment retained.
                            ## PERFORM PARTITION COUNT VALIDATION
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Agg_Netezza.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No partition queries configured for {db}.{table_name} in Fwk_Gold_Tables for Netezza. No partition count validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No partition queries configured for {db}.{table_name} in Fwk_Gold_Tables for Netezza. No partition count validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                partition_count_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                query_PARTITION_CNT_NZ = Query_Agg_Netezza.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_PARTITION_CNT_NZ == ''):
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                    partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                    print('No Partition Query is configured. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(('No Partition Query is configured. No partition count validation performed'), style= 'List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Attempting to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Attempting to perform partition count validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                        query_PARTITION_CNT_NZ = query_PARTITION_CNT_NZ.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_partition_cnt_df` so it can be reused later.
                                        netezza_partition_cnt_df = readfromNetizza(Env, query_PARTITION_CNT_NZ, db)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                        if Partition_Type == 'YEAR' :
# Assign the result on the right-hand side to `Data_Lake_df_PARTITION_CNT` so it can be reused later.
                                            Data_Lake_df_PARTITION_CNT=spark.sql(f"select DATE_FORMAT({Partition_col}, 'yyyy') as {Partition_col}, count(*) as CNT from Data_Lake_df group by DATE_FORMAT({Partition_col}, 'yyyy')")
# Check the next condition if the previous condition was not met.
                                        elif Partition_Type == 'MONTH' :
# Assign the result on the right-hand side to `Data_Lake_df_PARTITION_CNT` so it can be reused later.
                                            Data_Lake_df_PARTITION_CNT=spark.sql(f"select DATE_FORMAT({Partition_col}, 'yyyy MMM') as {Partition_col}, count(*) as CNT from Data_Lake_df group by DATE_FORMAT({Partition_col}, 'yyyy MMM')")
# Check the next condition if the previous condition was not met.
                                        elif Partition_Type == 'DAY' :
# Assign the result on the right-hand side to `Data_Lake_df_PARTITION_CNT` so it can be reused later.
                                            Data_Lake_df_PARTITION_CNT=spark.sql(f"select DATE_FORMAT({Partition_col}, 'yyyy MMM dd') as {Partition_col}, count(*) as CNT from Data_Lake_df group by DATE_FORMAT({Partition_col}, 'yyyy MMM dd')")

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df` so it can be reused later.
                                            partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df = find_Mismatched(netezza_partition_cnt_df,Data_Lake_df_PARTITION_CNT,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed partition count validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed partition count validation for {db}.{table_name}", style = 'List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                            partition_count_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                            partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Failed to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Failed to perform partition count validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                        partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad partition query configured for {db}.{table_name}. Please correct the query. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad partition query configured for {db}.{table_name}. Please correct the query. No partition count validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')


# Original notebook comment retained.
                            ## PERFORM MEASURE LEVEL VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Measure_Netezza.count() == 0) | (Query_Measure_Gold.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                measure_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_measure_gold` so it can be reused later.
                                query_measure_gold = Query_Measure_Gold.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_measure_netezza == '') | (query_measure_gold == ''):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                    measure_validation_status = 'False' 
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Attempting to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Attempting to perform measure level validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                        query_measure_netezza = query_measure_netezza.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                        netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Assign the result on the right-hand side to `adls_gold_df_measure` so it can be reused later.
                                        adls_gold_df_measure = spark.sql(f"{query_measure_gold}")

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff, measure_level_mismatched_record_df, measure_level_mismatched_detail_cnt, measure_level_mismatched_detail_df` so it can be reused later.
                                            measure_level_mismatched_cnt_diff, measure_level_mismatched_record_df, measure_level_mismatched_detail_cnt, measure_level_mismatched_detail_df = find_Mismatched(netezza_df_measure,adls_gold_df_measure,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed measure level validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed measure level validation for {db}.{table_name}", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                            measure_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                            measure_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Failed to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Failed to perform measure level validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                        measure_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')



# Original notebook comment retained.
                            ## PERFORM PARTITION LEVEL VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Field_Level_Netezza.count()==0) | (Query_Field_Level_Gold.count()==0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                partition_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_PARTITION_NZ` so it can be reused later.
                                query_PARTITION_NZ = Query_Field_Level_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_PARTITION_Gold` so it can be reused later.
                                query_PARTITION_Gold = Query_Field_Level_Gold.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_PARTITION_NZ == '') | (query_PARTITION_Gold == ''):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                    partition_validation_status = 'False'  
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_PARTITION_NZ` so it can be reused later.
                                        query_PARTITION_NZ = query_PARTITION_NZ.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_partition` so it can be reused later.
                                        netezza_df_partition=readfromNetizza(Env,query_PARTITION_NZ,db)
# Assign the result on the right-hand side to `Data_Lake_df_1` so it can be reused later.
                                        Data_Lake_df_1=spark.sql(query_PARTITION_Gold)  

# Original notebook comment retained.
                                        ## Validate that the partition size is within the max row permitted limit
# Print a message or value to the notebook output for validation or debugging.
                                        print('Attempting to perform partition level validation. Need to confirm that the partition size is within acceptable limit')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph('Attempting to perform partition level validation. Need to confirm that the partition size is within acceptable limit', style = 'List Bullet')

# Assign the result on the right-hand side to `cnt_netezza_df_partition` so it can be reused later.
                                        cnt_netezza_df_partition = netezza_df_partition.limit(max_rows_permitted + 1).count()
# Assign the result on the right-hand side to `cnt_Data_Lake_df_1` so it can be reused later.
                                        cnt_Data_Lake_df_1 = Data_Lake_df_1.limit(max_rows_permitted + 1).count()

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                        if (cnt_netezza_df_partition == 0) | (cnt_Data_Lake_df_1 == 0):
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Partition for table {db}.{table_name} is empty either in Netezza or in Gold. No partition validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Partition for table {db}.{table_name} is empty either in Netezza or in Gold. No partition validation performed', style = 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                            partition_validation_status = 'False'

# Check the next condition if the previous condition was not met.
                                        elif (cnt_netezza_df_partition > max_rows_permitted) | (cnt_Data_Lake_df_1 > max_rows_permitted):
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Partition for {db}.{table_name} is greater than {max_rows_permitted}. No partition validation performed. Please reconfigure the partition for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Partition for {db}.{table_name} is greater than {max_rows_permitted}. No partition validation performed. Please reconfigure the partition for {db}.{table_name}', style = 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                            partition_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                                        else:
# Original notebook comment retained.
                                            ## Perform validation on the partition
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Partition for {db}.{table_name} is less than or equal to {max_rows_permitted}.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Partition for {db}.{table_name} is less than or equal to {max_rows_permitted}.', style = 'List Bullet')
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                            try:
# Original notebook comment retained.
                                                #partition_level_diff = get_pandas(netezza_df_partition.subtract(Data_Lake_df_1))
# Assign the result on the right-hand side to `partition_level_diff` so it can be reused later.
                                                partition_level_diff = get_pandas(trim_space(netezza_df_partition).subtract(trim_space(Data_Lake_df_1)))
# Assign the result on the right-hand side to `partition_level_diff_cnt` so it can be reused later.
                                                partition_level_diff_cnt = len(partition_level_diff)
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Successfully performed partition level validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Successfully performed partition level validation for {db}.{table_name}", style= 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                                partition_validation_status = 'True'

# Handle an error raised in the preceding try block.
                                            except Exception as e:
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                                partition_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                                print(f'Failed to perform partiton level validation for {db}.{table_name}_(Netezza v. Gold)')
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'Failed to perform partiton level validation for {db}.{table_name}_(Netezza v. Gold)', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                        partition_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad field validation query configured for {db}.{table_name}. Please correct the query. No partition level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad field validation query configured for {db}.{table_name}. Please correct the query. No partition level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')


# Original notebook comment retained.
                            ## Define excel writer
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Gold).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                    ## Write missing and mismatched information to sheets in excel workbook
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_validation_status == 'True':
# Assign the result on the right-hand side to `partition_level_diff.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        partition_level_diff.head(max_excel_row).to_excel(writer, sheet_name = f"difference_record_partition", startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if measure_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.sort_values(by` so it can be reused later.
                                            measure_level_mismatched_record_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `measure_level_mismatched_detail_df.sort_values(by` so it can be reused later.
                                            measure_level_mismatched_detail_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `measure_level_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            measure_level_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_detail", startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_count_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `partition_count_mismatched_df.sort_values(by` so it can be reused later.
                                            partition_count_mismatched_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_count_mismatched_detail_df.sort_values(by` so it can be reused later.
                                            partition_count_mismatched_detail_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `partition_count_mismatched_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_count_mismatched_df.head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_count_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_count_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_detail", startrow = 0, startcol=0, index = False)
# Print a message or value to the notebook output for validation or debugging.
                                print('Detailed report generated')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('Detailed report generated', style = 'List Bullet')
# Assign the result on the right-hand side to `detailed_report_status` so it can be reused later.
                                detailed_report_status = 'True'
# Handle an error raised in the preceding try block.
                            except Exception as e:
# Assign the result on the right-hand side to `detailed_report_status` so it can be reused later.
                                detailed_report_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                print('No detailed report generated')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('No detailed report generated', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'{e}', style= 'Intense Quote')


# Original notebook comment retained.
                            ## Upload workbook to storage
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if detailed_report_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                    workbook_path = path + f"{db}-{table_name}_(Netezza v. Gold).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                    shutil.copy2(f"{db}-{table_name}_(Netezza v. Gold).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                    os.remove(f"{db}-{table_name}_(Netezza v. Gold).xlsx")

# Original notebook comment retained.
                                    ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_table_name.append(table_name)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_count_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_count_mismatched_cnt.append(str(partition_count_mismatched_cnt))
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_count_mismatched_cnt.append('')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if measure_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_measure_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_table_level_missing_cnt.append('')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_level_diff_cnt.append(str(partition_level_diff_cnt))
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_gold_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                    print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Gold)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Gold)", style='List Bullet')


# Handle an error raised in the preceding try block.
                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Gold)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Gold)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')


# Start a new indented code block for the statement above.
                        else :
# Original notebook comment retained.
                            ## This condition is to perform detailed validation for small and medium size tables
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Attempting to perform detailed validation on {db}.{table_name}. Need to confirm it is a small to medium size table')



# Original notebook comment retained.
                            ## Retrieve measure level query
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f" select distinct MEASURE_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_NETEZZA is not null")
# Original notebook comment retained.
                            ##TODO- Include Query_Measure_Gold if it exists. Confirm the Query_Measure_Gold
# Assign the result on the right-hand side to `Query_Measure_Gold` so it can be reused later.
                            Query_Measure_Gold=spark.sql(f" select distinct ADLS_QUERY_FOR_MEASURE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and ADLS_QUERY_FOR_MEASURE is not null")

# Print a message or value to the notebook output for validation or debugging.
                            print("else Full load Non Transactional ")
# Original notebook comment retained.
                            ## Get FULL netezza table for validation
# Assign the result on the right-hand side to `Query_Netezza` so it can be reused later.
                            Query_Netezza = f"""select * from {db_tmp}.{table_name}"""
# Assign the result on the right-hand side to `netezza_df` so it can be reused later.
                            netezza_df=readfromNetizza(Env,Query_Netezza,db)



# Original notebook comment retained.
                            ## Confirm table is small or medium size (i.e. Not greater than 5 million rows)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:
# Original notebook comment retained.
                                ## Retrieve row count
# Execute this line as part of the notebook's workflow logic.
                                cnt_netezza_df = int(report_dataframe_1.select(f'NETEZZA_TABLE_ROW_COUNT ({netezza_sub})').filter((col('NETEZZA_SCHEMA') == f'{db}') & (col('NETEZZA_TABLE') == f'{table_name}')).collect()[0][0]) #netezza_df.limit(max_rows_permitted + 1).count()
# Execute this line as part of the notebook's workflow logic.
                                cnt_df_ADLS_gold = int(report_dataframe_1.select('GOLD_TABLE_ROW_COUNT').filter((col('SYNAPSE_SCHEMA') == f'{db}') & (col('SYNAPSE_TABLE') == f'{table_name}')).collect()[0][0])#df_synapse.limit(max_rows_permitted + 1).count()
# Handle an error raised in the preceding try block.
                            except:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'{db}.{table_name} not available in summary report')
# Assign the result on the right-hand side to `cnt_netezza_df` so it can be reused later.
                                cnt_netezza_df = netezza_df.limit(max_rows_permitted + 1).count()
# Assign the result on the right-hand side to `cnt_df_ADLS_gold` so it can be reused later.
                                cnt_df_ADLS_gold = Data_Lake_df.limit(max_rows_permitted + 1).count()

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (cnt_netezza_df == 0) | (cnt_df_ADLS_gold == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Table {db}.{table_name} is empty either in Netezza or in Gold. No detailed validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Table {db}.{table_name} is empty either in Netezza or in Gold. No detailed validation performed', style = 'List Bullet')

# Check the next condition if the previous condition was not met.
                            elif (cnt_netezza_df > max_rows_permitted) | (cnt_df_ADLS_gold > max_rows_permitted):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Table {db}.{table_name} is a big table, with no partition column configured. No table level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Table {db}.{table_name} is a big table, with no partition column configured. No table_level vallidation performed', style='List Bullet')

# Print a message or value to the notebook output for validation or debugging.
                                print('Will attempt to perform measure level validation')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('Attempting measure level validation for a big table with no partition configured', style= 'List Bullet')

# Original notebook comment retained.
                                ## PERFORM MEASURE LEVEL VALIDATION
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (Query_Measure_Netezza.count() == 0) | (Query_Measure_Gold.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                    measure_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                    query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_measure_gold` so it can be reused later.
                                    query_measure_gold = Query_Measure_Gold.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if (query_measure_netezza == '') | (query_measure_gold == ''):
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                        measure_validation_status = 'False' 
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Measure queries configured. Attempting to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Measure queries configured. Attempting to perform measure level validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                            query_measure_netezza = query_measure_netezza.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                            netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Assign the result on the right-hand side to `adls_gold_df_measure` so it can be reused later.
                                            adls_gold_df_measure = spark.sql(f"{query_measure_gold}")

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                            try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df` so it can be reused later.
                                                measure_level_mismatched_record_df = get_pandas(adls_gold_df_measure.subtract(netezza_df_measure))
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff` so it can be reused later.
                                                measure_level_mismatched_cnt_diff = len(measure_level_mismatched_record_df)
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                                measure_validation_status = 'True'

# Original notebook comment retained.
                                                ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                                with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Gold).xlsx", engine="openpyxl", mode = 'w') as writer:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                                    measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = f"measure_mismatched_record", startrow = 0, startcol=0, index = False)

# Original notebook comment retained.
                                                ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                                try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                                    workbook_path = path + f"{db}-{table_name}_(Netezza v. Gold).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                                    shutil.copy2(f"{db}-{table_name}_(Netezza v. Gold).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                                    os.remove(f"{db}-{table_name}_(Netezza v. Gold).xlsx")

# Original notebook comment retained.
                                                    ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_gold_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                                    print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Gold)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                    doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Gold)", style='List Bullet')

# Handle an error raised in the preceding try block.
                                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                                        print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Gold)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                        doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Gold)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                        doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
                                            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Failed to perform detailed validation on {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Failed to perform detailed validation on {db}.{table_name}", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'{e}', style='Intense Quote')
# Execute this line as part of the notebook's workflow logic.
                                                continue

# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')


# Run the fallback branch when the earlier conditions do not match.
                            else:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Table size is less than or equal to {max_rows_permitted}. Performing detailed validation on full table {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Table size is less than or equal to {max_rows_permitted}. Performing detailed validation on full table {db}.{table_name}', style = 'List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:
# Original notebook comment retained.
                                    #table_level_diff = get_pandas(netezza_df.subtract(Data_Lake_df))
# Assign the result on the right-hand side to `table_level_diff` so it can be reused later.
                                    table_level_diff = get_pandas(trim_space(netezza_df).subtract(trim_space(Data_Lake_df)))
# Assign the result on the right-hand side to `table_level_diff_cnt` so it can be reused later.
                                    table_level_diff_cnt = len(table_level_diff)
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Successfully performed full table validation on {db}.{table_name}, comparing Netezza to Gold table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Successfully performed full table validation on {db}.{table_name}, comparing Netezza to Gold table', style='List Bullet')

# Original notebook comment retained.
                                    ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                    with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Gold).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                        ## Write missing and mismatched information to sheets in excel workbook
# Assign the result on the right-hand side to `table_level_diff.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        table_level_diff.head(max_excel_row).to_excel(writer, sheet_name = "difference_record", startrow = 0, startcol=0, index = False)



# Original notebook comment retained.
                                    ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                        workbook_path = path + f"{db}-{table_name}_(Netezza v. Gold).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                        shutil.copy2(f"{db}-{table_name}_(Netezza v. Gold).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                        os.remove(f"{db}-{table_name}_(Netezza v. Gold).xlsx")

# Original notebook comment retained.
                                        ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_measure_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_table_level_diff_cnt.append(table_level_diff_cnt)
# Execute this line as part of the notebook's workflow logic.
                                        netezza_gold_detailed_report_path.append(workbook_path)  

# Print a message or value to the notebook output for validation or debugging.
                                        print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Gold)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Gold)", style='List Bullet')


# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                        print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Gold)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Gold)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Gold table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Gold table', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                    print(f'ERROR OCCURRED: Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Gold table')
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'ERROR OCCURRED: Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Gold table', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'{e}', style='Intense Quote')
# Original notebook comment retained.
                    #continue                   

# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph('')

# Original notebook comment retained.
        ## Upload log file
# Execute this line as part of the notebook's workflow logic.
        save_log(doc)

# Handle an error raised in the preceding try block.
    except Exception as e:
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph("Data Validation Run Failed", style="List Bullet")
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph('')
# Print a message or value to the notebook output for validation or debugging.
        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph(f'{e}', style='Intense Quote')

# Original notebook comment retained.
        ## Upload log file
# Execute this line as part of the notebook's workflow logic.
        save_log(doc)

# Original notebook comment retained.
        ## Exit the notebook run
# Execute this line as part of the notebook's workflow logic.
        dbutils.notebook.exit("Data Validation Run Failed") 





## Command cell 38

This section corresponds to command 38 from the original Databricks notebook.


In [ ]:
# Assign the result on the right-hand side to `netezza_silver_database_name` so it can be reused later.
netezza_silver_database_name = []
# Assign the result on the right-hand side to `netezza_silver_table_name` so it can be reused later.
netezza_silver_table_name = []
# Assign the result on the right-hand side to `netezza_silver_partition_count_mismatched_cnt` so it can be reused later.
netezza_silver_partition_count_mismatched_cnt = []
# Assign the result on the right-hand side to `netezza_silver_measure_level_mismatched_cnt_diff` so it can be reused later.
netezza_silver_measure_level_mismatched_cnt_diff = []
# Assign the result on the right-hand side to `netezza_silver_partition_level_mismatched_cnt_diff` so it can be reused later.
netezza_silver_partition_level_mismatched_cnt_diff = []
# Assign the result on the right-hand side to `netezza_silver_partition_level_missing_cnt` so it can be reused later.
netezza_silver_partition_level_missing_cnt = []
# Assign the result on the right-hand side to `netezza_silver_table_level_mismatched_cnt_diff` so it can be reused later.
netezza_silver_table_level_mismatched_cnt_diff = []
# Assign the result on the right-hand side to `netezza_silver_table_level_missing_cnt` so it can be reused later.
netezza_silver_table_level_missing_cnt = []
# Assign the result on the right-hand side to `netezza_silver_measure_level_diff_cnt` so it can be reused later.
netezza_silver_measure_level_diff_cnt = []
# Assign the result on the right-hand side to `netezza_silver_partition_level_diff_cnt` so it can be reused later.
netezza_silver_partition_level_diff_cnt = []
# Assign the result on the right-hand side to `netezza_silver_table_level_diff_cnt` so it can be reused later.
netezza_silver_table_level_diff_cnt = []
# Assign the result on the right-hand side to `netezza_silver_detailed_report_path` so it can be reused later.
netezza_silver_detailed_report_path = []

## Command cell 39

This section corresponds to command 39 from the original Databricks notebook.


In [ ]:
# Assign the result on the right-hand side to `tab_si_cnt` so it can be reused later.
tab_si_cnt = 0

## Command cell 40

This section corresponds to command 40 from the original Databricks notebook.


In [ ]:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if (PL_TL != '7GS') & (Load_Type == 'Historical'):     

# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.functions import *.
    from pyspark.sql.functions import *
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.types import DecimalType, IntegerType, DateType, StructType, TimestampType.
    from pyspark.sql.types import DecimalType, IntegerType, DateType, StructType, TimestampType
# Import specific objects from a module so they can be used directly in later code: from datetime import date, timedelta, datetime.
    from datetime import date, timedelta, datetime
# Import specific objects from a module so they can be used directly in later code: from delta.tables import *.
    from delta.tables import *

# Original notebook comment retained.
    # Header for log document
# Add a heading to the Word document being used as a log or report.
    doc.add_heading('Log From Detailed Report Generation Comparing Netezza v. Silver', 2)
# Add a heading to the Word document being used as a log or report.
    doc.add_heading('KEY TYPE UNIQUE = Y',3)
# Assign the result on the right-hand side to `container` so it can be reused later.
    container = 'silver'

# Begin a protected block so the notebook can handle runtime errors more gracefully.
    try:

# Start a loop so the same logic is applied repeatedly across multiple items.
        for db in relevant_silver_db:

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
            if Env == 'QA':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = f"GDC_{db}"
# Check the next condition if the previous condition was not met.
            elif Env == 'PROD_DR':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = f"{db}_DR"
# Run the fallback branch when the earlier conditions do not match.
            else:
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = db
# Print a message or value to the notebook output for validation or debugging.
            print('DATABASE: ', db)


# Start a loop so the same logic is applied repeatedly across multiple items.
            for table_dict in [v for d in validation_tables_info_2 for k,v in d.items() if k == db][0]:
# Print a message or value to the notebook output for validation or debugging.
                print(table_dict)
# Assign the result on the right-hand side to `table_name` so it can be reused later.
                table_name=list(table_dict.keys())[0]
# Print a message or value to the notebook output for validation or debugging.
                print('TABLE NAME: ',table_name)    

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                try:     
# Assign the result on the right-hand side to `partition_key` so it can be reused later.
                    partition_key=list(table_dict.values())[0][0]
# Print a message or value to the notebook output for validation or debugging.
                    print(type(partition_key))
# Assign the result on the right-hand side to `primary_key` so it can be reused later.
                    primary_key=list(table_dict.values())[0]
# Print a message or value to the notebook output for validation or debugging.
                    print('PRIMARY KEY: ',primary_key)
# Assign the result on the right-hand side to `Partition_column` so it can be reused later.
                    Partition_column=spark.sql(f"select distinct PARTITION_COLUMN from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_COLUMN is not null")
# Assign the result on the right-hand side to `KEY_TYPE` so it can be reused later.
                    KEY_TYPE=spark.sql(f"select distinct KEY_TYPE_UNIQUE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and KEY_TYPE_UNIQUE is not null")
# Assign the result on the right-hand side to `KEY_TYPES` so it can be reused later.
                    KEY_TYPES=KEY_TYPE.collect()[0][0]
# Print a message or value to the notebook output for validation or debugging.
                    print('KEY_TYPES: ',KEY_TYPES)



# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                    if KEY_TYPES == "Y" :
# Assign the result on the right-hand side to `tab_si_cnt +` so it can be reused later.
                        tab_si_cnt += 1
# Print a message or value to the notebook output for validation or debugging.
                        print(tab_si_cnt)

# Add a heading to the Word document being used as a log or report.
                        doc.add_heading(f"{db}.{table_name}", 4)

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if Partition_column.count() == 0:
# Assign the result on the right-hand side to `Partition_col` so it can be reused later.
                            Partition_col = None
# Print a message or value to the notebook output for validation or debugging.
                            print(f'No partition column configured for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'No partition column configured for {db}.{table_name}', style='List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                        else:
# Assign the result on the right-hand side to `Partition_col` so it can be reused later.
                            Partition_col =Partition_column.collect()[0][0]
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Partition column configured for {db}.{table_name} is {Partition_col}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Partition column configured for {db}.{table_name} is {Partition_col}', style='List Bullet')

# Original notebook comment retained.
                        ## The ADLS silver table
# Read data from storage into a DataFrame or Python object.
                        Data_Lake_df=spark.read.format("delta").option("header","true").load("/mnt/silver/"+db+"/ADMIN/"+table_name)
# Execute this line as part of the notebook's workflow logic.
                        Data_Lake_df.createOrReplaceTempView("Data_Lake_df")

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if ((Partition_col != None) and (Partition_col != 'NA') ):
# Original notebook comment retained.
                            ## Retrieve dataframe of the queries for row count per partition for Netezza tables, and Retrieve dataframe of query for generating a measure from Netezza and ADLS_Gold
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Retrieving Partition, Measure and Field Validation Queries from Fwk_Gold_Tables for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Retrieving Partition, Measure and Field Validation Queries from Fwk_Gold_Tables for {db}.{table_name}', style='List Bullet')

# Assign the result on the right-hand side to `Query_Agg_Netezza` so it can be reused later.
                            Query_Agg_Netezza=spark.sql(f"select distinct PARTITION_QUERY_NETEZZA from gold_df where TABLE_NAME = '{table_name}' and DATABASE ='{db}' and PARTITION_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f" select distinct MEASURE_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Silver` so it can be reused later.
                            Query_Measure_Silver=spark.sql(f" select distinct ADLS_QUERY_FOR_MEASURE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and ADLS_QUERY_FOR_MEASURE is not null")

# Assign the result on the right-hand side to `Query_Field_Level_Netezza` so it can be reused later.
                            Query_Field_Level_Netezza=spark.sql(f"select distinct FIELD_VALIDATION_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and FIELD_VALIDATION_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Field_Level_Silver` so it can be reused later.
                            Query_Field_Level_Silver=spark.sql(f"select distinct ADLS_QUERY_FOR_FIELD from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and ADLS_QUERY_FOR_FIELD is not null")  


# Original notebook comment retained.
                            ## Retrieve dataframe of the field used for validation (YEAR, MONTH, DAY)
# Assign the result on the right-hand side to `PARTITION_ON` so it can be reused later.
                            PARTITION_ON=spark.sql(f"select PARTITION_ON from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_ON is not null")
# Assign the result on the right-hand side to `Partition_Type` so it can be reused later.
                            Partition_Type=PARTITION_ON.collect()[0][0].upper()
# Print a message or value to the notebook output for validation or debugging.
                            print('PARTITION TYPE:', Partition_Type)

# Original notebook comment retained.
                            ## Skip table if no queries are configured
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Field_Level_Netezza.count() == 0) & (Query_Measure_Netezza.count() == 0) & (Query_Agg_Netezza.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f"No field validation query or measure query or partition query configured. No detailed report generated for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f"No field validation query or measure query or partition query configured. No detailed report generated for {db}.{table_name}", style = 'List Bullet')
# Execute this line as part of the notebook's workflow logic.
                                continue


# Original notebook comment retained.
                            ## Get the various queries, partition field and partition value
# Execute this line as part of the notebook's workflow logic.
                            '''
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if Query_Agg_Netezza.count()!=0:
# Assign the result on the right-hand side to `query_table_load_netezza` so it can be reused later.
                                query_table_load_netezza =Query_Agg_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `netezza_partition_cnt_df` so it can be reused later.
                                netezza_partition_cnt_df=readfromNetizza(Env,query_table_load_netezza,db)
# Execute this line as part of the notebook's workflow logic.
                            ''' 


# Original notebook comment retained.
                            ## PERFORM PARTITION COUNT VALIDATION
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Agg_Netezza.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No partition queries configured for {db}.{table_name} in Fwk_Gold_Tables for Netezza. No partition count validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No partition queries configured for {db}.{table_name} in Fwk_Gold_Tables for Netezza. No partition count validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                partition_count_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                query_PARTITION_CNT_NZ = Query_Agg_Netezza.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_PARTITION_CNT_NZ == ''):
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                    partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                    print('No Partition Query is configured. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(('No Partition Query is configured. No partition count validation performed'), style= 'List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Attempting to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Attempting to perform partition count validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                        query_PARTITION_CNT_NZ = query_PARTITION_CNT_NZ.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_partition_cnt_df` so it can be reused later.
                                        netezza_partition_cnt_df = readfromNetizza(Env, query_PARTITION_CNT_NZ, db)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                        if Partition_Type == 'YEAR' :
# Assign the result on the right-hand side to `Data_Lake_df_PARTITION_CNT` so it can be reused later.
                                            Data_Lake_df_PARTITION_CNT=spark.sql(f"select DATE_FORMAT({Partition_col}, 'yyyy') as {Partition_col}, count(*) as CNT from Data_Lake_df group by DATE_FORMAT({Partition_col}, 'yyyy')")
# Check the next condition if the previous condition was not met.
                                        elif Partition_Type == 'MONTH' :
# Assign the result on the right-hand side to `Data_Lake_df_PARTITION_CNT` so it can be reused later.
                                            Data_Lake_df_PARTITION_CNT=spark.sql(f"select DATE_FORMAT({Partition_col}, 'yyyy MMM') as {Partition_col}, count(*) as CNT from Data_Lake_df group by DATE_FORMAT({Partition_col}, 'yyyy MMM')")
# Check the next condition if the previous condition was not met.
                                        elif Partition_Type == 'DAY' :
# Assign the result on the right-hand side to `Data_Lake_df_PARTITION_CNT` so it can be reused later.
                                            Data_Lake_df_PARTITION_CNT=spark.sql(f"select DATE_FORMAT({Partition_col}, 'yyyy MMM dd') as {Partition_col}, count(*) as CNT from Data_Lake_df group by DATE_FORMAT({Partition_col}, 'yyyy MMM dd')")

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df` so it can be reused later.
                                            partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df = find_Mismatched(netezza_partition_cnt_df,Data_Lake_df_PARTITION_CNT,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed partition count validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed partition count validation for {db}.{table_name}", style = 'List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                            partition_count_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                            partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Failed to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Failed to perform partition count validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                        partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad partition query configured for {db}.{table_name}. Please correct the query. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad partition query configured for {db}.{table_name}. Please correct the query. No partition count validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')



# Original notebook comment retained.
                            ## PERFORM MEASURE LEVEL VALIDATION
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Measure_Netezza.count() == 0) | (Query_Measure_Silver.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                measure_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_measure_silver` so it can be reused later.
                                query_measure_silver = Query_Measure_Silver.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_measure_netezza == '') | (query_measure_silver == ''):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                    measure_validation_status = 'False' 
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Attempting to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Attempting to perform measure level validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                        query_measure_netezza = query_measure_netezza.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                        netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Assign the result on the right-hand side to `adls_silver_df_measure` so it can be reused later.
                                        adls_silver_df_measure = spark.sql(f"{query_measure_silver}")

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff, measure_level_mismatched_record_df, measure_level_mismatched_detail_cnt, measure_level_mismatched_detail_df` so it can be reused later.
                                            measure_level_mismatched_cnt_diff, measure_level_mismatched_record_df, measure_level_mismatched_detail_cnt, measure_level_mismatched_detail_df = find_Mismatched(netezza_df_measure,adls_silver_df_measure,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed measure level validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed measure level validation for {db}.{table_name}", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                            measure_validation_status = 'True'

# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                            measure_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Failed to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Failed to perform measure level validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                        measure_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')


# Original notebook comment retained.
                            ## PERFORM PARTITION LEVEL VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Field_Level_Netezza.count()==0) | (Query_Field_Level_Silver.count()==0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                partition_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_PARTITION_NZ` so it can be reused later.
                                query_PARTITION_NZ = Query_Field_Level_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_PARTITION_Silver` so it can be reused later.
                                query_PARTITION_Silver = Query_Field_Level_Silver.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_PARTITION_NZ == '') | (query_PARTITION_Silver == ''):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                    partition_validation_status = 'False'  
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_PARTITION_NZ` so it can be reused later.
                                        query_PARTITION_NZ = query_PARTITION_NZ.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_partition` so it can be reused later.
                                        netezza_df_partition=readfromNetizza(Env,query_PARTITION_NZ,db)
# Assign the result on the right-hand side to `Data_Lake_df_1` so it can be reused later.
                                        Data_Lake_df_1=spark.sql(query_PARTITION_Silver)  

# Original notebook comment retained.
                                        ## Validate that the partition size is within the max row permitted limit
# Print a message or value to the notebook output for validation or debugging.
                                        print('Attempting to perform partition level validation. Need to confirm that the partition size is within acceptable limit')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph('Attempting to perform partition level validation. Need to confirm that the partition size is within acceptable limit', style = 'List Bullet')

# Assign the result on the right-hand side to `cnt_netezza_df_partition` so it can be reused later.
                                        cnt_netezza_df_partition = netezza_df_partition.limit(max_rows_permitted + 1).count()
# Assign the result on the right-hand side to `cnt_Data_Lake_df_1` so it can be reused later.
                                        cnt_Data_Lake_df_1 = Data_Lake_df_1.limit(max_rows_permitted + 1).count()

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                        if (cnt_netezza_df_partition == 0) | (cnt_Data_Lake_df_1 == 0):
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Partition for table {db}.{table_name} is empty either in Netezza or in Gold. No partition validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Partition for table {db}.{table_name} is empty either in Netezza or in Gold. No partition validation performed', style = 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                            partition_validation_status = 'False'

# Check the next condition if the previous condition was not met.
                                        elif (cnt_netezza_df_partition > max_rows_permitted) | (cnt_Data_Lake_df_1 > max_rows_permitted):
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Partition for {db}.{table_name} is greater than {max_rows_permitted}. No partition validation performed. Please reconfigure the partition for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Partition for {db}.{table_name} is greater than {max_rows_permitted}. No partition validation performed. Please reconfigure the partition for {db}.{table_name}', style = 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                            partition_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                                        else:
# Original notebook comment retained.
                                            ## Perform validation on the partition
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Partition for {db}.{table_name} is less than or equal to {max_rows_permitted}.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Partition for {db}.{table_name} is less than or equal to {max_rows_permitted}.', style = 'List Bullet')
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                            try:
# Original notebook comment retained.
                                                #partition_level_mismatched_cnt_diff, partition_level_mismatched_record_df, partition_level_mismatched_record_detail_cnt, partition_level_mismatched_record_detail_df, partition_level_missing_cnt, partition_level_missing_output_df = find_difference(netezza_df_partition, Data_Lake_df_1, primary_key)
# Assign the result on the right-hand side to `partition_level_mismatched_cnt_diff, partition_level_mismatched_record_df, partition_level_mismatched_record_detail_cnt, partition_level_mismatched_record_detail_df, partition_level_missing_cnt, partition_level_missing_output_df` so it can be reused later.
                                                partition_level_mismatched_cnt_diff, partition_level_mismatched_record_df, partition_level_mismatched_record_detail_cnt, partition_level_mismatched_record_detail_df, partition_level_missing_cnt, partition_level_missing_output_df = find_difference(trim_space(netezza_df_partition), trim_space(Data_Lake_df_1), primary_key)
# Execute this line as part of the notebook's workflow logic.
                                                partition_level_mismatched_record_df.spark.cache()
# Execute this line as part of the notebook's workflow logic.
                                                partition_level_mismatched_record_detail_df.spark.cache()
# Execute this line as part of the notebook's workflow logic.
                                                partition_level_missing_output_df.spark.cache()


# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Successfully performed partition level validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Successfully performed partition level validation for {db}.{table_name}", style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                                partition_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                            except Exception as e:
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                                partition_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                                print(f'Failed to generate detailed report for {db}.{table_name}_(Netezza v. Silver)')
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'Failed to generate detailed report for {db}.{table_name}_(Netezza v. Silver)', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'{e}', style='Intense Quote') 
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                        partition_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad field validation query configured for {db}.{table_name}. Please correct the query. No partition level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad field validation query configured for {db}.{table_name}. Please correct the query. No partition level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')


# Original notebook comment retained.
                            ## Define excel writer
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Silver).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                    ## Write missing and mismatched information to sheets in excel workbook
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `partition_level_missing_output_df.sort_values(by` so it can be reused later.
                                            partition_level_missing_output_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = f"missing_record_partition", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `partition_level_missing_output_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_level_missing_output_df.head(max_excel_row).to_excel(writer, sheet_name = f"missing_record_partition", startrow = 0, startcol=0, index = False)
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `partition_level_mismatched_record_df.sort_values(by` so it can be reused later.
                                            partition_level_mismatched_record_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = f"mismatched_record_partition", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_level_mismatched_record_detail_df.sort_values(by` so it can be reused later.
                                            partition_level_mismatched_record_detail_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = f"mismatched_detail_partition", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `partition_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = f"mismatched_record_partition", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_level_mismatched_record_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_level_mismatched_record_detail_df.head(max_excel_row).to_excel(writer, sheet_name = f"mismatched_detail_partition", startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if measure_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.sort_values(by` so it can be reused later.
                                            measure_level_mismatched_record_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `measure_level_mismatched_detail_df.sort_values(by` so it can be reused later.
                                            measure_level_mismatched_detail_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `measure_level_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            measure_level_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_detail", startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_count_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `partition_count_mismatched_df.sort_values(by` so it can be reused later.
                                            partition_count_mismatched_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_count_mismatched_detail_df.sort_values(by` so it can be reused later.
                                            partition_count_mismatched_detail_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `partition_count_mismatched_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_count_mismatched_df.head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_count_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_count_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_detail", startrow = 0, startcol=0, index = False)
# Print a message or value to the notebook output for validation or debugging.
                                print('Detailed report generated')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('Detailed report generated', style = 'List Bullet')
# Assign the result on the right-hand side to `detailed_report_status` so it can be reused later.
                                detailed_report_status = 'True'
# Handle an error raised in the preceding try block.
                            except Exception as e:
# Assign the result on the right-hand side to `detailed_report_status` so it can be reused later.
                                detailed_report_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                print('No detailed report generated')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('No detailed report generated', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'{e}', style= 'Intense Quote')

# Original notebook comment retained.
                            ## Upload workbook to storage
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if detailed_report_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                    workbook_path = path + f"{db}-{table_name}_(Netezza v. Silver).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                    shutil.copy2(f"{db}-{table_name}_(Netezza v. Silver).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                    os.remove(f"{db}-{table_name}_(Netezza v. Silver).xlsx")

# Original notebook comment retained.
                                    ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_table_name.append(table_name)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_count_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_count_mismatched_cnt.append(str(partition_count_mismatched_cnt))
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_count_mismatched_cnt.append('')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if measure_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_measure_level_mismatched_cnt_diff.append('')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_level_mismatched_cnt_diff.append(str(partition_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_level_missing_cnt.append(str(partition_level_missing_cnt))
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                    print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Silver)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Silver)", style='List Bullet')

# Handle an error raised in the preceding try block.
                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Silver)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Silver)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')


# Start a new indented code block for the statement above.
                        else :
# Original notebook comment retained.
                            ## This condition is to perform detailed validation for small and medium size tables
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Attempting to perform detailed validation on {db}.{table_name}. Need to confirm it is a small to medium size table')



# Original notebook comment retained.
                            ## Retrieve measure level query
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f" select distinct MEASURE_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_NETEZZA is not null")
# Original notebook comment retained.
                            ##TODO- Include Query_Measure_Silver if it exists. Confirm the Query_Measure_Silver
# Assign the result on the right-hand side to `Query_Measure_Silver` so it can be reused later.
                            Query_Measure_Silver=spark.sql(f" select distinct ADLS_QUERY_FOR_MEASURE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and ADLS_QUERY_FOR_MEASURE is not null")

# Print a message or value to the notebook output for validation or debugging.
                            print("else Full load Non Transactional ")
# Original notebook comment retained.
                            ## Get FULL netezza table for validation
# Assign the result on the right-hand side to `Query_Netezza` so it can be reused later.
                            Query_Netezza = f"""select * from {db_tmp}.{table_name}"""
# Assign the result on the right-hand side to `netezza_df` so it can be reused later.
                            netezza_df=readfromNetizza(Env,Query_Netezza,db)



# Original notebook comment retained.
                            ## Confirm table is small or medium size (i.e. Not greater than 5 million rows)
# Original notebook comment retained.
                            #max_rows_permitted = 5000000

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:
# Original notebook comment retained.
                                ## Retrieve row count
# Execute this line as part of the notebook's workflow logic.
                                cnt_netezza_df = int(report_dataframe_2.select(f'NETEZZA_TABLE_ROW_COUNT ({netezza_sub})').filter((col('NETEZZA_SCHEMA') == f'{db}') & (col('NETEZZA_TABLE') == f'{table_name}')).collect()[0][0]) #netezza_df.limit(max_rows_permitted + 1).count()
# Execute this line as part of the notebook's workflow logic.
                                cnt_df_ADLS_silver = int(report_dataframe_2.select('SILVER_TABLE_ROW_COUNT').filter((col('SILVER_SCHEMA') == f'{db}') & (col('SILVER_TABLE') == f'{table_name}')).collect()[0][0])#df_synapse.limit(max_rows_permitted + 1).count()
# Handle an error raised in the preceding try block.
                            except:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'{db}.{table_name} not available in summary report')
# Assign the result on the right-hand side to `cnt_netezza_df` so it can be reused later.
                                cnt_netezza_df = netezza_df.limit(max_rows_permitted + 1).count()
# Assign the result on the right-hand side to `cnt_df_ADLS_silver` so it can be reused later.
                                cnt_df_ADLS_silver = Data_Lake_df.limit(max_rows_permitted + 1).count()

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (cnt_netezza_df == 0) | (cnt_df_ADLS_silver == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Table {db}.{table_name} is empty either in Netezza or in Silver. No detailed validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Table {db}.{table_name} is empty either in Netezza or in Silver. No detailed validation performed', style = 'List Bullet')

# Check the next condition if the previous condition was not met.
                            elif (cnt_netezza_df > max_rows_permitted) | (cnt_df_ADLS_silver > max_rows_permitted):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Table {db}.{table_name} is a big table, with no partition column configured. No table level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Table {db}.{table_name} is a big table, with no partition column configures. No table level validation performed', style='List Bullet')

# Print a message or value to the notebook output for validation or debugging.
                                print('Will attempt to perform measure level validation')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('Attempting measure level validation for a big table with no partition configured', style= 'List Bullet')

# Original notebook comment retained.
                                ## PERFORM MEASURE LEVEL VALIDATION
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (Query_Measure_Netezza.count() == 0) | (Query_Measure_Silver.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                    measure_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                    query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_measure_silver` so it can be reused later.
                                    query_measure_silver = Query_Measure_Silver.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if (query_measure_netezza == '') | (query_measure_silver == ''):
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                        measure_validation_status = 'False' 
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Measure queries configured. Attempting to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Measure queries configured. Attempting to perform measure level validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                            query_measure_netezza = query_measure_netezza.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                            netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Assign the result on the right-hand side to `adls_silver_df_measure` so it can be reused later.
                                            adls_silver_df_measure = spark.sql(f"{query_measure_silver}")

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                            try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df` so it can be reused later.
                                                measure_level_mismatched_record_df = get_pandas(adls_silver_df_measure.subtract(netezza_df_measure))
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff` so it can be reused later.
                                                measure_level_mismatched_cnt_diff = len(measure_level_mismatched_record_df)
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                                measure_validation_status = 'True'

# Original notebook comment retained.
                                                ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                                with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Silver).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                                    ## Write missing and mismatched information to sheets in excel workbook
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                                    measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)

# Original notebook comment retained.
                                                ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                                try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                                    workbook_path = path + f"{db}-{table_name}_(Netezza v. Silver).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                                    shutil.copy2(f"{db}-{table_name}_(Netezza v. Silver).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                                    os.remove(f"{db}-{table_name}_(Netezza v. Silver).xlsx")

# Original notebook comment retained.
                                                    ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                                    print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Silver)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                    doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Silver)", style='List Bullet')

# Handle an error raised in the preceding try block.
                                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                                    print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Silver)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                    doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Silver)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                    doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
                                            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Failed to perform detailed validation on {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Failed to perform detailed validation on {db}.{table_name}", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'{e}', style='Intense Quote')
# Execute this line as part of the notebook's workflow logic.
                                                continue

# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed', style = 'List Bullet')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Netezza measure query: {query_measure_netezza}', style = 'Intense Quote')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Silver measure query: {query_measure_silver}', style = 'Intense Quote')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')

# Run the fallback branch when the earlier conditions do not match.
                            else:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Table size is less than or equal to {max_rows_permitted}. Performing detailed validation on full table {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Table size is less than or equal to {max_rows_permitted}. Performing detailed validation on full table {db}.{table_name}', style = 'List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:
# Original notebook comment retained.
                                    #table_level_mismatched_cnt_diff, table_level_mismatched_record_df, table_level_mismatched_record_detail_cnt, table_level_mismatched_record_detail_df, table_level_missing_cnt, table_level_missing_output_df = find_difference(netezza_df, Data_Lake_df, primary_key)
# Assign the result on the right-hand side to `table_level_mismatched_cnt_diff, table_level_mismatched_record_df, table_level_mismatched_record_detail_cnt, table_level_mismatched_record_detail_df, table_level_missing_cnt, table_level_missing_output_df` so it can be reused later.
                                    table_level_mismatched_cnt_diff, table_level_mismatched_record_df, table_level_mismatched_record_detail_cnt, table_level_mismatched_record_detail_df, table_level_missing_cnt, table_level_missing_output_df = find_difference(trim_space(netezza_df), trim_space(Data_Lake_df), primary_key)
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Successfully performed full table validation on {db}.{table_name}, comparing Netezza to Silver table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Successfully performed full table validation on {db}.{table_name}, comparing Netezza to Silver table', style='List Bullet')

# Original notebook comment retained.
                                    ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                    with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Silver).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                        ## Write missing and mismatched information to sheets in excel workbook
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `table_level_missing_output_df.sort_values(by` so it can be reused later.
                                            table_level_missing_output_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = "missing_record", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `table_level_missing_output_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            table_level_missing_output_df.head(max_excel_row).to_excel(writer, sheet_name = "missing_record", startrow = 0, startcol=0, index = False)
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `table_level_mismatched_record_df.sort_values(by` so it can be reused later.
                                            table_level_mismatched_record_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = "mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `table_level_mismatched_record_detail_df.sort_values(by` so it can be reused later.
                                            table_level_mismatched_record_detail_df.sort_values(by=primary_key).head(max_excel_row).to_excel(writer, sheet_name = "mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `table_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            table_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = "mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `table_level_mismatched_record_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            table_level_mismatched_record_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "mismatched_detail", startrow = 0, startcol=0, index = False)


# Original notebook comment retained.
                                    ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                        workbook_path = path + f"{db}-{table_name}_(Netezza v. Silver).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                        shutil.copy2(f"{db}-{table_name}_(Netezza v. Silver).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                        os.remove(f"{db}-{table_name}_(Netezza v. Silver).xlsx")

# Original notebook comment retained.
                                        ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_measure_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_table_level_mismatched_cnt_diff.append(str(table_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_table_level_missing_cnt.append(str(table_level_missing_cnt))
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                        print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Silver)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Silver)", style='List Bullet')


# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                        print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Silver)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Silver)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Silver table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Silver table', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                    print(f'ERROR OCCURRED: Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Silver table')
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'ERROR OCCURRED: Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Silver table', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'{e}', style='Intense Quote')
# Original notebook comment retained.
                    #continue

# Original notebook comment retained.
        ## Upload log file
# Execute this line as part of the notebook's workflow logic.
        save_log(doc)

# Handle an error raised in the preceding try block.
    except Exception as e:
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph(f'{e}', style='Intense Quote')
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph("Data Validation Run Failed", style="List Bullet")
# Print a message or value to the notebook output for validation or debugging.
        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph(f'{e}', style='Intense Quote')
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph('')

# Original notebook comment retained.
        ## Upload log file
# Execute this line as part of the notebook's workflow logic.
        save_log(doc)

# Original notebook comment retained.
        ## Exit the notebook run
# Execute this line as part of the notebook's workflow logic.
        dbutils.notebook.exit("Data Validation Run Failed") 


## Command cell 41

This section corresponds to command 41 from the original Databricks notebook.


In [ ]:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if (PL_TL != '7GS') & (Load_Type == 'Historical'): 

# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.functions import *.
    from pyspark.sql.functions import *
# Import specific objects from a module so they can be used directly in later code: from pyspark.sql.types import DecimalType, IntegerType, DateType, StructType, TimestampType.
    from pyspark.sql.types import DecimalType, IntegerType, DateType, StructType, TimestampType
# Import specific objects from a module so they can be used directly in later code: from datetime import date, timedelta, datetime.
    from datetime import date, timedelta, datetime
# Import specific objects from a module so they can be used directly in later code: from delta.tables import *.
    from delta.tables import *



# Original notebook comment retained.
    # Header for log document
# Add a heading to the Word document being used as a log or report.
    doc.add_heading('KEY TYPE UNIQUE = N',3)
# Assign the result on the right-hand side to `container` so it can be reused later.
    container = 'silver'

# Begin a protected block so the notebook can handle runtime errors more gracefully.
    try:

# Start a loop so the same logic is applied repeatedly across multiple items.
        for db in relevant_silver_db:

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
            if Env == 'QA':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = f"GDC_{db}"
# Check the next condition if the previous condition was not met.
            elif Env == 'PROD_DR':
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = f"{db}_DR"
# Run the fallback branch when the earlier conditions do not match.
            else:
# Assign the result on the right-hand side to `db_tmp` so it can be reused later.
                db_tmp = db
# Print a message or value to the notebook output for validation or debugging.
            print('DATABASE: ', db)


# Start a loop so the same logic is applied repeatedly across multiple items.
            for table_dict in [v for d in validation_tables_info_2 for k,v in d.items() if k == db][0]:
# Print a message or value to the notebook output for validation or debugging.
                print(table_dict)
# Assign the result on the right-hand side to `table_name` so it can be reused later.
                table_name=list(table_dict.keys())[0]
# Print a message or value to the notebook output for validation or debugging.
                print('TABLE NAME: ',table_name)     

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                try:   

# Assign the result on the right-hand side to `Partition_column` so it can be reused later.
                    Partition_column=spark.sql(f"select distinct PARTITION_COLUMN from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_COLUMN is not null")
# Assign the result on the right-hand side to `KEY_TYPE` so it can be reused later.
                    KEY_TYPE=spark.sql(f"select distinct KEY_TYPE_UNIQUE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and KEY_TYPE_UNIQUE is not null")
# Assign the result on the right-hand side to `KEY_TYPES` so it can be reused later.
                    KEY_TYPES=KEY_TYPE.collect()[0][0]
# Print a message or value to the notebook output for validation or debugging.
                    print('KEY_TYPES: ',KEY_TYPES)



# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                    if KEY_TYPES == "N" :
# Assign the result on the right-hand side to `tab_si_cnt +` so it can be reused later.
                        tab_si_cnt += 1
# Print a message or value to the notebook output for validation or debugging.
                        print(tab_si_cnt)

# Add a heading to the Word document being used as a log or report.
                        doc.add_heading(f"{db}.{table_name}", 4)

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if Partition_column.count() == 0:
# Assign the result on the right-hand side to `Partition_col` so it can be reused later.
                            Partition_col = None
# Print a message or value to the notebook output for validation or debugging.
                            print(f'No partition column configured for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'No partition column configured for {db}.{table_name}', style='List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                        else:
# Assign the result on the right-hand side to `Partition_col` so it can be reused later.
                            Partition_col =Partition_column.collect()[0][0]
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Partition column configured for {db}.{table_name} is {Partition_col}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Partition column configured for {db}.{table_name} is {Partition_col}', style='List Bullet')

# Original notebook comment retained.
                        ## The ADLS silver table
# Read data from storage into a DataFrame or Python object.
                        Data_Lake_df=spark.read.format("delta").option("header","true").load("/mnt/silver/"+db+"/ADMIN/"+table_name)
# Execute this line as part of the notebook's workflow logic.
                        Data_Lake_df.createOrReplaceTempView("Data_Lake_df")

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                        if ((Partition_col != None) and (Partition_col != 'NA') ):
# Original notebook comment retained.
                            ## Retrieve dataframe of the queries for row count per partition for Netezza tables, and Retrieve dataframe of query for generating a measure from Netezza and ADLS_Gold
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Retrieving Partition, Measure and Field Validation Queries from Fwk_Gold_Tables for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                            doc.add_paragraph(f'Retrieving Partition, Measure and Field Validation Queries from Fwk_Gold_Tables for {db}.{table_name}', style='List Bullet')


# Assign the result on the right-hand side to `Query_Agg_Netezza` so it can be reused later.
                            Query_Agg_Netezza=spark.sql(f"select distinct PARTITION_QUERY_NETEZZA from gold_df where TABLE_NAME = '{table_name}' and DATABASE ='{db}' and PARTITION_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f" select distinct MEASURE_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_NETEZZA is not null")
# Original notebook comment retained.
                            ##TODO- Include Query_Measure_Silver if it exists. Confirm the Query_Measure_Silver
# Assign the result on the right-hand side to `Query_Measure_Silver` so it can be reused later.
                            Query_Measure_Silver=spark.sql(f" select distinct ADLS_QUERY_FOR_MEASURE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and ADLS_QUERY_FOR_MEASURE is not null")

# Assign the result on the right-hand side to `Query_Field_Level_Netezza` so it can be reused later.
                            Query_Field_Level_Netezza=spark.sql(f"select distinct FIELD_VALIDATION_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and FIELD_VALIDATION_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Field_Level_Silver` so it can be reused later.
                            Query_Field_Level_Silver=spark.sql(f"select distinct ADLS_QUERY_FOR_FIELD from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and ADLS_QUERY_FOR_FIELD is not null")  


# Original notebook comment retained.
                            ## Retrieve dataframe of the field used for validation (YEAR, MONTH, DAY)
# Assign the result on the right-hand side to `PARTITION_ON` so it can be reused later.
                            PARTITION_ON=spark.sql(f"select PARTITION_ON from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and PARTITION_ON is not null")
# Assign the result on the right-hand side to `Partition_Type` so it can be reused later.
                            Partition_Type=PARTITION_ON.collect()[0][0].upper()
# Print a message or value to the notebook output for validation or debugging.
                            print('PARTITION TYPE:', Partition_Type)

# Original notebook comment retained.
                            ## Skip table if no queries are configured
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Field_Level_Netezza.count() == 0) & (Query_Measure_Netezza.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f"No field validation query or measure query configured. No detailed report generated for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f"No field validation query or measure query configured. No detailed report generated for {db}.{table_name}", style = 'List Bullet')
# Execute this line as part of the notebook's workflow logic.
                                continue


# Original notebook comment retained.
                            ## Get the various queries, partition field and partition value
# Execute this line as part of the notebook's workflow logic.
                            '''
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if Query_Agg_Netezza.count()!=0:
# Assign the result on the right-hand side to `query_table_load_netezza` so it can be reused later.
                                query_table_load_netezza =Query_Agg_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `netezza_df` so it can be reused later.
                                netezza_df=readfromNetizza(Env,query_table_load_netezza,db)
# Execute this line as part of the notebook's workflow logic.
                            '''

# Original notebook comment retained.
                            ## PERFORM PARTITION COUNT VALIDATION
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Agg_Netezza.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No partition queries configured for {db}.{table_name} in Fwk_Gold_Tables for Netezza. No partition count validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No partition queries configured for {db}.{table_name} in Fwk_Gold_Tables for Netezza. No partition count validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                partition_count_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                query_PARTITION_CNT_NZ = Query_Agg_Netezza.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_PARTITION_CNT_NZ == ''):
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                    partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                    print('No Partition Query is configured. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(('No Partition Query is configured. No partition count validation performed'), style= 'List Bullet')
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Attempting to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Attempting to perform partition count validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_PARTITION_CNT_NZ` so it can be reused later.
                                        query_PARTITION_CNT_NZ = query_PARTITION_CNT_NZ.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_partition_cnt_df` so it can be reused later.
                                        netezza_partition_cnt_df = readfromNetizza(Env, query_PARTITION_CNT_NZ, db)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                        if Partition_Type == 'YEAR' :
# Assign the result on the right-hand side to `Data_Lake_df_PARTITION_CNT` so it can be reused later.
                                            Data_Lake_df_PARTITION_CNT=spark.sql(f"select DATE_FORMAT({Partition_col}, 'yyyy') as {Partition_col}, count(*) as CNT from Data_Lake_df group by DATE_FORMAT({Partition_col}, 'yyyy')")
# Check the next condition if the previous condition was not met.
                                        elif Partition_Type == 'MONTH' :
# Assign the result on the right-hand side to `Data_Lake_df_PARTITION_CNT` so it can be reused later.
                                            Data_Lake_df_PARTITION_CNT=spark.sql(f"select DATE_FORMAT({Partition_col}, 'yyyy MMM') as {Partition_col}, count(*) as CNT from Data_Lake_df group by DATE_FORMAT({Partition_col}, 'yyyy MMM')")
# Check the next condition if the previous condition was not met.
                                        elif Partition_Type == 'DAY' :
# Assign the result on the right-hand side to `Data_Lake_df_PARTITION_CNT` so it can be reused later.
                                            Data_Lake_df_PARTITION_CNT=spark.sql(f"select DATE_FORMAT({Partition_col}, 'yyyy MMM dd') as {Partition_col}, count(*) as CNT from Data_Lake_df group by DATE_FORMAT({Partition_col}, 'yyyy MMM dd')")

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df` so it can be reused later.
                                            partition_count_mismatched_cnt, partition_count_mismatched_df, partition_count_mismatched_detail_cnt, partition_count_mismatched_detail_df = find_Mismatched(netezza_partition_cnt_df,Data_Lake_df_PARTITION_CNT,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed partition count validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed partition count validation for {db}.{table_name}", style = 'List Bullet')
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                            partition_count_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                            partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Failed to perform partition count validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Failed to perform partition count validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `partition_count_validation_status` so it can be reused later.
                                        partition_count_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad partition query configured for {db}.{table_name}. Please correct the query. No partition count validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad partition query configured for {db}.{table_name}. Please correct the query. No partition count validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')



# Original notebook comment retained.
                            ## PERFORM MEASURE LEVEL VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Measure_Netezza.count() == 0) | (Query_Measure_Silver.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                measure_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_measure_silver` so it can be reused later.
                                query_measure_silver = Query_Measure_Silver.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_measure_netezza == '') | (query_measure_silver == ''):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                    measure_validation_status = 'False' 
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Attempting to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Attempting to perform measure level validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                        query_measure_netezza = query_measure_netezza.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                        netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Assign the result on the right-hand side to `adls_silver_df_measure` so it can be reused later.
                                        adls_silver_df_measure = spark.sql(f"{query_measure_silver}")

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff, measure_level_mismatched_record_df, measure_level_mismatched_detail_cnt, measure_level_mismatched_detail_df` so it can be reused later.
                                            measure_level_mismatched_cnt_diff, measure_level_mismatched_record_df, measure_level_mismatched_detail_cnt, measure_level_mismatched_detail_df = find_Mismatched(netezza_df_measure,adls_silver_df_measure,Partition_col)
# Print a message or value to the notebook output for validation or debugging.
                                            print(f"Successfully performed measure level validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f"Successfully performed measure level validation for {db}.{table_name}", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                            measure_validation_status = 'True'

# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                            measure_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Failed to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Failed to perform measure level validation for {db}.{table_name}', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                        measure_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')


# Original notebook comment retained.
                            ## PERFORM PARTITION LEVEL VALIDATION

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (Query_Field_Level_Netezza.count()==0) | (Query_Field_Level_Silver.count()==0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                partition_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                            else:
# Assign the result on the right-hand side to `query_PARTITION_NZ` so it can be reused later.
                                query_PARTITION_NZ = Query_Field_Level_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_PARTITION_Silver` so it can be reused later.
                                query_PARTITION_Silver = Query_Field_Level_Silver.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (query_PARTITION_NZ == '') | (query_PARTITION_Silver == ''):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No field validation queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No partition level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                    partition_validation_status = 'False'  
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:
# Assign the result on the right-hand side to `query_PARTITION_NZ` so it can be reused later.
                                        query_PARTITION_NZ = query_PARTITION_NZ.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_partition` so it can be reused later.
                                        netezza_df_partition=readfromNetizza(Env,query_PARTITION_NZ,db)
# Assign the result on the right-hand side to `Data_Lake_df_1` so it can be reused later.
                                        Data_Lake_df_1=spark.sql(query_PARTITION_Silver)  

# Original notebook comment retained.
                                        ## Validate that the partition size is within the max row permitted limit
# Print a message or value to the notebook output for validation or debugging.
                                        print('Attempting to perform partition level validation. Need to confirm that the partition size is within acceptable limit')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph('Attempting to perform partition level validation. Need to confirm that the partition size is within acceptable limit', style = 'List Bullet')

# Assign the result on the right-hand side to `cnt_netezza_df_partition` so it can be reused later.
                                        cnt_netezza_df_partition = netezza_df_partition.limit(max_rows_permitted + 1).count()
# Assign the result on the right-hand side to `cnt_Data_Lake_df_1` so it can be reused later.
                                        cnt_Data_Lake_df_1 = Data_Lake_df_1.limit(max_rows_permitted + 1).count()

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                        if (cnt_netezza_df_partition == 0) | (cnt_Data_Lake_df_1 == 0):
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Partition for table {db}.{table_name} is empty either in Netezza or in Gold. No partition validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Partition for table {db}.{table_name} is empty either in Netezza or in Gold. No partition validation performed', style = 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                            partition_validation_status = 'False'

# Check the next condition if the previous condition was not met.
                                        elif (cnt_netezza_df_partition > max_rows_permitted) | (cnt_Data_Lake_df_1 > max_rows_permitted):
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Partition for {db}.{table_name} is greater than {max_rows_permitted}. No partition validation performed. Please reconfigure the partition for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Partition for {db}.{table_name} is greater than {max_rows_permitted}. No partition validation performed. Please reconfigure the partition for {db}.{table_name}', style = 'List Bullet')
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                            partition_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                                        else:
# Original notebook comment retained.
                                            ## Perform validation on the partition
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Partition for {db}.{table_name} is less than or equal to {max_rows_permitted}.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Partition for {db}.{table_name} is less than or equal to {max_rows_permitted}.', style = 'List Bullet')
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                            try:
# Original notebook comment retained.
                                                #partition_level_diff = get_pandas(netezza_df_partition.subtract(Data_Lake_df_1))
# Assign the result on the right-hand side to `partition_level_diff` so it can be reused later.
                                                partition_level_diff = get_pandas(trim_space(netezza_df_partition).subtract(trim_space(Data_Lake_df_1)))
# Assign the result on the right-hand side to `partition_level_diff_cnt` so it can be reused later.
                                                partition_level_diff_cnt = len(partition_level_diff)
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Successfully performed partition level validation for {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Successfully performed partition level validation for {db}.{table_name}", style='List Bullet')
# Assign the result on the right-hand side to `partiton_validation_status` so it can be reused later.
                                                partiton_validation_status = 'True'
# Handle an error raised in the preceding try block.
                                            except Exception as e:
# Assign the result on the right-hand side to `partiton_validation_status` so it can be reused later.
                                                partiton_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                                print(f'Failed to perform partiton level validation for {db}.{table_name}_(Netezza v. Silver)')
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'Failed to perform partiton level validation for {db}.{table_name}_(Netezza v. Silver)', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'{e}', style='Intense Quote') 
# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Assign the result on the right-hand side to `partition_validation_status` so it can be reused later.
                                        partition_validation_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Bad field validation query configured for {db}.{table_name}. Please correct the query. No partition level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Bad field validation query configured for {db}.{table_name}. Please correct the query. No partition level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')


# Original notebook comment retained.
                            ## Define excel writer
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:             
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Silver).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                    ## Write missing and mismatched information to sheets in excel workbook
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partiton_validation_status == 'True':
# Assign the result on the right-hand side to `partition_level_diff.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        partition_level_diff.head(max_excel_row).to_excel(writer, sheet_name = f"partition_difference_record", startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if measure_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.sort_values(by` so it can be reused later.
                                            measure_level_mismatched_record_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `measure_level_mismatched_detail_df.sort_values(by` so it can be reused later.
                                            measure_level_mismatched_detail_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `measure_level_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            measure_level_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "measure_mismatched_detail", startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_count_validation_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `partition_count_mismatched_df.sort_values(by` so it can be reused later.
                                            partition_count_mismatched_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_count_mismatched_detail_df.sort_values(by` so it can be reused later.
                                            partition_count_mismatched_detail_df.sort_values(by=Partition_col).head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_detail", startrow = 0, startcol=0, index = False)
# Handle an error raised in the preceding try block.
                                        except:
# Assign the result on the right-hand side to `partition_count_mismatched_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_count_mismatched_df.head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_record", startrow = 0, startcol=0, index = False)
# Assign the result on the right-hand side to `partition_count_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                            partition_count_mismatched_detail_df.head(max_excel_row).to_excel(writer, sheet_name = "partition_count_mismatched_detail", startrow = 0, startcol=0, index = False)
# Print a message or value to the notebook output for validation or debugging.
                                print('Detailed report generated')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('Detailed report generated', style = 'List Bullet')
# Assign the result on the right-hand side to `detailed_report_status` so it can be reused later.
                                detailed_report_status = 'True'
# Handle an error raised in the preceding try block.
                            except Exception as e:
# Assign the result on the right-hand side to `detailed_report_status` so it can be reused later.
                                detailed_report_status = 'False'
# Print a message or value to the notebook output for validation or debugging.
                                print('No detailed report generated')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('No detailed report generated', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'{e}', style= 'Intense Quote')


# Original notebook comment retained.
                            ## Upload workbook to storage
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if detailed_report_status == 'True':
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                    workbook_path = path + f"{db}-{table_name}_(Netezza v. Silver).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                    shutil.copy2(f"{db}-{table_name}_(Netezza v. Silver).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                    os.remove(f"{db}-{table_name}_(Netezza v. Silver).xlsx")

# Original notebook comment retained.
                                    ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_table_name.append(table_name)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partition_count_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_count_mismatched_cnt.append(str(partition_count_mismatched_cnt))
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_count_mismatched_cnt.append('')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if measure_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_measure_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_table_level_missing_cnt.append('')
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if partiton_validation_status == 'True':
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_level_diff_cnt.append(str(partition_level_diff_cnt))
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                    netezza_silver_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                    print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Silver)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Silver)", style='List Bullet')


# Handle an error raised in the preceding try block.
                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Silver)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Silver)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')





# Start a new indented code block for the statement above.
                        else :
# Original notebook comment retained.
                            ## This condition is to perform detailed validation for small and medium size tables
# Print a message or value to the notebook output for validation or debugging.
                            print(f'Attempting to perform detailed validation on {db}.{table_name}. Need to confirm it is a small to medium size table')



# Original notebook comment retained.
                            ## Retrieve measure level query
# Assign the result on the right-hand side to `Query_Measure_Netezza` so it can be reused later.
                            Query_Measure_Netezza=spark.sql(f" select distinct MEASURE_QUERY_NETEZZA from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and MEASURE_QUERY_NETEZZA is not null")
# Assign the result on the right-hand side to `Query_Measure_Silver` so it can be reused later.
                            Query_Measure_Silver=spark.sql(f" select distinct ADLS_QUERY_FOR_MEASURE from gold_df where TABLE_NAME='{table_name}' and DATABASE ='{db}' and ADLS_QUERY_FOR_MEASURE is not null")

# Print a message or value to the notebook output for validation or debugging.
                            print("else Full load Non Transactional ")
# Original notebook comment retained.
                            ## Get FULL netezza table for validation
# Assign the result on the right-hand side to `Query_Netezza` so it can be reused later.
                            Query_Netezza = f"""select * from {db_tmp}.{table_name}"""
# Assign the result on the right-hand side to `netezza_df` so it can be reused later.
                            netezza_df=readfromNetizza(Env,Query_Netezza,db)
# Execute this line as part of the notebook's workflow logic.
                            netezza_df.cache()



# Original notebook comment retained.
                            ## Confirm table is small or medium size (i.e. Not greater than 5 million rows)
# Original notebook comment retained.
                            #max_rows_permitted = 5000000

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                            try:
# Original notebook comment retained.
                                ## Retrieve row count
# Execute this line as part of the notebook's workflow logic.
                                cnt_netezza_df = int(report_dataframe_2.select(f'NETEZZA_TABLE_ROW_COUNT ({netezza_sub})').filter((col('NETEZZA_SCHEMA') == f'{db}') & (col('NETEZZA_TABLE') == f'{table_name}')).collect()[0][0]) #netezza_df.limit(max_rows_permitted + 1).count()
# Execute this line as part of the notebook's workflow logic.
                                cnt_df_ADLS_silver = int(report_dataframe_2.select('SILVER_TABLE_ROW_COUNT').filter((col('SILVER_SCHEMA') == f'{db}') & (col('SILVER_TABLE') == f'{table_name}')).collect()[0][0])#df_synapse.limit(max_rows_permitted + 1).count()
# Handle an error raised in the preceding try block.
                            except:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'{db}.{table_name} not available in summary report')
# Assign the result on the right-hand side to `cnt_netezza_df` so it can be reused later.
                                cnt_netezza_df = netezza_df.limit(max_rows_permitted + 1).count()
# Assign the result on the right-hand side to `cnt_df_ADLS_silver` so it can be reused later.
                                cnt_df_ADLS_silver = Data_Lake_df.limit(max_rows_permitted + 1).count()

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                            if (cnt_netezza_df == 0) | (cnt_df_ADLS_silver == 0):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Table {db}.{table_name} is empty either in Netezza or in Silver. No detailed validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Table {db}.{table_name} is empty either in Netezza or in Silver. No detailed validation performed', style = 'List Bullet')

# Check the next condition if the previous condition was not met.
                            elif (cnt_netezza_df > max_rows_permitted) | (cnt_df_ADLS_silver > max_rows_permitted):
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Table {db}.{table_name} is a big table, with no partition column configured. No table level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Table {db}.{table_name} is a big table, with no partition column configured. No table level validation performed', style='List Bullet')

# Print a message or value to the notebook output for validation or debugging.
                                print('Will attempt to perform measure level validation')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph('Attempting measure level validation for a big table with no partition configured', style= 'List Bullet')

# Original notebook comment retained.
                                ## PERFORM MEASURE LEVEL VALIDATION
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                if (Query_Measure_Netezza.count() == 0) | (Query_Measure_Silver.count() == 0):
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                    measure_validation_status = 'False'
# Run the fallback branch when the earlier conditions do not match.
                                else:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                    query_measure_netezza =Query_Measure_Netezza.collect()[0][0]
# Assign the result on the right-hand side to `query_measure_silver` so it can be reused later.
                                    query_measure_silver = Query_Measure_Silver.collect()[0][0]
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                                    if (query_measure_netezza == '') | (query_measure_silver == ''):
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'No measure queries configured for {db}.{table_name} in Fwk_Gold_Tables, either for Netezza or ADLS. No measure level validation performed.', style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                        measure_validation_status = 'False' 
# Run the fallback branch when the earlier conditions do not match.
                                    else:
# Print a message or value to the notebook output for validation or debugging.
                                        print(f'Measure queries configured. Attempting to perform measure level validation for {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'Measure queries configured. Attempting to perform measure level validation for {db}.{table_name}', style='List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                        try:
# Assign the result on the right-hand side to `query_measure_netezza` so it can be reused later.
                                            query_measure_netezza = query_measure_netezza.replace('db_tmp', db_tmp)
# Assign the result on the right-hand side to `netezza_df_measure` so it can be reused later.
                                            netezza_df_measure=readfromNetizza(Env,query_measure_netezza,db)
# Assign the result on the right-hand side to `adls_silver_df_measure` so it can be reused later.
                                            adls_silver_df_measure = spark.sql(f"{query_measure_silver}")

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                            try:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df` so it can be reused later.
                                                measure_level_mismatched_record_df = get_pandas(adls_silver_df_measure.subtract(netezza_df_measure))
# Assign the result on the right-hand side to `measure_level_mismatched_cnt_diff` so it can be reused later.
                                                measure_level_mismatched_cnt_diff = len(measure_level_mismatched_record_df)
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Successfully performed measure mismatched validation on {db}.{table_name}. This is a big table does not have a partition column configured", style='List Bullet')
# Assign the result on the right-hand side to `measure_validation_status` so it can be reused later.
                                                measure_validation_status = 'True'


# Original notebook comment retained.
                                                ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                                with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Silver).xlsx", engine="openpyxl", mode = 'w') as writer:
# Assign the result on the right-hand side to `measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                                    measure_level_mismatched_record_df.head(max_excel_row).to_excel(writer, sheet_name = f"measure_mismatched_record", startrow = 0, startcol=0, index = False)

# Original notebook comment retained.
                                                ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                                try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                                    workbook_path = path + f"{db}-{table_name}_(Netezza v. Silver).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                                    shutil.copy2(f"{db}-{table_name}_(Netezza v. Silver).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                                    os.remove(f"{db}-{table_name}_(Netezza v. Silver).xlsx")

# Original notebook comment retained.
                                                    ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_measure_level_mismatched_cnt_diff.append(str(measure_level_mismatched_cnt_diff))
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_table_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                                    netezza_silver_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                                    print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Silver)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                    doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Silver)", style='List Bullet')

# Handle an error raised in the preceding try block.
                                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                                    print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Silver)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                    doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Silver)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                    doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
                                            except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                                print(f"Failed to perform detailed validation on {db}.{table_name}")
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f"Failed to perform detailed validation on {db}.{table_name}", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                                print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                                doc.add_paragraph(f'{e}', style='Intense Quote')
# Execute this line as part of the notebook's workflow logic.
                                                continue

# Handle an error raised in the preceding try block.
                                        except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                            print(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed')
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'Bad measure query configured for {db}.{table_name}. Please correct the query. No measure level validation performed', style = 'List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                            print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                            doc.add_paragraph(f'{e}', style='Intense Quote')


# Run the fallback branch when the earlier conditions do not match.
                            else:
# Print a message or value to the notebook output for validation or debugging.
                                print(f'Table size is less than or equal to {max_rows_permitted}. Performing detailed validation on full table {db}.{table_name}')
# Add a paragraph to the Word document so the log captures another message or detail.
                                doc.add_paragraph(f'Table size is less than or equal to {max_rows_permitted}. Performing detailed validation on full table {db}.{table_name}', style = 'List Bullet')

# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                try:
# Original notebook comment retained.
                                    #table_level_diff = get_pandas(netezza_df.subtract(Data_Lake_df))
# Assign the result on the right-hand side to `table_level_diff` so it can be reused later.
                                    table_level_diff = get_pandas(trim_space(netezza_df).subtract(trim_space(Data_Lake_df)))
# Assign the result on the right-hand side to `table_level_diff_cnt` so it can be reused later.
                                    table_level_diff_cnt = len(table_level_diff)
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Successfully performed full table validation on {db}.{table_name}, comparing Netezza to Silver table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Successfully performed full table validation on {db}.{table_name}, comparing Netezza to Silver table', style='List Bullet')


# Original notebook comment retained.
                                    ## Define excel writer
# Open a context-managed resource so it is handled safely and closed automatically after use.
                                    with pd.ExcelWriter(f"{db}-{table_name}_(Netezza v. Silver).xlsx", engine="openpyxl", mode = 'w') as writer:

# Original notebook comment retained.
                                        ## Write missing and mismatched information to sheets in excel workbook
# Assign the result on the right-hand side to `table_level_diff.head(max_excel_row).to_excel(writer, sheet_name` so it can be reused later.
                                        table_level_diff.head(max_excel_row).to_excel(writer, sheet_name = "difference_record", startrow = 0, startcol=0, index = False)


# Original notebook comment retained.
                                    ## Upload workbook to storage
# Begin a protected block so the notebook can handle runtime errors more gracefully.
                                    try:        
# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
                                        workbook_path = path + f"{db}-{table_name}_(Netezza v. Silver).xlsx" 
# Execute this line as part of the notebook's workflow logic.
                                        shutil.copy2(f"{db}-{table_name}_(Netezza v. Silver).xlsx",workbook_path)
# Execute this line as part of the notebook's workflow logic.
                                        os.remove(f"{db}-{table_name}_(Netezza v. Silver).xlsx")

# Original notebook comment retained.
                                        ## Append detail summary to lists
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_database_name.append(db)
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_table_name.append(table_name)
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_count_mismatched_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_measure_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_table_level_mismatched_cnt_diff.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_table_level_missing_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_partition_level_diff_cnt.append('')
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_table_level_diff_cnt.append(table_level_diff_cnt)
# Execute this line as part of the notebook's workflow logic.
                                        netezza_silver_detailed_report_path.append(workbook_path)

# Print a message or value to the notebook output for validation or debugging.
                                        print(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Silver)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"Successfully uploaded detailed report for {db}.{table_name}_(Netezza v. Silver)", style='List Bullet')


# Handle an error raised in the preceding try block.
                                    except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                        print(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Silver)")
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f"Failed to upload detailed report for {db}.{table_name}_(Netezza v. Silver)", style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                        doc.add_paragraph(f'{e}', style='Intense Quote')


# Handle an error raised in the preceding try block.
                                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                                    print(f'Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Silver table')
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Silver table', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                                    doc.add_paragraph(f'{e}', style='Intense Quote')

# Handle an error raised in the preceding try block.
                except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
                    print(f'ERROR OCCURRED: Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Silver table')
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'ERROR OCCURRED: Failed to generate detailed report for {db}.{table_name}, when comparing Netezza to Silver table', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
                    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
                    doc.add_paragraph(f'{e}', style='Intense Quote')
# Original notebook comment retained.
                    #continue

# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph('')

# Original notebook comment retained.
        ## Upload log file
# Execute this line as part of the notebook's workflow logic.
        save_log(doc)

# Handle an error raised in the preceding try block.
    except Exception as e:
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph("Data Validation Run Failed", style="List Bullet")
# Print a message or value to the notebook output for validation or debugging.
        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph(f'{e}', style='Intense Quote')
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph('')

# Original notebook comment retained.
        ## Upload log file
# Execute this line as part of the notebook's workflow logic.
        save_log(doc)

# Original notebook comment retained.
        ## Exit the notebook run
# Execute this line as part of the notebook's workflow logic.
        dbutils.notebook.exit("Data Validation Run Failed") 



## Command cell 42

This section corresponds to command 42 from the original Databricks notebook.


In [ ]:
# Add a heading to the Word document being used as a log or report.
doc.add_heading('Update summary report', level=2)

# Begin a protected block so the notebook can handle runtime errors more gracefully.
try:

# Original notebook comment retained.
    ## Detail summary for Netezza v. Synapse

# Assign the result on the right-hand side to `dt_netezza_synapse` so it can be reused later.
    dt_netezza_synapse = pd.DataFrame({'NETEZZA_SCHEMA':netezza_synapse_database_name, 'NETEZZA_TABLE':netezza_synapse_table_name,  'PARTITION_COUNT_MISMATCHED_RECORDS(NETEZZA v. SYNAPSE)_CNT': netezza_synapse_partition_count_mismatched_cnt, 'MEASURE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SYNAPSE)_CNT': netezza_synapse_measure_level_mismatched_cnt_diff, 'PARTITION_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SYNAPSE)_CNT':netezza_synapse_partition_level_mismatched_cnt_diff, 'PARTITION_LEVEL_MISSING_RECORDS(NETEZZA v. SYNAPSE)_CNT':netezza_synapse_partition_level_missing_cnt, 'TABLE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SYNAPSE)_CNT': netezza_synapse_table_level_mismatched_cnt_diff, 'TABLE_LEVEL_MISSING_RECORDS(NETEZZA v. SYNAPSE)_CNT':netezza_synapse_table_level_missing_cnt, 'PARTITION_ROW_DIFFERENCE_CNT(NETEZZA v. SYNAPSE)(NON-UNIQUE KEY-TYPE)':netezza_synapse_partition_level_diff_cnt, 'TABLE_ROW_DIFFERENCE_CNT(NETEZZA v. SYNAPSE)(NON-UNIQUE KEY-TYPE)': netezza_synapse_table_level_diff_cnt, 'DETAILED_REPORT_PATH(NETEZZA v. SYNAPSE)':netezza_synapse_detailed_report_path})

# Original notebook comment retained.
    ## Detail summary dataframe for Netezza v. Gold

# Assign the result on the right-hand side to `dt_netezza_gold` so it can be reused later.
    dt_netezza_gold = pd.DataFrame({'NETEZZA_SCHEMA':netezza_gold_database_name, 'NETEZZA_TABLE':netezza_gold_table_name,  'PARTITION_COUNT_MISMATCHED_RECORDS(NETEZZA v. GOLD)_CNT': netezza_gold_partition_count_mismatched_cnt, 'MEASURE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. GOLD)_CNT': netezza_gold_measure_level_mismatched_cnt_diff, 'PARTITION_LEVEL_MISMATCHED_RECORDS(NETEZZA v. GOLD)_CNT':netezza_gold_partition_level_mismatched_cnt_diff, 'PARTITION_LEVEL_MISSING_RECORDS(NETEZZA v. GOLD)_CNT':netezza_gold_partition_level_missing_cnt, 'TABLE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. GOLD)_CNT': netezza_gold_table_level_mismatched_cnt_diff, 'TABLE_LEVEL_MISSING_RECORDS(NETEZZA v. GOLD)_CNT':netezza_gold_table_level_missing_cnt, 'PARTITION_ROW_DIFFERENCE_CNT(NETEZZA v. GOLD)(NON-UNIQUE KEY-TYPE)':netezza_gold_partition_level_diff_cnt, 'TABLE_ROW_DIFFERENCE_CNT(NETEZZA v. GOLD)(NON-UNIQUE KEY-TYPE)': netezza_gold_table_level_diff_cnt, 'DETAILED_REPORT_PATH(NETEZZA v. GOLD)':netezza_gold_detailed_report_path})

# Original notebook comment retained.
    ## Detail summary dataframe for Netezza v. Silver

# Assign the result on the right-hand side to `dt_netezza_silver` so it can be reused later.
    dt_netezza_silver = pd.DataFrame({'NETEZZA_SCHEMA':netezza_silver_database_name, 'NETEZZA_TABLE':netezza_silver_table_name, 'PARTITION_COUNT_MISMATCHED_RECORDS(NETEZZA v. SILVER)_CNT': netezza_silver_partition_count_mismatched_cnt, 'MEASURE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SILVER)_CNT': netezza_silver_measure_level_mismatched_cnt_diff, 'PARTITION_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SILVER)_CNT':netezza_silver_partition_level_mismatched_cnt_diff, 'PARTITION_LEVEL_MISSING_RECORDS(NETEZZA v. SILVER)_CNT':netezza_silver_partition_level_missing_cnt, 'TABLE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SILVER)_CNT': netezza_silver_table_level_mismatched_cnt_diff, 'TABLE_LEVEL_MISSING_RECORDS(NETEZZA v. SILVER)_CNT':netezza_silver_table_level_missing_cnt,  'PARTITION_ROW_DIFFERENCE_CNT(NETEZZA v. SILVER)(NON-UNIQUE KEY-TYPE)':netezza_silver_partition_level_diff_cnt, 'TABLE_ROW_DIFFERENCE_CNT(NETEZZA v. SILVER)(NON-UNIQUE KEY-TYPE)': netezza_silver_table_level_diff_cnt, 'DETAILED_REPORT_PATH(NETEZZA v. SILVER)':netezza_silver_detailed_report_path})

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if (PL_TL == '7GS') | (Load_Type != 'Historical'): 
# Assign the result on the right-hand side to `summary_report_cols_1` so it can be reused later.
        summary_report_cols_1 = ['NETEZZA_SCHEMA','NETEZZA_TABLE', 'SYNAPSE_SCHEMA', 'SYNAPSE_TABLE','MISMATCHED_COLUMNS (NETEZZA - SYNAPSE)', 'MISMATCHED_COLUMNS (SYNAPSE - NETEZZA)',f'NETEZZA_TABLE_ROW_COUNT ({netezza_sub})', 'SYNAPSE_TABLE_ROW_COUNT', 'ROW_COUNT_DIFFERENCE(NETEZZA - SYNAPSE)','PARTITION_COUNT_MISMATCHED_RECORDS(NETEZZA v. SYNAPSE)_CNT',  'MEASURE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SYNAPSE)_CNT',  'PARTITION_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SYNAPSE)_CNT', 'PARTITION_LEVEL_MISSING_RECORDS(NETEZZA v. SYNAPSE)_CNT', 'PARTITION_ROW_DIFFERENCE_CNT(NETEZZA v. SYNAPSE)(NON-UNIQUE KEY-TYPE)', 'TABLE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SYNAPSE)_CNT', 'TABLE_LEVEL_MISSING_RECORDS(NETEZZA v. SYNAPSE)_CNT',  'TABLE_ROW_DIFFERENCE_CNT(NETEZZA v. SYNAPSE)(NON-UNIQUE KEY-TYPE)', 'DETAILED_REPORT_PATH(NETEZZA v. SYNAPSE)','DETAILED_REPORT_PATH(NETEZZA v. GOLD)','Rpt_Exec_Ts']
# Run the fallback branch when the earlier conditions do not match.
    else:
# Assign the result on the right-hand side to `summary_report_cols_1` so it can be reused later.
        summary_report_cols_1 = ['NETEZZA_SCHEMA','NETEZZA_TABLE', 'SYNAPSE_SCHEMA', 'SYNAPSE_TABLE','MISMATCHED_COLUMNS (NETEZZA - SYNAPSE)', 'MISMATCHED_COLUMNS (SYNAPSE - NETEZZA)',f'NETEZZA_TABLE_ROW_COUNT ({netezza_sub})', 'SYNAPSE_TABLE_ROW_COUNT','GOLD_TABLE_ROW_COUNT', 'ROW_COUNT_DIFFERENCE(NETEZZA - SYNAPSE)', 'ROW_COUNT_DIFFERENCE(NETEZZA - GOLD)','PARTITION_COUNT_MISMATCHED_RECORDS(NETEZZA v. SYNAPSE)_CNT', 'PARTITION_COUNT_MISMATCHED_RECORDS(NETEZZA v. GOLD)_CNT','MEASURE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SYNAPSE)_CNT', 'MEASURE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. GOLD)_CNT', 'PARTITION_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SYNAPSE)_CNT', 'PARTITION_LEVEL_MISMATCHED_RECORDS(NETEZZA v. GOLD)_CNT', 'PARTITION_LEVEL_MISSING_RECORDS(NETEZZA v. SYNAPSE)_CNT','PARTITION_LEVEL_MISSING_RECORDS(NETEZZA v. GOLD)_CNT', 'PARTITION_ROW_DIFFERENCE_CNT(NETEZZA v. SYNAPSE)(NON-UNIQUE KEY-TYPE)', 'PARTITION_ROW_DIFFERENCE_CNT(NETEZZA v. GOLD)(NON-UNIQUE KEY-TYPE)', 'TABLE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SYNAPSE)_CNT', 'TABLE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. GOLD)_CNT', 'TABLE_LEVEL_MISSING_RECORDS(NETEZZA v. SYNAPSE)_CNT', 'TABLE_LEVEL_MISSING_RECORDS(NETEZZA v. GOLD)_CNT',  'TABLE_ROW_DIFFERENCE_CNT(NETEZZA v. SYNAPSE)(NON-UNIQUE KEY-TYPE)', 'TABLE_ROW_DIFFERENCE_CNT(NETEZZA v. GOLD)(NON-UNIQUE KEY-TYPE)', 'DETAILED_REPORT_PATH(NETEZZA v. SYNAPSE)','DETAILED_REPORT_PATH(NETEZZA v. GOLD)','Rpt_Exec_Ts']

# Original notebook comment retained.
    #if len(relevant_gold_db) > 0:
# Assign the result on the right-hand side to `detailed_summary_1` so it can be reused later.
    detailed_summary_1 = dt_netezza_synapse.merge(dt_netezza_gold, on = ['NETEZZA_SCHEMA', 'NETEZZA_TABLE'], how = 'outer')
# Original notebook comment retained.
        #detailed_summary_1 = detailed_summary_1[[summary_report_cols_1]]

# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
    if (Load_Type == 'Historical'): 
# Assign the result on the right-hand side to `summary_report_cols_2` so it can be reused later.
        summary_report_cols_2 = ['NETEZZA_SCHEMA','NETEZZA_TABLE', 'SILVER_SCHEMA','SILVER_TABLE', 'MISMATCHED_COLUMNS (NETEZZA - SILVER)', 'MISMATCHED_COLUMNS (SILVER - NETEZZA)', f'NETEZZA_TABLE_ROW_COUNT ({netezza_sub})', 'SILVER_TABLE_ROW_COUNT', 'ROW_COUNT_DIFFERENCE(NETEZZA - SILVER)','PARTITION_COUNT_MISMATCHED_RECORDS(NETEZZA v. SILVER)_CNT','MEASURE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SILVER)_CNT','PARTITION_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SILVER)_CNT','PARTITION_LEVEL_MISSING_RECORDS(NETEZZA v. SILVER)_CNT','PARTITION_ROW_DIFFERENCE_CNT(NETEZZA v. SILVER)(NON-UNIQUE KEY-TYPE)','TABLE_LEVEL_MISMATCHED_RECORDS(NETEZZA v. SILVER)_CNT','TABLE_LEVEL_MISSING_RECORDS(NETEZZA v. SILVER)_CNT','TABLE_ROW_DIFFERENCE_CNT(NETEZZA v. SILVER)(NON-UNIQUE KEY-TYPE)','DETAILED_REPORT_PATH(NETEZZA v. SILVER)', 'Rpt_Exec_Ts']

# Original notebook comment retained.
    #if len(relevant_silver_db) > 0:
# Assign the result on the right-hand side to `detailed_summary_2` so it can be reused later.
    detailed_summary_2 = dt_netezza_silver   #.loc[:,[summary_report_cols_2]] 



# Begin a protected block so the notebook can handle runtime errors more gracefully.
    try:
# Open a context-managed resource so it is handled safely and closed automatically after use.
        with pd.ExcelWriter("SUMMARY_REPORT.xlsx", engine="openpyxl", mode = 'w') as writer1:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
            if len(relevant_gold_db) > 0:
# Original notebook comment retained.
                ## Read summary report
# Assign the result on the right-hand side to `summary_report_updated_1` so it can be reused later.
                summary_report_updated_1 = report_dataframe_pd_1.merge(detailed_summary_1, on = ['NETEZZA_SCHEMA', 'NETEZZA_TABLE'], how = 'outer')
# Assign the result on the right-hand side to `summary_report_updated_1` so it can be reused later.
                summary_report_updated_1 = summary_report_updated_1.loc[:,summary_report_cols_1]
# Original notebook comment retained.
                ## Write summary report for Netezza, Synapse and Gold to sheet in excel workbook
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
                if (PL_TL == '7GS') | (Load_Type != 'Historical'): 
# Assign the result on the right-hand side to `sheet_name` so it can be reused later.
                    sheet_name =  "NETEZZA v. SYNAPSE"
# Run the fallback branch when the earlier conditions do not match.
                else:
# Assign the result on the right-hand side to `sheet_name` so it can be reused later.
                    sheet_name =  "NETEZZA v. SYNAPSE v. GOLD"
# Assign the result on the right-hand side to `summary_report_updated_1.to_excel(writer1, sheet_name` so it can be reused later.
                summary_report_updated_1.to_excel(writer1, sheet_name = sheet_name, startrow = 0, startcol=0, index = False)
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
            if (len(relevant_silver_db) > 0) & (Load_Type == 'Historical'):
# Original notebook comment retained.
                ## Read summary report
# Assign the result on the right-hand side to `summary_report_updated_2` so it can be reused later.
                summary_report_updated_2 = report_dataframe_pd_2.merge(detailed_summary_2, on = ['NETEZZA_SCHEMA', 'NETEZZA_TABLE'], how = 'outer')
# Assign the result on the right-hand side to `summary_report_updated_2` so it can be reused later.
                summary_report_updated_2 = summary_report_updated_2.loc[:,summary_report_cols_2]
# Original notebook comment retained.
                ## Write summary report for Netezza and Silver to sheet in excel workbook
# Assign the result on the right-hand side to `sheet_name` so it can be reused later.
                sheet_name =  "NETEZZA v. SILVER"
# Assign the result on the right-hand side to `summary_report_updated_2.to_excel(writer1, sheet_name` so it can be reused later.
                summary_report_updated_2.to_excel(writer1, sheet_name = sheet_name, startrow = 0, startcol=0, index = False)

# Assign the result on the right-hand side to `workbook_path` so it can be reused later.
        workbook_path = path + 'SUMMARY_REPORT.xlsx'            
# Execute this line as part of the notebook's workflow logic.
        shutil.copy2('SUMMARY_REPORT.xlsx',workbook_path)
# Print a message or value to the notebook output for validation or debugging.
        print('Successfully updated summary report')
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph('Successfully updated summary report', style='List Bullet')
# Execute this line as part of the notebook's workflow logic.
        os.remove('SUMMARY_REPORT.xlsx')

# Handle an error raised in the preceding try block.
    except Exception as e:
# Print a message or value to the notebook output for validation or debugging.
        print('Failed to update summary report')
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph('Failed to update summary report', style='List Bullet')
# Print a message or value to the notebook output for validation or debugging.
        print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
        doc.add_paragraph(f'{e}', style='Intense Quote')

# Original notebook comment retained.
    ## Upload log file
# Execute this line as part of the notebook's workflow logic.
    save_log(doc)


# Handle an error raised in the preceding try block.
except Exception as e:
# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph("Data Validation Run Failed", style="List Bullet")
# Print a message or value to the notebook output for validation or debugging.
    print(e)
# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph(f'{e}', style='Intense Quote')
# Add a paragraph to the Word document so the log captures another message or detail.
    doc.add_paragraph('')
# Original notebook comment retained.
    ## Upload log file
# Execute this line as part of the notebook's workflow logic.
    save_log(doc)

# Original notebook comment retained.
    ## Exit the notebook run
# Original notebook comment retained.
    #dbutils.notebook.exit("Data Validation Run Failed") 

## Command cell 43

This section corresponds to command 43 from the original Databricks notebook.


In [ ]:
# Import specific objects from a module so they can be used directly in later code: from datetime import datetime.
from datetime import datetime
# Capture the current timestamp so the notebook can stamp logs, paths, or outputs.
now = datetime.now()
# Format a date or time value into a string representation.
now.strftime("%H:%M:%S")

## Command cell 44

This section corresponds to command 44 from the original Databricks notebook.


In [ ]:
# Start a conditional branch so the notebook executes different logic depending on the current inputs or environment.
if RptType == 'Detailed Report':
# Execute this line as part of the notebook's workflow logic.
    dbutils.notebook.exit(RptType) 

## Command cell 45

This section corresponds to command 45 from the original Databricks notebook.


## Command cell 46

This section corresponds to command 46 from the original Databricks notebook.


## Command cell 47

This section corresponds to command 47 from the original Databricks notebook.


## Command cell 48

This section corresponds to command 48 from the original Databricks notebook.


## Command cell 49

This section corresponds to command 49 from the original Databricks notebook.


## Command cell 50

This section corresponds to command 50 from the original Databricks notebook.


## Command cell 51

This section corresponds to command 51 from the original Databricks notebook.


## Command cell 52

This section corresponds to command 52 from the original Databricks notebook.


## Command cell 53

This section corresponds to command 53 from the original Databricks notebook.


## Command cell 54

This section corresponds to command 54 from the original Databricks notebook.


## Command cell 55

This section corresponds to command 55 from the original Databricks notebook.


## Command cell 56

This section corresponds to command 56 from the original Databricks notebook.


## Command cell 57

This section corresponds to command 57 from the original Databricks notebook.


## Command cell 58

This section corresponds to command 58 from the original Databricks notebook.


## Command cell 59

This section corresponds to command 59 from the original Databricks notebook.


## Command cell 60

This section corresponds to command 60 from the original Databricks notebook.


## Command cell 61

This section corresponds to command 61 from the original Databricks notebook.
